# ARC-AGI-2 hybrid v7: NVARC baseline on 64 training sequences per task and all its decoding views (version 2's), then Qwen3.8-27B direct answers on the shortest quarter of the tasks, with per-output grid parsing and deadline-aware token limits (version 6nb's)

Phase 1 runs the public notebook mikelou1/arc-agi2-lb33-89-minimal-perfpatch (B0; the NVARC 2025 solution: Qwen3-4B fine-tuned model sorokin/qwen3_4b_grids15_sft139, per-task LoRA test-time training, depth-first decoding, augmentation rescoring, score_kgmon selection) with half the test-time training: 64 training sequences per task instead of 128. Before it starts, a background reader reads B0's kernel source and model files once, sequentially, into the page cache, so that B0's four ranks do not contend on the input storage at start-up (skipped for the weights when memory is short; B0's code and order are unchanged). B0's workers stop at 9 h 18 m; its two attempts per test output are then written to submission.json (the safety net). Phase 2 (version 7-vr) frees the GPUs, serves Qwen3.8-27B-FP8 with vLLM over the 4 L4 GPUs (version 7's serving ladder, start-up, deadlines and hard stop) and runs the verify-and-refine harness (arc_harness, solver vr) on the tasks, shortest first: each first sample asks for the test grids and a Python transform(grid) in one reply (reasoning effort xhigh, at most 49,152 generated tokens); the program runs on every demonstration (the verification bit); a reply cut inside its reasoning gets one rescue request without thinking (8,192 tokens); an unverified first sample gets one feedback round (the program's results on the demonstrations, wrong cells as prediction/correct, 24,000 tokens); time left buys second samples on unverified tasks, each on a rotated, reflected and recoloured view mapped back. The shortest quarter's every stage goes before the other tasks' first samples. Per test output the candidates are ranked verified programs first, then support, self-consistency and demonstration accuracy; a clean verified 27B answer takes attempt 1 and B0's attempt 1 moves to attempt 2, otherwise B0's attempt 1 is kept and the best 27B grid takes attempt 2. submission.json covers every task and is rewritten atomically every 20 s; no request runs past 11 h 20 m; hard stop 11 h 30 m. A commit run (not the competition rerun) takes the same path on 5 evaluation tasks in under an hour; it checks the path, not the answers.

Built on:
- [mikelou1/arc-agi2-lb33-89-minimal-perfpatch](https://www.kaggle.com/code/mikelou1/arc-agi2-lb33-89-minimal-perfpatch) (Apache-2.0; NVARC 2025 solution): arc_loader.py and arc_decoder.py unchanged; arc_solver.py and starter.py with the input paths, the commit-run task list and the training-set size changed; its environment cell.
- [sorokin/qwen3_4b_grids15_sft139](https://www.kaggle.com/models/sorokin/qwen3_4b_grids15_sft139) (Apache-2.0): the NVARC 2025 model, unmodified.
- [sorokin/pip-install-unsloth-flash-patch](https://www.kaggle.com/code/sorokin/pip-install-unsloth-flash-patch) (Kaggle public notebook, Apache-2.0): B0's kernel source (unsloth install), unmodified.
- [madarshbb/arc-qwen-3-8-27b-agent](https://www.kaggle.com/code/madarshbb/arc-qwen-3-8-27b-agent) by Adarsh (Apache-2.0): direct mode; run_direct.py, client.py and grids.py unchanged (phase 2 replaces run_direct's grid parser by a per-output one at run time); the server, sampling and direct-mode settings of its version 2.
- [madarshbb/arc-agent-sub-q38-direct-b98k-c16-792ef9](https://www.kaggle.com/code/madarshbb/arc-agent-sub-q38-direct-b98k-c16-792ef9) by Adarsh (Apache-2.0): submit_main.py (wheelhouse path from the config, server wait configurable).
- Qwen3.8-27B-FP8 (Apache-2.0) via [mikedan7/qwen3-8-27b-fp8-official](https://www.kaggle.com/models/mikedan7/qwen3-8-27b-fp8-official): unmodified.
- arc_harness (verify-and-refine harness; MIT-0), with prompt, feedback format and loop parts condensed from [poetiq-ai/poetiq-arc-agi-solver](https://github.com/poetiq-ai/poetiq-arc-agi-solver) (MIT, Copyright 2025 Poetiq, Inc.; notice in arc_harness/NOTICES.md).
- [mirzamilanfarabi/arc3-vllm-h100-wheelhouse-v3](https://www.kaggle.com/datasets/mirzamilanfarabi/arc3-vllm-h100-wheelhouse-v3) (Apache-2.0): vLLM 0.19 offline wheelhouse.
- [keithtyser/taaf-duck-qwen38-serving-v1](https://www.kaggle.com/datasets/keithtyser/taaf-duck-qwen38-serving-v1) (licence field: unknown): only its transformers 5.8.0 and huggingface_hub 1.5.0 wheels (Apache-2.0 packages), as the serving overlay.
- [python-build-standalone](https://github.com/astral-sh/python-build-standalone/releases/tag/20261003) (CPython PSF-2.0; build scripts MPL-2.0): CPython 3.12.15 for phase 2, the release archive unmodified, attached as purushottamanurag/cpython-3-12-standalone-bin.


In [1]:
# Hybrid time budget, seconds after this cell. The competition rerun runs every test task; a commit run (not
# the rerun) runs the SMOKE budget on the 5 tasks of the setup cell: the path (both phases, the model swap,
# torch on the GPUs, a server option set, the size route, the merge, a valid submission) within 57 min of
# launch, not answers.
import os
import time

NOTEBOOK_START_TIME = time.time()
HYBRID_RERUN = bool(os.getenv('KAGGLE_IS_COMPETITION_RERUN'))
COMPETITION_BUDGET = {
    'phase1_end_s': 9 * 3600 + 18 * 60,   # B0's workers take no new task and decode nothing after this (B0: 12 h - 10 min)
    'phase1_kill_s': 9 * 3600 + 26 * 60,  # B0's process group is killed if a task in flight still runs (then GPUs checked free)
    'preinstall_s': 15 * 60,              # phase-2 wheelhouse install starts in the background (CPU and disk only)
    'refresh_s': 30 * 60,                 # submission.json rebuilt from B0's candidate files this often during phase 1
    'server_give_up_s': 11 * 3600,        # no further vLLM option set is started after this
    'issue_end_s': 11 * 3600 + 15 * 60,   # no 27B request is sent after this
    'collect_end_s': 11 * 3600 + 20 * 60, # requests in flight are cut by their timeout; what finished is merged
    'p2_kill_s': 11 * 3600 + 25 * 60,     # the phase-2 process group is killed if still running; final merge
    'hard_stop_s': 11 * 3600 + 30 * 60,   # the notebook is done with a valid submission.json (Kaggle limit 12 h)
}
SMOKE_BUDGET = {
    'phase1_end_s': 12 * 60,              # B0's workers take no new task and decode nothing after this (B0: 12 h - 10 min)
    'phase1_kill_s': 14 * 60,             # B0's process group is killed if a task in flight still runs (then GPUs checked free)
    'preinstall_s': 1 * 60,               # phase-2 wheelhouse install starts in the background (CPU and disk only)
    'refresh_s': 5 * 60,                  # submission.json rebuilt from B0's candidate files this often during phase 1
    'server_give_up_s': 42 * 60,          # no further vLLM option set is started after this
    'issue_end_s': 47 * 60,               # no 27B request is sent after this
    'collect_end_s': 50 * 60,             # requests in flight are cut by their timeout; what finished is merged
    'p2_kill_s': 52 * 60,                 # the phase-2 process group is killed if still running; final merge
    'hard_stop_s': 54 * 60,               # the notebook is done with a valid submission.json (Kaggle limit 12 h)
}
HYBRID_BUDGET = dict(COMPETITION_BUDGET if HYBRID_RERUN else SMOKE_BUDGET)
for _b in (COMPETITION_BUDGET, SMOKE_BUDGET):
    _v = [_b[k] for k in ('phase1_end_s', 'phase1_kill_s', 'server_give_up_s', 'issue_end_s', 'collect_end_s',
                          'p2_kill_s', 'hard_stop_s')]
    assert _v == sorted(_v) and _b['hard_stop_s'] <= 12 * 3600 - 30 * 60, _b
# B0's name for its end time: workers take no new task and stop decoding after it
global_end_time = NOTEBOOK_START_TIME + HYBRID_BUDGET['phase1_end_s']
print(f"hybrid: {'competition rerun' if HYBRID_RERUN else 'commit run (SMOKE)'}; phase 1 ends "
      f"{HYBRID_BUDGET['phase1_end_s']} s, 27B requests until {HYBRID_BUDGET['issue_end_s']} s, hard stop "
      f"{HYBRID_BUDGET['hard_stop_s']} s after start")


hybrid: commit run (SMOKE); phase 1 ends 720 s, 27B requests until 2820 s, hard stop 3240 s after start


In [2]:
# Hybrid setup: inputs (both Kaggle mount layouts), the B0 start-up guard (background page-cache reader), mode
# and task set, a valid submission.json before anything runs, the phase-2 interpreter and settings. Never
# raises (a raise fails the competition rerun).
import glob
import json
import os
import shutil
import subprocess
import sys
import threading
import time
import traceback


# Kaggle mounts inputs in the current layout (/kaggle/input/{competitions,datasets,models}/<owner>/<slug>)
# or in the flat legacy layout (/kaggle/input/<slug>); use whichever exists.
def _kaggle_input(*candidates):
    for _c in candidates:
        if os.path.exists(_c):
            return _c
    return candidates[0]


COMPETITION_DIR = _kaggle_input('/kaggle/input/competitions/arc-prize-2026-arc-agi-2', '/kaggle/input/arc-prize-2026-arc-agi-2')
B0_MODEL_DIR = _kaggle_input('/kaggle/input/models/sorokin/qwen3_4b_grids15_sft139/transformers/bfloat16/1', '/kaggle/input/qwen3_4b_grids15_sft139/transformers/bfloat16/1')
WHEELHOUSE_DIR = _kaggle_input('/kaggle/input/datasets/mirzamilanfarabi/arc3-vllm-h100-wheelhouse-v3', '/kaggle/input/arc3-vllm-h100-wheelhouse-v3')
OVERLAY_DIR = _kaggle_input('/kaggle/input/datasets/keithtyser/taaf-duck-qwen38-serving-v1', '/kaggle/input/taaf-duck-qwen38-serving-v1')
Q27_MODEL_DIR = _kaggle_input('/kaggle/input/models/mikedan7/qwen3-8-27b-fp8-official/pytorch/hf-fp8/1', '/kaggle/input/qwen3-8-27b-fp8-official/pytorch/hf-fp8/1')


# B0 start-up guard. B0's start-up on Kaggle was bimodal (first rank 185-309 s after launch in 6 runs, 1,316-1,684 s
# in 3; weights loaded by 322-586 s vs 2,387-2,841 s): four ranks reading the kernel source and the 7.3 GB of weights
# at once from the input storage. One daemon thread, started here before phase 1 launches B0, reads every B0 model
# file (config and tokenizer, then the shards) and then every file under the unsloth kernel source once, sequentially
# in 16 MiB blocks, into the page cache (weights first: 7.3 GB at about 107 MB/s on the CPU twin, while the kernel
# source, 9.3 GB in 44,152 files, read at 26 MB/s). It never blocks or fails the notebook (exceptions caught; times
# and bytes in PREFETCH_INFO) and changes nothing in B0. A part is skipped (and logged) when the memory headroom is
# below what the reader has cached so far plus the part plus PREFETCH_RESERVE_BYTES; a part stops (logged) once B0 no
# longer needs it: the weights when 4 ranks have loaded them, the kernel source when 4 ranks have started.
PREFETCH_BLOCK_BYTES = 16 << 20
PREFETCH_RESERVE_BYTES = 16 << 30    # host memory kept free for B0's four ranks beyond the cached files
PREFETCH_INFO = {'status': 'not started'}


def _b0_kernel_source_dir():
    # where Kaggle mounted B0's kernel source (it is on PYTHONPATH): /kaggle/usr/lib/notebooks/<owner>/<slug>
    _cands = sorted(glob.glob('/kaggle/usr/lib/notebooks/*/pip_install_unsloth_flash_patch'))
    _cands += ['/kaggle/usr/lib/pip_install_unsloth_flash_patch']
    _cands += [_p for _p in os.environ.get('PYTHONPATH', '').split(os.pathsep) + sys.path
               if _p and os.path.basename(_p.rstrip('/')) == 'pip_install_unsloth_flash_patch']
    for _c in _cands:
        if os.path.isdir(_c):
            return _c
    try:
        import importlib.util
        _spec = importlib.util.find_spec('unsloth')         # locates the package without importing it
        if _spec is not None and _spec.origin and _spec.origin.startswith('/kaggle/usr/lib/'):
            return os.path.dirname(os.path.dirname(_spec.origin))
    except Exception:
        pass
    return None


def _prefetch_files(_d, _model=False):
    _out = []
    for _root, _dirs, _files in os.walk(_d):
        _dirs.sort()
        for _n in sorted(_files):
            _p = os.path.join(_root, _n)
            if os.path.isfile(_p) and not os.path.islink(_p):
                _out.append(_p)
    if _model:                                                # config and tokenizer first, then the shards in order
        _out.sort(key=lambda _p: (_p.endswith('.safetensors'), _p))
    return _out


def _memory_headroom():
    # MemAvailable, capped by a cgroup memory limit (limit minus the non-file memory in use)
    _m = {}
    try:
        with open('/proc/meminfo') as _f:
            for _l in _f:
                _k, _v = _l.split(':', 1)
                if _k in ('MemTotal', 'MemAvailable', 'Cached'):
                    _m[_k] = int(_v.split()[0]) * 1024
    except Exception as exc:
        _m['meminfo_error'] = repr(exc)[:200]
    _head = _m.get('MemAvailable')
    for _lim_p, _cur_p, _stat_p, _key in (
            ('/sys/fs/cgroup/memory.max', '/sys/fs/cgroup/memory.current', '/sys/fs/cgroup/memory.stat', 'file'),
            ('/sys/fs/cgroup/memory/memory.limit_in_bytes', '/sys/fs/cgroup/memory/memory.usage_in_bytes',
             '/sys/fs/cgroup/memory/memory.stat', 'total_cache')):
        try:
            with open(_lim_p) as _f:
                _lim = _f.read().strip()
        except Exception:
            continue
        try:
            if _lim == 'max' or int(_lim) >= 1 << 60:
                _m['cgroup_limit'] = None
                break
            with open(_cur_p) as _f:
                _cur = int(_f.read().strip())
            _file = 0
            with open(_stat_p) as _f:
                for _l in _f:
                    _kv = _l.split()
                    if len(_kv) == 2 and _kv[0] == _key:
                        _file = int(_kv[1])
            _m.update(cgroup_limit=int(_lim), cgroup_nonfile_used=_cur - _file)
            _cap = int(_lim) - (_cur - _file)
            _head = _cap if _head is None else min(_head, _cap)
        except Exception as exc:
            _m['cgroup_error'] = repr(exc)[:200]
        break
    _m['headroom'] = _head
    return _m


def _prefetch_no_longer_needed(_part):
    # B0's start-up as phase 1 records it from B0's own lines (P1_INFO exists once phase 1 runs)
    _b0s = (globals().get('P1_INFO') or {}).get('b0_startup') or {}
    if _part == 'weights' and len(_b0s.get('weights_loaded_s') or []) >= 4:
        return '4 B0 ranks have loaded the weights'
    if _part == 'kernel_source' and len(_b0s.get('rank_start_s') or {}) >= 4:
        return '4 B0 ranks have started (imports done)'
    return None


def _prefetch_read(_files, _part, _total):
    _buf = memoryview(bytearray(PREFETCH_BLOCK_BYTES))
    _t, _n, _b, _stop = time.time(), 0, 0, None
    for _p in _files:
        _stop = _prefetch_no_longer_needed(_part)
        if _stop:
            break
        try:
            with open(_p, 'rb', buffering=0) as _f:
                try:
                    os.posix_fadvise(_f.fileno(), 0, 0, os.POSIX_FADV_SEQUENTIAL)
                except Exception:
                    pass
                _blocks = 0
                while True:
                    _r = _f.readinto(_buf)
                    if not _r:
                        break
                    _b += _r
                    _blocks += 1
                    if _blocks % 64 == 0 and _prefetch_no_longer_needed(_part):
                        break
            _n += 1
        except Exception as exc:
            PREFETCH_INFO.setdefault('errors', []).append(f'{_p}: {exc!r}'[:300])
    _s = time.time() - _t
    PREFETCH_INFO[_part] = {'files': _n, 'bytes': _b, 's': round(_s, 1), 'mb_per_s': round(_b / 1e6 / max(_s, 1e-6), 1),
                            'start_s': round(_t - NOTEBOOK_START_TIME, 1),
                            'end_s': round(time.time() - NOTEBOOK_START_TIME, 1),
                            'stopped': (_stop or _prefetch_no_longer_needed(_part)) if _b < _total else None}
    return _b


def _prefetch_b0():
    _t = time.time()
    try:
        PREFETCH_INFO.update(status='running', start_s=round(_t - NOTEBOOK_START_TIME, 1), start_epoch=_t,
                             b0_model_dir=B0_MODEL_DIR, reserve_bytes=PREFETCH_RESERVE_BYTES)
        _cached = 0                                           # bytes this reader has put in the page cache
        for _part in ('weights', 'kernel_source'):
            # each part is listed only when its turn comes (listing the kernel source's ~44k files takes about a
            # minute on the input storage; the weights must not wait for it)
            if _part == 'weights':
                _files = _prefetch_files(B0_MODEL_DIR, _model=True) if os.path.isdir(B0_MODEL_DIR) else []
            else:
                _ks = _b0_kernel_source_dir()
                PREFETCH_INFO['kernel_source_dir'] = _ks
                _files = _prefetch_files(_ks) if _ks else []
            _bytes = sum(os.path.getsize(_p) for _p in _files)
            PREFETCH_INFO.update({f'{_part}_files': len(_files), f'{_part}_bytes': _bytes,
                                  f'{_part}_listed_s': round(time.time() - NOTEBOOK_START_TIME, 1)})
            _m = _memory_headroom()
            PREFETCH_INFO[f'memory_before_{_part}'] = _m
            if not _files:
                PREFETCH_INFO[f'{_part}_skipped'] = 'no files found'
            elif _m.get('headroom') is not None and _m['headroom'] < _cached + _bytes + PREFETCH_RESERVE_BYTES:
                PREFETCH_INFO[f'{_part}_skipped'] = (f"memory headroom {_m['headroom'] / 2**30:.1f} GiB < "
                                                     f"{_cached / 2**30:.1f} GiB cached + {_bytes / 2**30:.1f} GiB + "
                                                     f"reserve {PREFETCH_RESERVE_BYTES / 2**30:.0f} GiB")
            else:
                _cached += _prefetch_read(_files, _part, _bytes)
        PREFETCH_INFO['status'] = 'done'
    except BaseException:
        PREFETCH_INFO.update(status='error', error=traceback.format_exc()[-1500:])
    PREFETCH_INFO['s'] = round(time.time() - _t, 1)
    PREFETCH_INFO['end_s'] = round(time.time() - NOTEBOOK_START_TIME, 1)
    try:
        print(f"[hybrid {PREFETCH_INFO['end_s']:8.1f}s] B0 prefetch {PREFETCH_INFO['status']} in {PREFETCH_INFO['s']} s: "
              f"weights {PREFETCH_INFO.get('weights') or PREFETCH_INFO.get('weights_skipped')}; "
              f"kernel source {PREFETCH_INFO.get('kernel_source') or PREFETCH_INFO.get('kernel_source_skipped')}; "
              f"errors {len(PREFETCH_INFO.get('errors', []))}", flush=True)
    except Exception:
        pass


PREFETCH_THREAD = threading.Thread(target=_prefetch_b0, name='b0-prefetch', daemon=True)
try:
    PREFETCH_THREAD.start()
except Exception as exc:
    PREFETCH_INFO.update(status='not started', error=repr(exc)[:300])

TEST_CHALLENGES = os.path.join(COMPETITION_DIR, 'arc-agi_test_challenges.json')
EVAL_CHALLENGES = os.path.join(COMPETITION_DIR, 'arc-agi_evaluation_challenges.json')
EVAL_SOLUTIONS = os.path.join(COMPETITION_DIR, 'arc-agi_evaluation_solutions.json')
CHALLENGES = TEST_CHALLENGES if HYBRID_RERUN else EVAL_CHALLENGES
WORK_DIR = '/kaggle/working'
SUBMISSION = os.path.join(WORK_DIR, 'submission.json')
HYBRID_DIR = os.path.join(WORK_DIR, 'hybrid')
P2_DIR = os.path.join(HYBRID_DIR, 'p2')
B0_SUBMISSION = os.path.join(HYBRID_DIR, 'submission_b0.json')
B0_POOL = os.path.join(HYBRID_DIR, 'b0_pool.json')
B0_OUTPUTS = '/kaggle/inference_outputs'
VLLM_SITE = os.path.join(WORK_DIR, 'vllm-site-packages')   # submit_main.SITE
HYBRID_LOG = []   # submission.json validity at every stage (hybrid/_notebook.json)

# commit run: the 5 dev tasks of splits/arc2_eval_split_v1.json with the least confident B0 output in the
# H1 run (fewer than 2 candidates, then the smallest kgmon margin); source: experiments/runs/kaggle-A2-H1-v1/pool (H1 run, Kaggle evaluation file)
SMOKE_TASK_IDS = ["13e47133", "16de56c4", "4e34c42c", "abc82100", "cbebaa4b"]

os.makedirs(P2_DIR, exist_ok=True)
try:
    with open(CHALLENGES) as _f:
        CHALLENGES_DATA = json.load(_f)
except Exception as exc:
    CHALLENGES_DATA = {}
    print('*** challenges not readable:', CHALLENGES, repr(exc))
HYBRID_TASK_IDS = sorted(CHALLENGES_DATA) if HYBRID_RERUN else [t for t in SMOKE_TASK_IDS if t in CHALLENGES_DATA]

# a valid submission.json before anything runs: B0's placeholder [[0]] for both attempts of every test input
if CHALLENGES_DATA:
    with open(SUBMISSION + '.tmp', 'w') as _f:
        json.dump({_t: [{'attempt_1': [[0]], 'attempt_2': [[0]]} for _ in CHALLENGES_DATA[_t]['test']]
                   for _t in CHALLENGES_DATA}, _f)
    os.replace(SUBMISSION + '.tmp', SUBMISSION)
    HYBRID_LOG.append({'stage': 'setup: placeholder', 'valid': True, 's': round(time.time() - NOTEBOOK_START_TIME, 1)})


# Phase 2 runs under Python 3.12 (the vLLM wheelhouse holds cp312 wheels; B0 keeps this image's Python).
# The interpreter is a standalone CPython build (python-build-standalone, install_only, x86_64 linux) shipped as
# a dataset and unpacked here; it is used only by the phase-2 subprocesses (install, server, client).
PY312_DIR = _kaggle_input('/kaggle/input/datasets/purushottamanurag/cpython-3-12-standalone-bin', '/kaggle/input/cpython-3-12-standalone-bin')
PY312_ARCHIVE = 'cpython-3.12.15-20261003-x86_64-unknown-linux-gnu-install_only.tar.gz.bin'
PY312_SHA256 = 'f937814031eab4698ca6d07ec606ede1825768f3f3e99af76d9db3900bee03c5'
PY312_VERSION = '3.12.15'
PY312_HOME = os.path.join(WORK_DIR, 'py312')   # unpacked here (removed by the final cell)
PY312_INFO = {'dataset': PY312_DIR}


def _clean_env(python):
    # environment for a phase-2 subprocess: none of this kernel's Python settings (PYTHONPATH, PIP_*, ...)
    _env = {_k: _v for _k, _v in os.environ.items()
            if not _k.startswith(('PYTHON', 'PIP_', 'CONDA', 'VIRTUAL_ENV'))}
    _env['PYTHONNOUSERSITE'] = '1'
    if python and python.startswith(PY312_HOME + os.sep):
        _env['PATH'] = os.path.dirname(python) + os.pathsep + os.environ.get('PATH', '')
    return _env


def _unpack_py312():
    # the dataset holds the release archive (or, if Kaggle extracted it on upload, its python/ tree)
    _exe = os.path.join(PY312_HOME, 'python', 'bin', 'python3.12')
    try:
        _arch = os.path.join(PY312_DIR, PY312_ARCHIVE)
        if not os.path.isfile(_arch):
            # Kaggle may rename an uploaded file (it drops '+'); the sha256 check below still applies
            _alt = sorted(glob.glob(os.path.join(PY312_DIR, '*install_only.tar.gz*')))
            if len(_alt) == 1 and os.path.isfile(_alt[0]):
                _arch = _alt[0]
                PY312_INFO['archive_name'] = os.path.basename(_arch)
        if os.path.isfile(_arch):
            import hashlib
            import tarfile
            _h = hashlib.sha256()
            with open(_arch, 'rb') as _f:
                for _chunk in iter(lambda: _f.read(1 << 24), b''):
                    _h.update(_chunk)
            PY312_INFO.update({'form': 'archive', 'sha256_ok': _h.hexdigest() == PY312_SHA256,
                               'bytes': os.path.getsize(_arch)})
            shutil.rmtree(PY312_HOME, ignore_errors=True)
            os.makedirs(PY312_HOME)
            with tarfile.open(_arch) as _tf:
                if hasattr(tarfile, 'tar_filter'):
                    _tf.extractall(PY312_HOME, filter='tar')
                else:
                    _tf.extractall(PY312_HOME)
        else:
            _src = None
            for _root, _dirs, _files in os.walk(PY312_DIR):
                if _root.endswith(os.path.join('python', 'bin')) and 'python3.12' in _files:
                    _src = os.path.dirname(os.path.dirname(_root))
                    break
                if _root.count(os.sep) - PY312_DIR.count(os.sep) > 3:
                    _dirs[:] = []
            if _src is None:
                PY312_INFO['error'] = f'neither {PY312_ARCHIVE} nor python/bin/python3.12 in {PY312_DIR}'
                return None
            PY312_INFO['form'] = 'extracted'
            shutil.rmtree(PY312_HOME, ignore_errors=True)
            shutil.copytree(os.path.join(_src, 'python'), os.path.join(PY312_HOME, 'python'), symlinks=True)
            for _d in ('bin', 'lib'):
                for _n in os.listdir(os.path.join(PY312_HOME, 'python', _d)):
                    _p = os.path.join(PY312_HOME, 'python', _d, _n)
                    if os.path.isfile(_p) and not os.path.islink(_p) and (_d == 'bin' or '.so' in _n):
                        os.chmod(_p, 0o755)
        _r = subprocess.run([_exe, '-c', 'import sys; print(sys.version.split()[0])'], capture_output=True,
                            text=True, timeout=300, env=_clean_env(_exe))
        PY312_INFO['version'] = _r.stdout.strip()
        if _r.returncode or not PY312_INFO['version'].startswith('3.12.'):
            PY312_INFO['error'] = f'{_exe}: exit {_r.returncode} {_r.stderr.strip()[-300:]}'
            return None
        return _exe
    except Exception as exc:
        PY312_INFO['error'] = repr(exc)[:500]
        return None


def _python312():
    # the standalone build first; else a Python 3.12 on the image
    _exe = _unpack_py312()
    if _exe:
        return _exe
    if sys.version_info[:2] == (3, 12):
        return sys.executable
    for _c in (shutil.which('python3.12'), '/usr/bin/python3.12', '/usr/local/bin/python3.12',
               '/opt/conda/bin/python3.12'):
        if _c and os.path.isfile(_c):
            return _c
    return None


P2_PYTHON = _python312()
# a separate interpreter must not see this kernel's Python settings (packages built for another Python)
P2_ENV = None if P2_PYTHON in (None, sys.executable) else _clean_env(P2_PYTHON)
print('phase-2 interpreter:', P2_PYTHON, PY312_INFO)

# phase 2: server, sampling and direct mode of madarshbb/arc-qwen-3-8-27b-agent version 2; as in the Q27F direct run: 20 in
# flight (= max_num_seqs 20); the plan (hybrid v2): band_fraction, band_samples, beyond_band
Q27_CONFIG = {'mode': 'direct',
 'thinking_token_budget': 40960,
 'max_tokens': 49152,
 'rescue_tokens': 8192,
 'template_kwargs': {'reasoning_effort': 'xhigh'},
 'temperature': 1.0,
 'top_p': 0.95,
 'top_k': 20,
 'parsers': ['--tool-call-parser', 'qwen3_coder', '--reasoning-parser', 'qwen3'],
 'max_model_len': 81920,
 'server_ladder': [['--reasoning-config',
                    '{"reasoning_start_str": "<think>", "reasoning_end_str": "</think>"}',
                    '--kv-cache-dtype',
                    'fp8',
                    '--language-model-only',
                    '--speculative-config',
                    '{"method": "qwen3_5_mtp", "num_speculative_tokens": 2}'],
                   ['--reasoning-config',
                    '{"reasoning_start_str": "<think>", "reasoning_end_str": "</think>"}',
                    '--kv-cache-dtype',
                    'fp8',
                    '--speculative-config',
                    '{"method": "qwen3_5_mtp", "num_speculative_tokens": 2}'],
                   ['--reasoning-config',
                    '{"reasoning_start_str": "<think>", "reasoning_end_str": "</think>"}',
                    '--kv-cache-dtype',
                    'fp8',
                    '--speculative-config',
                    '{"method": "mtp", "num_speculative_tokens": 2}'],
                   ['--reasoning-config',
                    '{"reasoning_start_str": "<think>", "reasoning_end_str": "</think>"}',
                    '--kv-cache-dtype',
                    'fp8'],
                   ['--reasoning-config',
                    '{"reasoning_start_str": "<think>", "reasoning_end_str": "</think>"}'],
                   []],
 'concurrency': 20,
 'max_num_seqs': 20,
 'server_wait_s': 1200,
 'port': 1234,
 'band_fraction': 0.25,
 'band_samples': 2,
 'beyond_band': True}
# version 6: the 27B engine in one place. Q27_CONFIG takes max_model_len, max_num_seqs and the server option sets
# from here and the CPU twin its installed versions; an engine swap edits this block and the two inputs it names
# (WHEELHOUSE_DIR, OVERLAY_DIR above). vLLM 0.19 with MTP as version 5; max_model_len 81,920 as version 5 (v6nb:
# no band sample, so version 6's 114,688 is not needed)
HYBRID_ENGINE = {'label': "vLLM 0.19.0 with MTP (2 draft tokens), FP8 KV cache: version 5's option sets",
 'wheelhouse': 'mirzamilanfarabi/arc3-vllm-h100-wheelhouse-v3',
 'overlay': 'keithtyser/taaf-duck-qwen38-serving-v1',
 'key_dists': ['vllm-0.19.0', 'transformers-5.8.0', 'huggingface_hub-1.5.0'],
 'max_model_len': 81920,
 'max_num_seqs': 20,
 'server_ladder': [['--reasoning-config',
                    '{"reasoning_start_str": "<think>", "reasoning_end_str": "</think>"}',
                    '--kv-cache-dtype',
                    'fp8',
                    '--language-model-only',
                    '--speculative-config',
                    '{"method": "qwen3_5_mtp", "num_speculative_tokens": 2}'],
                   ['--reasoning-config',
                    '{"reasoning_start_str": "<think>", "reasoning_end_str": "</think>"}',
                    '--kv-cache-dtype',
                    'fp8',
                    '--speculative-config',
                    '{"method": "qwen3_5_mtp", "num_speculative_tokens": 2}'],
                   ['--reasoning-config',
                    '{"reasoning_start_str": "<think>", "reasoning_end_str": "</think>"}',
                    '--kv-cache-dtype',
                    'fp8',
                    '--speculative-config',
                    '{"method": "mtp", "num_speculative_tokens": 2}'],
                   ['--reasoning-config',
                    '{"reasoning_start_str": "<think>", "reasoning_end_str": "</think>"}',
                    '--kv-cache-dtype',
                    'fp8'],
                   ['--reasoning-config', '{"reasoning_start_str": "<think>", "reasoning_end_str": "</think>"}'],
                   []]}
Q27_CONFIG.update({k: HYBRID_ENGINE[k] for k in ('max_model_len', 'max_num_seqs', 'server_ladder')},
                  engine=HYBRID_ENGINE['label'])
# v6nb: the band is off (no band sample; the plan is version 5's, hybrid_core.issue_plan: the quarter's first
# samples, its second samples, then the rest by size at the normal cap). Per-output grid parsing; B0's DFS record
# (arc_solver.py writes one line per task; v7's check 1, in the final cell, reads it) and H2's DFS seconds (phase 2's
# 'v6' line only: its check (1), version 6's DFS time against H2, does not apply to v7)
Q27_CONFIG['band_98k'] = {'on': False}
Q27_CONFIG['parser'] = 'per-output'
Q27_CONFIG['b0_dfs_log'] = os.path.join(HYBRID_DIR, 'b0_dfs.jsonl')
Q27_CONFIG['v6_reference'] = {'h2_dfs_mean_s': 320.3, 'h2_dfs_s': {"13e47133": 984.52, "16de56c4": 165.03, "4e34c42c": 30.36, "abc82100": 226.5, "cbebaa4b": 263.63}}
Q27_CONFIG.update({
    'server': 'stub' if globals().get('PREFLIGHT', False) else 'vllm',
    'model_glob': os.path.join(Q27_MODEL_DIR, 'config.json'),
    'overlay': os.path.join(OVERLAY_DIR, 'serving_wheels', '*.whl'),
    'wheelhouse_dir': WHEELHOUSE_DIR,
    'challenges': CHALLENGES,
    'out_dir': P2_DIR,
    'run_t0': NOTEBOOK_START_TIME,
    'b0_submission': B0_SUBMISSION,
    'submission': SUBMISSION,
    'stub_solutions': EVAL_SOLUTIONS if os.path.isfile(EVAL_SOLUTIONS) and not HYBRID_RERUN else '',
})
with open(os.path.join(P2_DIR, '_install_config.json'), 'w') as _f:
    json.dump(Q27_CONFIG, _f, indent=1)

# the written B0 scripts (starter.py, arc_solver.py) read these
os.environ['HYBRID_B0_MODEL_DIR'] = B0_MODEL_DIR
os.environ['HYBRID_CHALLENGES'] = CHALLENGES
os.environ['HYBRID_SMOKE_TASKS'] = ','.join(HYBRID_TASK_IDS)
os.environ['HYBRID_TTT_SEQUENCES'] = '64'
os.environ['HYBRID_B0_DFS_LOG'] = Q27_CONFIG['b0_dfs_log']   # version 6: B0's DFS record

_current_roots = ('/kaggle/input/competitions/', '/kaggle/input/datasets/', '/kaggle/input/models/')
for _name, _p in (('COMPETITION_DIR', COMPETITION_DIR), ('B0_MODEL_DIR', B0_MODEL_DIR),
                  ('WHEELHOUSE_DIR', WHEELHOUSE_DIR), ('OVERLAY_DIR', OVERLAY_DIR), ('Q27_MODEL_DIR', Q27_MODEL_DIR),
                  ('PY312_DIR', PY312_DIR), ('CHALLENGES', CHALLENGES)):
    print(f"input {_name}: {_p} [{'current' if _p.startswith(_current_roots) else 'legacy'} layout, "
          f"{'present' if os.path.exists(_p) else 'MISSING'}]")
print(f"tasks: {len(HYBRID_TASK_IDS)} of {len(CHALLENGES_DATA)} in {os.path.basename(CHALLENGES)}; python "
      f"{sys.version.split()[0]}; phase-2 python {P2_PYTHON or 'NONE (phase 2 will be skipped)'}")


phase-2 interpreter: /kaggle/working/py312/python/bin/python3.12 {'dataset': '/kaggle/input/datasets/purushottamanurag/cpython-3-12-standalone-bin', 'form': 'archive', 'sha256_ok': True, 'bytes': 66924371, 'version': '3.12.15'}
input COMPETITION_DIR: /kaggle/input/competitions/arc-prize-2026-arc-agi-2 [current layout, present]
input B0_MODEL_DIR: /kaggle/input/models/sorokin/qwen3_4b_grids15_sft139/transformers/bfloat16/1 [current layout, present]
input WHEELHOUSE_DIR: /kaggle/input/datasets/mirzamilanfarabi/arc3-vllm-h100-wheelhouse-v3 [current layout, present]
input OVERLAY_DIR: /kaggle/input/datasets/keithtyser/taaf-duck-qwen38-serving-v1 [current layout, present]
input Q27_MODEL_DIR: /kaggle/input/models/mikedan7/qwen3-8-27b-fp8-official/pytorch/hf-fp8/1 [current layout, present]
input PY312_DIR: /kaggle/input/datasets/purushottamanurag/cpython-3-12-standalone-bin [current layout, present]
input CHALLENGES: /kaggle/input/competitions/arc-prize-2026-arc-agi-2/arc-agi_evaluation_chal

In [3]:
# Preserve the baseline environment workaround.
!pip uninstall -y tensorflow

Found existing installation: tensorflow 2.18.0
Uninstalling tensorflow-2.18.0:
  Successfully uninstalled tensorflow-2.18.0


In [4]:
%%writefile arc_loader.py
import json
import numpy as np
from transformers import AutoTokenizer


def convert_grid_to_string(grid) -> str:
    text = ""
    for row in grid:
        for cell in row:
            text += str(int(cell))
        text += "\n"
    return text.strip()

def is_valid_solution(guess):
    return isinstance(guess, np.ndarray) and guess.ndim == 2 and all(0 < x <= 30 for x in guess.shape)

def shuffled(data_list):
    return np.random.permutation(data_list).tolist()

def permute_mod(a, descriptor, invert=False):
    permutation = [int(i) for i in descriptor if str(i).isdigit()]
    assert sorted(permutation)==list(range(10))
    a = np.asarray(a)
    if a.ndim==3:
        if not invert: permutation = np.argsort(permutation)
        a = a[..., permutation]
    else:
        assert a.ndim==2
        if invert: permutation = np.argsort(permutation)
        a = np.asarray(permutation)[a]
    return a

def permute_rnd_all_(query):
    permutation = np.random.permutation(10).tolist()
    return 'permute' + ''.join(map(str, permutation))


class QwenFormatter:

    def __init__(self, tokenizer: AutoTokenizer):
        self.tokenizer = tokenizer

    def fmt_query(self, query) -> str:
        grid_input = convert_grid_to_string(query[0]["input"])
        return "<|im_start|>user\n" + grid_input + "<|im_end|><|im_start|>assistant\n"

    def fmt_reply(self, reply) -> str:
        return convert_grid_to_string(reply[0]) + "<|im_end|>"

    def fmt_train(self, train, last_is_challenge=False) -> str:
        if last_is_challenge:
            test = train[-1]
            train = train[:-1]
        else:
            test = None
        text = ""
        for x in train:
            grid_input = convert_grid_to_string(x["input"])
            grid_output = convert_grid_to_string(x["output"])
            text += f"<|im_start|>user\n{grid_input}<|im_end|><|im_start|>assistant\n{grid_output}<|im_end|>"
        if test is not None:
            text += self.fmt_query([test]) + self.fmt_reply([test["output"]])
        return text

    def max_new_tokens(self):
        max_sized_reply = np.zeros([30, 30], dtype=int)
        tokens = self.tokenizer.encode(self.fmt_reply([max_sized_reply]))
        return len(tokens) + 1

    def convert_tokens_to_array(self, tokens, limit_rows=30):
        if len(tokens) < 2:
            return None
        text = self.tokenizer.decode(tokens[:-1])
        try:
            lines = text.strip().split("\n")
            by_rows = [row for row in [[int(x) for x in line if x.isdigit()] for line in lines] if len(row)]
            if len(by_rows) > limit_rows:
                by_rows = by_rows[:limit_rows]
            array = np.array(by_rows, dtype=int)
            if is_valid_solution(array):
                return array
        except:
            pass
        return None


class ArcDataset:

    @staticmethod
    def forward_mod(a, key, use_perm=True):
        if a is None: return a
        for op in key.split('.')[1:]:
            if   op=='rot90':              a = np.rot90(a)
            elif op=='transpose':          a = np.swapaxes(a, 0, 1)
            elif op.startswith('permute'): a = permute_mod(a, op, invert=False) if use_perm else a
            elif op.startswith('copy'):    a = np.copy(a)
            elif op.startswith('out'):     a = a
            elif op.startswith('ex'):      a = a
            elif op.startswith('run'):     a = a
            else: raise NotImplementedError(f"Inversion of operation '{op}' unknown.")
        return a

    @staticmethod
    def invert_mod(a, key, inv_perm=True):
        if a is None: return a
        for op in key.split('.')[1:][::-1]:
            if   op=='rot90':              a = np.rot90(a, k=3)
            elif op=='transpose':          a = np.swapaxes(a, 0, 1)
            elif op.startswith('permute'): a = permute_mod(a, op, invert=True) if inv_perm else a
            elif op.startswith('copy'):    a = np.copy(a)
            elif op.startswith('out'):     a = a
            elif op.startswith('ex'):      a = a
            elif op.startswith('run'):     a = a
            else: raise NotImplementedError(f"Inversion of operation '{op}' unknown.")
        return a

    def __init__(self, queries, replies={}, keys=None, is_orig=False):
        if keys is not None: keys = [k for k in keys if k is not None]
        self.queries = queries if keys is None else {k: queries[k] for k in keys}
        self.replies = replies if keys is None else {k: replies[k] for k in keys if k in replies}
        self.is_orig = is_orig
        self.keys = sorted(queries.keys()) if keys is None else keys
        self.transposed_dataset = None

    def change_keys(self, keys, keep_flags=False):
        flags = dict(is_orig=self.is_orig) if keep_flags else {}
        return self.__class__(queries=self.queries, replies=self.replies, keys=keys, **flags)

    @classmethod
    def from_file(cls, queries_file, keys=None):
        with open(queries_file) as f:
            queries = f.read()
        return cls(
            queries=json.loads(queries),
            is_orig=True,
            keys=keys,
        )

    def load_replies(self, replies_file):
        print(f"*** Load solutions from '{replies_file}'...")
        with open(replies_file) as f: replies = f.read()
        replies_parsed = json.loads(replies)
        self.replies = {k: replies_parsed[k] for k in self.keys}
        return self

    def split_multi_replies(self):
        key_indices = [(k, i) for k in self.keys for i in range(len(self.queries[k]['test']))]
        return self.__class__(
            keys=[f'{k}_{i}' for k, i in key_indices],
            queries={f'{k}_{i}': {'train': self.queries[k]['train'], 'test': [self.queries[k]['test'][i]]} for k, i in key_indices},
            replies={f'{k}_{i}': [self.replies[k][i]] for k, i in key_indices if k in self.replies},
        )

    def shuffled(self):
        return self.__class__(queries=self.queries, replies=self.replies, keys=shuffled(self.keys))

    def append(*datasets):
        return datasets[0].__class__(
            queries={k: v for d in datasets for k, v in d.queries.items()},
            replies={k: v for d in datasets for k, v in d.replies.items()},
            keys   =[k    for d in datasets for k    in d.keys           ],
        )

    def mod_single(self, mod_func, descriptor, i, keep_key, inputs_only):
        queries = {}
        replies = {}
        keys    = []
        for k0 in self.keys:
            desc = (('copy{i}' if mod_func is np.copy else mod_func.__name__) if descriptor is None else descriptor if isinstance(descriptor, str) else descriptor(self.queries[k0])).format(i=i)
            func = lambda a, d: np.asarray(mod_func(a) if descriptor is None else mod_func(a, d)).tolist()
            k1 = k0 if keep_key else f"{k0}.{'I' if inputs_only else ''}{desc}"
            keys.append(k1)
            queries[k1] = {m: [{t: (func(a, desc) if t=='input' or not inputs_only else a) for t, a in x.items()} for x in e] for m, e in self.queries[k0].items()}
            if k0 in self.replies:
                replies[k1] = [func(a, desc) for a in self.replies[k0]]
        ret = self.__class__(queries=queries, replies=replies, keys=keys)
        return ret

    def mod(self, mod_func, descriptor=None, n=1, stack=None, keep=False, keep_key=False, shuffle=False, join=True, inputs_only=False):
        assert not (keep and keep_key)
        cur = self
        ret = [cur.shuffled() if shuffle else cur] if keep else []
        if stack is None: stack = mod_func.__name__.startswith('rot')
        for i in range(n):
            cur = (cur if stack else self).mod_single(mod_func, descriptor, i=i, keep_key=keep_key, inputs_only=inputs_only)
            ret.append(cur.shuffled() if shuffle else cur)
        return self.__class__.append(*ret) if join else ret

    def get(self, key, formatter: QwenFormatter):
        train = formatter.fmt_train(self.queries[key]['train'])
        query = formatter.fmt_query(self.queries[key]['test'])
        reply = formatter.fmt_reply(self.replies[key]) if key in self.replies else ''
        text = train+query+reply if reply else formatter.fmt_train(self.queries[key]['train'], last_is_challenge=True)
        return dict(key=key, train=train, query=query, reply=reply, input=train+query, text=text)

    def as_list(self, formatter: QwenFormatter):
        return [self.get(key, formatter) for key in self.keys]

    def get_length(self, key, formatter: QwenFormatter, name, max_of_transposed=False):
        if formatter is None:
            if   name=='input': return sum(np.prod(np.shape(v)) for v3 in self.queries[key].values() for v2 in v3 for v in v2.values())
            elif name=='reply': return sum(np.prod(np.shape(v)) for v in self.replies[key])
            else: assert False
        else:
            datasets = [self]
            if max_of_transposed:
                if self.transposed_dataset is None: self.transposed_dataset = self.mod(np.transpose, keep=False, keep_key=True)
                datasets.append(self.transposed_dataset)
            return max(len(formatter.tokenizer.encode(ds.get(key, formatter=formatter)[name])) for ds in datasets)

    def cut_to_len(self, formatter, name, max_len, from_end=False):
        temp_ds = self.change_keys(self.keys)
        new_keys = []
        new_queries = {}
        new_replies = {}
        for key in self.keys:
            reply = temp_ds.replies.get(key)
            while max_len<temp_ds.get_length(key, formatter=formatter, name=name):
                query = temp_ds.queries[key]
                if not key.split('.')[-1].startswith('ex'):
                    key = f"{key}.ex{''.join(map(str, range(len(query['train']))))}"
                key_split = key.split('.')
                assert key_split[-1].startswith('ex')
                key = '.'.join(key_split[:-1] + [f'ex{key_split[-1][2:-1] if from_end else key_split[-1][3:]}'])
                temp_ds.queries[key] = {k: ((v[:-1] if from_end else v[1:]) if k=='train' else v) for k, v in query.items()}
                if reply is not None:
                    temp_ds.replies[key] = reply
            new_keys.append(key)
            new_queries[key] = temp_ds.queries[key]
            if reply is not None: new_replies[key] = reply
        return self.__class__(keys=new_keys, queries=new_queries, replies=new_replies)
    
    def shuffle_ex(self, perm=None, keep_max=None):
        new_keys = []
        new_queries = {}
        new_replies = {}
        for key in self.keys:
            n = len(self.queries[key]['train'])
            p = np.random.permutation(n) if perm is None else perm
            if keep_max is not None: p = p[:keep_max]
            new_key = f'{key}.ex' + ('-' if (p.max()>9) else '').join(map(str, p.tolist()))
            new_keys.append(new_key)
            new_queries[new_key] = {k: (np.array(v, dtype=object)[p].tolist() if k=='train' else v) for k, v in self.queries[key].items()}
            if key in self.replies: new_replies[new_key] = self.replies[key]
        return self.__class__(queries=new_queries, replies=new_replies, keys=new_keys)

    def augment(self, n=1, shfl_keys=False, seed=42):
        np.random.seed(seed)
        d = self
        d = d.mod(np.transpose, keep=True)
        d = d.mod(np.rot90, n=3, keep=True)
        d = d.mod(permute_mod, permute_rnd_all_, n=n, shuffle=shfl_keys, keep=False)
        d = d.shuffle_ex()
        return d

    def get_submission(self, results=None):
        assert self.is_orig==True, 'Must be run on original dataset.'
        submission = {k: [{f'attempt_{i+1}': [[0]] for i in range(2)} for _ in range(len(self.queries[k]['test']))] for k in self.keys}
        if results is not None: self.fill_submission(results, submission)
        return submission

    @staticmethod
    def fill_submission(results, submission):
        print(f'*** Generating submission for {len(results)} outputs...')
        for k, v in results.items():
            base_id, base_nr = k.split('_')
            target_dict = submission[base_id][int(base_nr)]
            for i, g in enumerate(v[:len(target_dict)]):
                target_dict[f'attempt_{i+1}'] = g.tolist()

    def validate_submission(self, submission):
        assert self.is_orig==True, 'Must be run on original dataset.'
        score = 0
        for k, v in self.replies.items():
            for i, r in enumerate(v):
                for attempt in ['attempt_1', 'attempt_2']:
                    if np.array_equal(r, submission[k][i][attempt]):
                        score += 1 / len(v)
                        break
        return score

Writing arc_loader.py


In [5]:
%%writefile arc_decoder.py
import os
import bz2
import pickle
import numpy as np

def hashable(guess):
    return tuple(map(tuple, guess))

def score_sum(guesses, getter):
    guess_list = list(guesses.values())
    scores = {}
    for g in guess_list:
        h = hashable(g["solution"])
        x = scores[h] = scores.get(h, [[], g["solution"]])
        x[0].append(g)
    scores = [(getter(sc), o) for sc, o in scores.values()]
    scores = sorted(scores, key=(lambda x: x[0]), reverse=True)
    ordered_outputs = [x[-1] for x in scores]
    return ordered_outputs

def getter_full_probmul_3(guesses, baseline=3):
    inf_score = np.sum([baseline-g["beam_score"] for g in guesses])
    aug_score = np.mean([np.sum([baseline-s for s in g["score_aug"]]) for g in guesses])
    return inf_score + aug_score

def score_full_probmul_3(guesses):
    return score_sum(guesses, getter_full_probmul_3)

def getter_kgmon(guesses):
    inf_score = len(guesses)
    aug_score = np.mean([np.mean(g["score_aug"]) for g in guesses])
    return inf_score - aug_score

def score_kgmon(guesses):
    return score_sum(guesses, getter_kgmon)


selection_algorithms = [
    score_full_probmul_3,
    score_kgmon,
]


class ArcDecoder:
    
    def __init__(self, dataset, n_guesses):
        self.dataset = dataset
        self.n_guesses = n_guesses
        self.decoded_results = {}

    def load_decoded_results(self, store, run_name=""):
        for key in os.listdir(store):
            with bz2.BZ2File(os.path.join(store, key)) as f:
                outputs = pickle.load(f)
            base_key = key.split(".")[0]
            self.decoded_results[base_key] = self.decoded_results.get(base_key, {})
            for i, sample in enumerate(outputs):
                self.decoded_results[base_key][f"{key}{run_name}.out{i}"] = sample

    def run_selection_algo(self, selection_algorithm=score_kgmon):
        return {bk: selection_algorithm({k: g for k, g in v.items()}) for bk, v in self.decoded_results.items()}

    def benchmark_selection_algos(self):
        print("*** Benchmark selection algorithms...")

        labels = {}
        num_tasks_per_puzzle = {}
        num_solved_keys = 0
        num_total_keys = 0

        correct_beam_scores = []

        for basekey, basevalues in self.decoded_results.items():

            mult_key, mult_sub = basekey.split("_")
            num_tasks_per_puzzle[mult_key] = max(num_tasks_per_puzzle.get(mult_key, 0), int(mult_sub) + 1)

            labels[basekey] = correct_solution = self.dataset.replies[basekey][0]

            for subkey, sample in basevalues.items():

                solution = sample["solution"]
                beam_score = sample["beam_score"]
                aug_mean = np.mean(sample["score_aug"])

                if np.shape(correct_solution) != np.shape(solution):
                    corr_str = "bad_xy_size"
                elif np.array_equal(correct_solution, solution):
                    corr_str = "ALL_CORRECT"
                    num_solved_keys += 1
                    correct_beam_scores.append(beam_score)
                else:
                    corr_str = "bad_content"

                output_len = f"{solution.shape[0]}x{solution.shape[1]}"

                if corr_str == "ALL_CORRECT":
                    print(f"{corr_str}:{beam_score:8.5f} - {aug_mean:8.5f} {output_len:5s} [{subkey}]")
                num_total_keys += 1

        print(f" subkeys: {num_solved_keys}/{num_total_keys}")
        print(f" avg correct beam score: {np.mean(correct_beam_scores):8.5f}")
        print(f" max correct beam score: {np.max(correct_beam_scores):8.5f}")

        num_puzzles = len(num_tasks_per_puzzle)

        for selection_algorithm in selection_algorithms:
            name = selection_algorithm.__name__
            selected = self.run_selection_algo(selection_algorithm)
            correct_puzzles = {k for k, v in selected.items() if any(np.array_equal(guess, labels[k]) for guess in v[:self.n_guesses])}
            print(correct_puzzles)
            score = sum(1/num_tasks_per_puzzle[k.split("_")[0]] for k in correct_puzzles)
            print(f" acc: {score:5.1f}/{num_puzzles:3} ('{name}')")

Writing arc_decoder.py


In [6]:
%%writefile arc_solver.py
from unsloth import FastLanguageModel, UnslothTrainingArguments, UnslothTrainer
from arc_loader import ArcDataset, QwenFormatter

import gc
import json  # hybrid: B0's DFS record
import os
import io
import time
import torch
import numpy as np
from tqdm import tqdm
from datasets import Dataset
from collections import defaultdict

from typing import Any, Union
from transformers import DataCollatorForLanguageModeling

import logging
from contextlib import redirect_stdout, redirect_stderr

from peft import get_peft_model_state_dict, set_peft_model_state_dict

import bz2
import pickle

logging.disable(logging.WARNING)

ARC_VOCAB = {
    "0": 0,
    "1": 1,
    "2": 2,
    "3": 3,
    "4": 4,
    "5": 5,
    "6": 6,
    "7": 7,
    "8": 8,
    "9": 9,
    "Ċ": 10,
    "<|im_end|>": 15,
}

ARC_TOKENS = list(ARC_VOCAB.values())
USER_TOKEN_ID = 11
ASSISTANT_TOKEN_ID = 12
PAD_ID = 13
EOS_ID = 15


class UnslothFixedTrainer(UnslothTrainer):

    # Issue https://github.com/unslothai/unsloth/issues/2435

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        """Fixed compute_loss that handles Unsloth's view tensor issue"""
        if self.label_smoother is not None and "labels" in inputs:
            labels = inputs.pop("labels")
        else:
            labels = None
        outputs = model(**inputs)
        if labels is not None:
            unwrapped_model = self.accelerator.unwrap_model(model)
            if hasattr(unwrapped_model, "_get_name") and "unsloth" in unwrapped_model._get_name().lower():
                loss = self.label_smoother(outputs, labels, shift_labels=True)
            else:
                loss = self.label_smoother(outputs, labels)
        else:
            loss = outputs["loss"] if isinstance(outputs, dict) else outputs[0]
        # 🔧 KEY FIX: Clone the loss tensor before in-place operations
        if hasattr(loss, "clone"):
            loss = loss.clone()  # Converts view tensor to independent tensor
        # Now safe for DDP gradient scaling
        if self.accelerator.num_processes > 1:
            loss = loss * self.accelerator.num_processes
        return (loss, outputs) if return_outputs else loss


class QwenDataCollatorForCompletionOnlyLM(DataCollatorForLanguageModeling):

    def torch_call(self, examples: list[Union[list[int], Any, dict[str, Any]]]) -> dict[str, Any]:
        batch = super().torch_call(examples)
        for i in range(len(examples)):
            labels = batch["input_ids"][i].clone()
            user_start_idx = np.where(labels == USER_TOKEN_ID)[0].tolist()
            assistant_start_idx = np.where(labels == ASSISTANT_TOKEN_ID)[0].tolist()
            start_idx = sorted(user_start_idx + assistant_start_idx)
            end_idx = np.where(labels == EOS_ID)[0]
            batch["labels"][i, :] = -100
            for j, (start, end) in enumerate(zip(start_idx, end_idx)):
                assert start < end
                if j % 2 == 1:
                    start += 2
                    end += 1
                    batch["labels"][i, start:end] = labels[start:end]
        return batch


# Minimal performance patch: preserve the baseline beam set and ranking, but transfer
# only the 12 ARC-token NLL values to CPU instead of every Qwen vocabulary logit.
_ARC_TOKEN_ID_CACHE = {}


def _arc_token_ids(device):
    key = str(device)
    token_ids = _ARC_TOKEN_ID_CACHE.get(key)
    if token_ids is None:
        token_ids = torch.tensor(ARC_TOKENS, dtype=torch.long, device=device)
        _ARC_TOKEN_ID_CACHE[key] = token_ids
    return token_ids


def turbo_dfs(model, logits, max_new_tokens, max_score, scores, pos, cache, start_time, end_time) -> dict:

    n = logits.size(0)

    # Algebraically identical to: scores - logits.float().cpu().log_softmax(-1),
    # restricted to the same ARC_TOKENS used by the baseline DFS loop.
    logits_f = logits.float()
    token_ids = _arc_token_ids(logits.device)
    arc_logits = logits_f.index_select(-1, token_ids)
    nll = (
        torch.as_tensor(scores, dtype=torch.float32, device=logits.device).view(n, 1)
        + torch.logsumexp(logits_f, dim=-1, keepdim=True)
        - arc_logits
    ).cpu()

    suffixes = defaultdict(list)

    candidates = dict()

    for i in range(n):
        candidates[i] = []
        for token_idx, t in enumerate(ARC_TOKENS):
            score = nll[i, token_idx].item()
            if score < max_score:
                if t == EOS_ID:
                    suffixes[i].append((score, [t]))
                elif max_new_tokens > 1:
                    candidates[i].append((score, t))

    for i in range(n):
        candidates[i] = sorted(candidates[i], key=lambda x:x[0]) #[:5]
    
    while time.time() - start_time < 540 and time.time() < end_time:

        batch_tokens = []
        batch_scores = []
        num_alive_beams = 0

        for i in range(n):
            if len(candidates[i]) == 0:
                batch_tokens.append(PAD_ID)
                batch_scores.append(1000)
            else:
                score, t = candidates[i].pop(0)
                batch_tokens.append(t)
                batch_scores.append(score)
                num_alive_beams += 1

        if num_alive_beams == 0:
            break

        outputs = model(
            input_ids=torch.tensor(batch_tokens, device=model.device, dtype=torch.long).view(-1, 1),
            position_ids=torch.full((n, 1), pos, device=model.device),
            past_key_values=cache,
            return_dict=True,
            use_cache=True,
        )

        next_suffixes = turbo_dfs(
            model,
            logits=outputs.logits[:, -1],
            max_new_tokens=max_new_tokens-1,
            max_score=max_score,
            scores=batch_scores,
            pos=pos+1,
            cache=outputs.past_key_values,
            start_time=start_time,
            end_time=end_time,
        )

        for batch_id, beams in next_suffixes.items():
            for score, suffix_tokens in beams:
                suffix_tokens.insert(0, batch_tokens[batch_id])
                suffixes[batch_id].append((score, suffix_tokens))

    return suffixes


@torch.no_grad()
def inference_turbo_dfs(model, prefix_tokens, max_new_tokens, max_score, end_time):
    input_ids = torch.tensor(prefix_tokens, device=model.device, dtype=torch.long)
    outputs = model(input_ids=input_ids, return_dict=True, use_cache=True)
    suffixes = turbo_dfs(
        model,
        logits=outputs.logits[:, -1],
        max_new_tokens=max_new_tokens,
        max_score=max_score,
        scores=[0.0] * input_ids.size(0),
        pos=input_ids.size(1),
        cache=outputs.past_key_values,
        start_time=time.time(),
        end_time=end_time,
    )
    result = []
    for batch_id, beams in suffixes.items():
        sorted_beams = sorted(beams, key=lambda x:x[0])
        result.append((batch_id, sorted_beams))
    return result


@torch.no_grad()
def calc_scores(queries, answers, tokenizer, model):
    batch_query_tokens = []
    batch_answer_tokens = []
    batch_tokens = []
    batch_lengths = []
    for query, answer in zip(queries, answers):
        query_tokens = tokenizer.encode(query)
        answer_tokens = tokenizer.encode(answer)
        tokens = query_tokens + answer_tokens
        batch_query_tokens.append(query_tokens)
        batch_answer_tokens.append(answer_tokens)
        batch_tokens.append(tokens)
        batch_lengths.append(len(tokens))
    max_len = max(batch_lengths)
    padded_tokens = []
    for tokens in batch_tokens:
        padded = tokens + [PAD_ID] * (max_len - len(tokens))
        padded_tokens.append(padded)
    input_ids = torch.tensor(padded_tokens, device=model.device, dtype=torch.long)

    # Keep logits on GPU and gather only the target-token scores. KV cache is not
    # consumed by teacher-forced scoring, so disabling it removes redundant writes.
    outputs = model(input_ids=input_ids, return_dict=True, use_cache=False)
    batch_logits = outputs.logits.float()
    batch_log_norm = torch.logsumexp(batch_logits, dim=-1)
    result = []
    for row_id, (query_tokens, answer_tokens) in enumerate(zip(batch_query_tokens, batch_answer_tokens)):
        query_length = len(query_tokens)
        answer_length = len(answer_tokens)
        positions = torch.arange(
            query_length - 1,
            query_length - 1 + answer_length,
            device=model.device,
        )
        target_tokens = torch.tensor(answer_tokens, device=model.device, dtype=torch.long)
        answer_log_probs = (
            batch_logits[row_id, positions, target_tokens]
            - batch_log_norm[row_id, positions]
        )
        result.append(-answer_log_probs.sum().item())
    return result


def worker(rank, queue, end_time):

    rerun_mode = os.getenv("KAGGLE_IS_COMPETITION_RERUN")

    peft_params = dict(
        r=256,
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj", "embed_tokens", "lm_head"],
        lora_alpha=32,
        lora_dropout=0.0,
        bias="none",
        use_gradient_checkpointing=False,
        random_state=42,
        use_rslora=True,
        loftq_config=None,
    )

    train_args = dict(
        per_device_eval_batch_size=1,
        per_device_train_batch_size=1,
        gradient_accumulation_steps=1,
        num_train_epochs=1,
        warmup_steps=0,
        warmup_ratio=0.1,
        max_grad_norm=1.0,
        learning_rate=5e-5,
        optim="adamw_torch",
        weight_decay=0.0,
        lr_scheduler_type="cosine",
        seed=42,
        report_to="none",
        save_strategy="no",
        eval_strategy="no",
        logging_strategy="no",
        fp16=False,
        bf16=True,
        # Disable FSDP (use standard DDP)
        fsdp="",
        ddp_find_unused_parameters=False,
        dataloader_num_workers=0,
        gradient_checkpointing=False,
    )

    max_seq_length = 8192

    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=os.environ["HYBRID_B0_MODEL_DIR"],  # hybrid: resolved by _kaggle_input in the setup cell
        full_finetuning=False,
        load_in_4bit=False,
        local_files_only=True,
        use_gradient_checkpointing=False,
        max_seq_length=max_seq_length,
    )

    model = FastLanguageModel.get_peft_model(model, **peft_params)

    for name, param in model.named_parameters():
        if param.dtype == torch.float32:
            param.data = param.data.to(torch.bfloat16)

    default_weights = get_peft_model_state_dict(model, adapter_name="default")
    default_weights = {k: v.clone().detach() for k, v in default_weights.items()}

    collator = QwenDataCollatorForCompletionOnlyLM(
        tokenizer=tokenizer,
        mlm=False,
    )

    formatter = QwenFormatter(tokenizer=tokenizer)

    max_new_tokens = formatter.max_new_tokens()

    max_score = -np.log(0.2)

    # hybrid: test challenges in the rerun, evaluation challenges in a commit run (resolved in the setup cell)
    test_path = os.environ["HYBRID_CHALLENGES"]

    arc_test_set = ArcDataset.from_file(test_path)

    dir_outputs = "/kaggle/inference_outputs"
    os.makedirs(dir_outputs, exist_ok=True)

    while not queue.empty():

        if time.time() > end_time:
            print(f"[Rank {rank}] stop!")
            break

        key = queue.get()
        if key is None:
            break
        
        start_time = time.time()
        
        torch.cuda.reset_peak_memory_stats()

        load_result = set_peft_model_state_dict(
            model,
            default_weights.copy(),
            adapter_name="default",
        )

        model = FastLanguageModel.for_training(model)

        puzzle_ds = arc_test_set.change_keys([key])

        train_ds = puzzle_ds.augment(n=8, shfl_keys=True, seed=1)  # hybrid (H1): 64 training sequences (B0: n=16, 128)
        if len(train_ds.keys) != int(os.environ["HYBRID_TTT_SEQUENCES"]):
            raise RuntimeError(f"hybrid: {len(train_ds.keys)} training sequences != HYBRID_TTT_SEQUENCES {os.environ['HYBRID_TTT_SEQUENCES']}")
        train_ds = train_ds.cut_to_len(formatter=formatter, name="text", max_len=max_seq_length)

        with io.StringIO() as buf, redirect_stdout(buf), redirect_stderr(buf):
            
            trainer = UnslothFixedTrainer(
                model=model,
                tokenizer=tokenizer,
                data_collator=collator,
                train_dataset=Dataset.from_list(train_ds.as_list(formatter)),
                dataset_text_field="text",
                max_seq_length=max_seq_length,
                args=UnslothTrainingArguments(**train_args),
            )

            stats = trainer.train()

            model = trainer.accelerator.unwrap_model(model, keep_fp32_wrapper=False)

            del trainer

        model = FastLanguageModel.for_inference(model)
        
        gc.collect()
        torch.cuda.empty_cache()
            
        memory_allocated = torch.cuda.max_memory_allocated() // 1024**2
        print(f"[Rank {rank}] allocated {memory_allocated}MB for training")

        torch.cuda.reset_peak_memory_stats()
        
        print(f"[Rank {rank}] training stats for puzzle {key}: {stats}")

        puzzle_ds_multi = puzzle_ds.split_multi_replies()

        eval_ds = puzzle_ds_multi.augment(n=2, seed=2)
        eval_ds = eval_ds.cut_to_len(formatter=formatter, name="input", max_len=max_seq_length-max_new_tokens)

        test_id_to_subkeys = defaultdict(list)
        for subkey in sorted(eval_ds.keys):
            test_id = subkey.split(".")[0].split("_")[1]
            test_id_to_subkeys[test_id].append(subkey)

        batches = []
        for test_id, subkeys in test_id_to_subkeys.items():
            # 0: permute x 2
            # 4: rot90.rot90.permute x 2
            batch = []
            for offset in [0, 4]:
                batch.extend(subkeys[offset:offset+2])
            batches.append(batch)
            # 2: permute.rot90 x 2
            # 6: rot90.rot90.rot90.permute x 2
            batch = []
            for offset in [2, 6]:
                batch.extend(subkeys[offset:offset+2])
            batches.append(batch)
        for test_id, subkeys in test_id_to_subkeys.items():
            # 8: transpose.permute x 2
            # 12: transpose.rot90.rot90.permute x 2
            batch = []
            for offset in [8, 12]:
                batch.extend(subkeys[offset:offset+2])
            batches.append(batch)
            # 10: transpose.rot90.permute x 2
            # 14: transpose.rot90.rot90.rot90.permute x 2
            batch = []
            for offset in [10, 14]:
                batch.extend(subkeys[offset:offset+2])
            batches.append(batch)

        with torch.inference_mode():
                
            known_scores = {}
            _v6_dfs_s, _v6_dfs_batches = 0.0, 0  # hybrid: DFS seconds and batches decoded for this task

            for subkeys in batches:

                spend_time = time.time() - start_time
                if spend_time > 1200 or time.time() > end_time:
                    print(f"[Rank {rank}] timeout after {spend_time:.1f}s for puzzle {key}")
                    break

                print(f"[Rank {rank}] decoding {subkeys}")

                tokens = []
                for subkey in subkeys:
                    data = eval_ds.get(subkey, formatter)
                    tokens.append(tokenizer.encode(data["input"]))

                _v6_t = time.time()
                dfs_result = inference_turbo_dfs(model, tokens, max_new_tokens, max_score, end_time)
                _v6_dfs_s += time.time() - _v6_t
                _v6_dfs_batches += 1

                for subkey_id, scored_beams in dfs_result:

                    subkey = subkeys[subkey_id]
                    bk = subkey.split(".")[0]
                    decoded_result = []

                    for beam_score, tokens in scored_beams:

                        array = formatter.convert_tokens_to_array(tokens)
                        if array is None:
                            continue

                        solution = puzzle_ds_multi.invert_mod(array, subkey, inv_perm=True)

                        grid_id = (bk, tuple(map(tuple, solution)))

                        if grid_id in known_scores:
                            augmented_scores = known_scores[grid_id]
                        else:
                            print(f"[Rank {rank}] scoring {subkey} #{len(decoded_result)}")
                            aug_dataset = ArcDataset(
                                keys=[bk],
                                queries={bk: puzzle_ds_multi.queries.get(bk)},
                                replies={bk: [solution.tolist()]},
                            )
                            aug_dataset = aug_dataset.augment(seed=hash(bk) % 1024**2)
                            aug_dataset = aug_dataset.cut_to_len(formatter=formatter, name="input", max_len=max_seq_length-max_new_tokens)
                            aug_queries = []
                            aug_answers = []
                            for augmented_sample in aug_dataset.as_list(formatter):
                                aug_queries.append(augmented_sample["input"])
                                aug_answers.append(augmented_sample["reply"])
                            augmented_scores1 = calc_scores(aug_queries[:4], aug_answers[:4], tokenizer, model)
                            augmented_scores2 = calc_scores(aug_queries[4:], aug_answers[4:], tokenizer, model)
                            augmented_scores = augmented_scores1 + augmented_scores2
                            known_scores[grid_id] = augmented_scores
                        
                        decoded_result.append({
                            "beam_score": beam_score,
                            "score_aug": augmented_scores,
                            "solution": solution,
                        })

                    if len(decoded_result):
                        with bz2.BZ2File(os.path.join(dir_outputs, subkey), "w") as f:
                            pickle.dump(decoded_result, f)

        memory_allocated = torch.cuda.max_memory_allocated() // 1024**2
        print(f"[Rank {rank}] allocated {memory_allocated}MB for inference")
        
        spend_time = time.time() - start_time
        print(f"[Rank {rank}] finished {key} in {spend_time:.1f}s")
        # hybrid: this task's DFS seconds and batches (4 per test output, 2 of them transposed), in the log and as
        # one JSON line in HYBRID_B0_DFS_LOG (phase 2 and the final cell read it)
        _v6_rec = {"task_id": key, "rank": rank, "outputs": len(test_id_to_subkeys), "batches_built": len(batches),
                   "batches_decoded": _v6_dfs_batches,
                   "transposed_batches": sum(1 for b in batches if any(".transpose" in s for s in b)),
                   "dfs_s": round(_v6_dfs_s, 2), "task_s": round(spend_time, 2), "end_time_reached": time.time() > end_time}
        print(f"[Rank {rank}] hybrid dfs {key}: {_v6_dfs_batches} of {len(batches)} batches for "
              f"{len(test_id_to_subkeys)} outputs, {_v6_dfs_s:.1f}s DFS")
        try:
            with open(os.environ["HYBRID_B0_DFS_LOG"], "a") as _f:
                _f.write(json.dumps(_v6_rec) + "\n")
        except Exception as exc:
            print(f"[Rank {rank}] hybrid dfs record not written: {exc!r}")

Writing arc_solver.py


In [7]:
%%writefile starter.py
import os
import time
import json
import torch
import argparse
import torch.multiprocessing as mp


def local_worker(rank, queue, end_time):
    
    os.environ["CUDA_VISIBLE_DEVICES"] = str(rank)

    torch.set_default_device("cpu")

    # Fix Unsloth patching issue
    if rank > 0:
        while not os.path.exists(f"/kaggle/worker{rank-1}"):
            time.sleep(5)
    
    from arc_solver import worker

    with open(f"/kaggle/worker{rank}", "w") as f:
        f.write("Ok")
    
    print(f"[Rank {rank}] start!")
    
    worker(rank, queue, end_time)
    
    print(f"[Rank {rank}] done!")


if __name__ == "__main__":

    parser = argparse.ArgumentParser()
    parser.add_argument("--end-time", type=float, default=0.0)
    args = parser.parse_args()

    rerun_mode = os.getenv("KAGGLE_IS_COMPETITION_RERUN")

    # hybrid: test challenges in the rerun, evaluation challenges in a commit run (resolved in the setup cell)
    test_path = os.environ["HYBRID_CHALLENGES"]
    smoke_tasks = os.environ["HYBRID_SMOKE_TASKS"].split(",")  # commit run: the 5 tasks of the setup cell

    with open(test_path, "r") as f:
        data = json.load(f)

    queue = mp.Manager().Queue()

    for key in sorted(data.keys()):
        if not rerun_mode:
            if key not in smoke_tasks:
                continue
        queue.put(key)
    for _ in range(4):
        queue.put(None)
    
    mp.spawn(local_worker, args=(queue, args.end_time), nprocs=4)

Writing starter.py


In [8]:
%%writefile /kaggle/working/hybrid/grids.py
"""Grid <-> text, task prompts and failure feedback for the program-synthesis agent.

Grids are shown as rows of digits with no separators, the same layout NVARC's
formatter uses, plus explicit dimensions -- reasoning models miscount rows and
columns of long digit strings otherwise.
"""

SYSTEM = (
    "You are an expert at solving ARC-AGI puzzles by writing Python programs. "
    "Each puzzle has a few input->output example pairs that share one hidden "
    "transformation rule. Grids are 2D lists of integers 0-9 (colours); 0 is "
    "usually the background."
)

INSTRUCTIONS = """Find the transformation rule that maps EVERY example input to its output, then write it as a Python function.

Requirements:
- Define `def transform(grid: list[list[int]]) -> list[list[int]]:`.
- Use only the Python standard library and numpy (already importable as `np` if you `import numpy as np`).
- The function must work for every example input AND the test input(s); do not hard-code example outputs.
- Return a new 2D list of ints 0-9 (numpy arrays are accepted).

Reason about the rule first, then give the final program in ONE ```python code block."""


def grid_text(g):
    return "\n".join("".join(str(int(v)) for v in row) for row in g)


def shape(g):
    return (len(g), len(g[0]) if g else 0)


def show(g, name):
    r, c = shape(g)
    return f"{name} ({r}x{c}):\n{grid_text(g)}"


def task_prompt(task):
    parts = [INSTRUCTIONS, ""]
    for i, pair in enumerate(task["train"], 1):
        parts.append(f"## Example {i}")
        parts.append(show(pair["input"], "Input"))
        parts.append(show(pair["output"], "Output"))
        parts.append("")
    for j, t in enumerate(task["test"], 1):
        parts.append(f"## Test input {j}")
        parts.append(show(t["input"], "Input"))
        parts.append("")
    return "\n".join(parts).rstrip()


def diff_text(expected, got, max_cells=900):
    """Expected grid with wrong cells marked, so the model sees WHERE it failed.

    '.' marks a correct cell; a wrong cell shows `expected/got` inline is too
    wide, so two aligned grids are given: the expected output, and the
    program's output with correct cells replaced by '.'.
    """
    re_, ce = shape(expected)
    rg, cg = shape(got)
    if (re_, ce) != (rg, cg):
        return (f"Wrong output SIZE: expected {re_}x{ce}, your program returned {rg}x{cg}.\n"
                + show(expected, "Expected") + "\n" + show(got, "Your output"))
    if re_ * ce > max_cells:
        wrong = sum(expected[r][c] != got[r][c] for r in range(re_) for c in range(ce))
        return f"{wrong} of {re_ * ce} cells are wrong (grid too large to print the diff)."
    wrong = 0
    rows = []
    for r in range(re_):
        s = []
        for c in range(ce):
            if expected[r][c] == got[r][c]:
                s.append(".")
            else:
                s.append(str(int(got[r][c])))
                wrong += 1
        rows.append("".join(s))
    return (f"{wrong} of {re_ * ce} cells are wrong.\n"
            + show(expected, "Expected") + "\n"
            + f"Your output, with CORRECT cells shown as '.' ({re_}x{ce}):\n" + "\n".join(rows))


def feedback_prompt(results, train, max_examples=3):
    """Tell the model which examples failed and how; ask for a corrected program."""
    lines = []
    n_ok = sum(r["ok"] for r in results)
    lines.append(f"Your program reproduced {n_ok} of {len(results)} examples.")
    shown = 0
    for i, (r, pair) in enumerate(zip(results, train), 1):
        if r["ok"]:
            continue
        if shown >= max_examples:
            break
        shown += 1
        lines.append(f"\n## Example {i} failed")
        if r.get("error"):
            lines.append(f"Error: {r['error']}")
        else:
            lines.append(diff_text(pair["output"], r["output"]))
    lines.append("\nRe-examine the rule against ALL examples, then give the corrected "
                 "complete program in ONE ```python code block.")
    return "\n".join(lines)


Writing /kaggle/working/hybrid/grids.py


In [9]:
%%writefile /kaggle/working/hybrid/client.py
"""Minimal OpenAI-compatible chat client for a local vLLM server (stdlib only).

Returns a normalised dict: content, reasoning, tool_calls [{id, name, arguments}],
finish_reason, prompt_tokens, completion_tokens. If the server's tool parser
misses a call that the model wrote as qwen3_coder markup in its text, the call
is recovered from the markup, as the Tufa harness does.
"""
import json
import re
import urllib.error
import urllib.request

TOOL_BLOCK = re.compile(r"<tool_call>\s*<function=([\w\-]+)>(.*?)</function>\s*</tool_call>", re.S)
PARAM = re.compile(r"<parameter=([\w\-]+)>\n?(.*?)\n?</parameter>", re.S)


def recover_markup(text):
    calls = []
    for name, body in TOOL_BLOCK.findall(text or ""):
        args = {k: v for k, v in PARAM.findall(body)}
        calls.append({"id": f"markup-{len(calls) + 1}", "name": name, "arguments": args})
    return calls


def strip_markup(text):
    return TOOL_BLOCK.sub("", text or "").strip()


class ChatClient:
    def __init__(self, base_url, model, temperature=0.6, top_p=0.95, top_k=20,
                 max_tokens=16384, thinking=True, timeout=1800, reasoning_effort=None,
                 thinking_token_budget=None, template_kwargs=None):
        self.url = base_url.rstrip("/") + "/chat/completions"
        self.model = model
        # top_k=None leaves it to the server (gpt-oss wants no top-k).
        self.sampling = {k: v for k, v in {"temperature": temperature, "top_p": top_p,
                                           "top_k": top_k}.items() if v is not None}
        self.reasoning_effort = reasoning_effort
        # vLLM >= 0.19 forces </think> after this many reasoning tokens (needs the server
        # flag --reasoning-config). Qwen3.8 otherwise spends whole 16k turns thinking.
        self.thinking_token_budget = thinking_token_budget
        # Extra chat-template kwargs, e.g. Qwen3.8's reasoning_effort (xhigh default /
        # medium / low) -- the template itself turns it into a system instruction.
        self.template_kwargs = template_kwargs or {}
        self.max_tokens = max_tokens
        self.thinking = thinking
        self.timeout = timeout

    def chat(self, messages, tools=None, max_tokens=None, seed=None, timeout=None,
             tool_choice=None, template_kwargs=None):
        # template_kwargs: per-request overrides on top of the client's own, e.g. the
        # reasoning-effort ladder stepping one turn down to "medium".
        payload = {"model": self.model, "messages": messages,
                   "max_tokens": max_tokens or self.max_tokens,
                   "chat_template_kwargs": {"enable_thinking": self.thinking, **self.template_kwargs,
                                            **(template_kwargs or {})},
                   **self.sampling}
        if seed is not None:
            payload["seed"] = seed
        if self.reasoning_effort:          # gpt-oss: low / medium / high
            payload["reasoning_effort"] = self.reasoning_effort
        if self.thinking_token_budget:
            payload["thinking_token_budget"] = self.thinking_token_budget
        if tools:
            payload["tools"] = tools
            # A named tool forces that call (vLLM guided decoding); default lets it choose.
            payload["tool_choice"] = ({"type": "function", "function": {"name": tool_choice}}
                                      if tool_choice else "auto")
        req = urllib.request.Request(self.url, data=json.dumps(payload).encode(),
                                     headers={"Content-Type": "application/json"})
        try:
            with urllib.request.urlopen(req, timeout=timeout or self.timeout) as r:
                body = json.loads(r.read().decode())
            return normalise(body)
        except urllib.error.HTTPError as e:
            detail = e.read().decode(errors="replace")[:500]
            overflow = "maximum context length" in detail or "context length" in detail
            if (e.code == 400 and not overflow and "thinking_token_budget" in payload
                    and any(w in detail.lower() for w in ("thinking", "budget", "reasoning"))):
                # A server started without --reasoning-config (the last rungs of the startup
                # ladder) may refuse the budget: drop it for every later call, not fail them all.
                self.thinking_token_budget = None
                return self.chat(messages, tools, max_tokens, seed, timeout, tool_choice, template_kwargs)
            return {"error": f"HTTP {e.code}: {detail}", "context_overflow": overflow}
        except Exception as e:
            # probe24 lost every result to one uncaught socket TimeoutError.
            text = f"{type(e).__name__}: {e}"
            return {"error": text[:300], "context_overflow": False,
                    "timeout": isinstance(e, TimeoutError) or "timed out" in text}


def normalise(body):
    ch = body["choices"][0]
    msg = ch.get("message", {}) or {}
    content = msg.get("content") or ""
    reasoning = msg.get("reasoning_content") or msg.get("reasoning") or ""
    calls = []
    for tc in msg.get("tool_calls") or []:
        fn = tc.get("function", {})
        try:
            args = json.loads(fn.get("arguments") or "{}")
        except json.JSONDecodeError:
            args = {"_raw": fn.get("arguments")}
        calls.append({"id": tc.get("id") or f"call-{len(calls) + 1}",
                      "name": fn.get("name"), "arguments": args})
    if not calls:
        calls = recover_markup(content) or recover_markup(reasoning)
        content = strip_markup(content)
    usage = body.get("usage") or {}
    return {"content": content, "reasoning": reasoning, "tool_calls": calls,
            "finish_reason": ch.get("finish_reason"),
            "prompt_tokens": usage.get("prompt_tokens", 0),
            "completion_tokens": usage.get("completion_tokens", 0)}


Writing /kaggle/working/hybrid/client.py


In [10]:
%%writefile /kaggle/working/hybrid/run_direct.py
"""Direct-answer mode: the model reasons (e.g. Qwen3.8 at reasoning_effort xhigh) and
writes the test output grid(s) itself -- no tools. Two samples per task give the two
attempts. This is how headline ARC-AGI-2 numbers for chat models are usually measured,
so it is the comparison the REPL agent has to beat.

    python run_direct.py '<json config>'

Config: base_url, model, task_ids, samples (2), concurrency, max_tokens, deadline_epoch,
temperature/top_p/top_k, template_kwargs, out, data_dir.
Writes <out> (JSON) and <out>.jsonl (one line per finished sample).
"""
import json
import os
import re
import sys
import threading
import time
from concurrent.futures import ThreadPoolExecutor

sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
from client import ChatClient                       # noqa: E402
from grids import show                              # noqa: E402

# grids.SYSTEM tells the model it solves puzzles "by writing Python programs" -- right for
# the REPL agent, wrong when the answer is the grid itself.
DIRECT_SYSTEM = (
    "You are an expert at solving ARC-AGI puzzles. Each puzzle has a few input->output "
    "example pairs that share one hidden transformation rule. Grids are 2D arrays of digits "
    "0-9 (colours); 0 is usually the background."
)

# A reply cut off before any grid (the thinking budget missing or ignored, or the answer
# itself cut) gets ONE short follow-up without thinking: the tail of its own reasoning,
# and an order to commit. An unanswered task scores nothing; a committed guess may score.
RESCUE = """{prompt}

## Your notes so far (they were cut off)
{notes}

Stop analysing now. Apply the rule you were converging on and give the final test output grid(s): each as rows of digits in its own ```grid code block, in test order, with nothing else in the block."""

DIRECT_INSTRUCTIONS = """Solve this ARC puzzle. The examples share one hidden transformation rule. Work out the rule, check it against every example, then apply it to the test input(s).

At the very end of your answer, give each test output grid as rows of digits inside its own ```grid code block, in test order, with nothing else in the block."""


def direct_prompt(task):
    parts = [DIRECT_INSTRUCTIONS, ""]
    for i, p in enumerate(task["train"], 1):
        parts += [f"## Example {i}", show(p["input"], "Input"), show(p["output"], "Output"), ""]
    for j, t in enumerate(task["test"], 1):
        parts += [f"## Test input {j}", show(t["input"], "Input"), ""]
    return "\n".join(parts).rstrip()


def parse_grids(text, n_tests):
    """The last n_tests digit-only code blocks (```grid preferred, any fence accepted)."""
    blocks = re.findall(r"```(?:grid)?[^\n]*\n(.*?)```", text or "", re.S)
    grids = []
    for b in blocks:
        rows = [r.strip() for r in b.strip().splitlines() if r.strip()]
        rows = [r.replace(" ", "").replace(",", "") for r in rows]
        if rows and all(r.isdigit() for r in rows) and len({len(r) for r in rows}) == 1:
            grids.append([[int(c) for c in r] for r in rows])
    return grids[-n_tests:] if len(grids) >= n_tests else None


def answer(client, task, seed, deadline, max_tokens=None, rescue_tokens=8192, notes_chars=16000):
    """One direct answer to `task`: reason, then write the test grid(s). Returns a record;
    rec["grids"] is the list of test grids, or None when nothing parseable came back."""
    n = len(task["test"])
    prompt = direct_prompt(task)

    def timeout():
        return max(60.0, min(4 * 3600, deadline - time.time() + 60))

    mt = max_tokens or client.max_tokens
    for _ in range(3):
        r = client.chat([{"role": "system", "content": DIRECT_SYSTEM},
                         {"role": "user", "content": prompt}],
                        seed=seed, max_tokens=mt, timeout=timeout())
        if not (r.get("context_overflow") and mt > 8192):
            break
        mt = int(mt * 0.75)                    # a long prompt: shrink the reply, never give up
    if "error" in r:
        return {"status": "error: " + r["error"][:200], "grids": None, "gen_tokens": 0}
    rec = {"finish": r["finish_reason"], "gen_tokens": r["completion_tokens"],
           "reasoning_chars": len(r["reasoning"] or ""),
           "content_tail": (r["content"] or "")[-1500:],
           "reasoning_tail": (r["reasoning"] or "")[-3000:]}
    grids = parse_grids(r["content"], n)
    if grids:
        return dict(rec, status="answered", grids=grids)
    notes = ((r["reasoning"] or "") + "\n" + (r["content"] or "")).strip()
    if notes and deadline - time.time() > 120:
        r2 = client.chat([{"role": "system", "content": DIRECT_SYSTEM},
                          {"role": "user", "content": RESCUE.format(prompt=prompt,
                                                                    notes=notes[-notes_chars:])}],
                         seed=seed + 1, max_tokens=rescue_tokens, timeout=timeout(),
                         template_kwargs={"enable_thinking": False, "reasoning_effort": "low"})
        if "error" in r2:
            rec["rescue_error"] = r2["error"][:200]
        else:
            rec["gen_tokens"] += r2["completion_tokens"]
            rec["rescue_finish"] = r2["finish_reason"]
            rec["rescue_tail"] = ((r2["content"] or "") or (r2["reasoning"] or ""))[-1500:]
            # Without thinking, a reasoning parser may still file the reply as reasoning.
            grids = parse_grids(r2["content"], n) or parse_grids(r2["reasoning"], n)
            if grids:
                return dict(rec, status="rescued", grids=grids)
    return dict(rec, status="no grid", grids=None)


def main(cfg):
    t0 = time.time()
    root = cfg.get("data_dir", "/kaggle/input")
    import glob
    ch = json.load(open(glob.glob(f"{root}/**/arc-agi_evaluation_challenges.json", recursive=True)[0]))
    sol = json.load(open(glob.glob(f"{root}/**/arc-agi_evaluation_solutions.json", recursive=True)[0]))
    ids = cfg["task_ids"]
    client = ChatClient(cfg["base_url"], cfg["model"], max_tokens=cfg.get("max_tokens", 49152),
                        temperature=cfg.get("temperature", 1.0), top_p=cfg.get("top_p", 0.95),
                        top_k=cfg.get("top_k", 20), thinking=cfg.get("thinking", True),
                        thinking_token_budget=cfg.get("thinking_token_budget"),
                        template_kwargs=cfg.get("template_kwargs"), timeout=4 * 3600)
    out = cfg.get("out", "/kaggle/working/direct_results.json")
    jsonl = out.replace(".json", ".jsonl")
    deadline = cfg.get("deadline_epoch", float("inf"))
    lock = threading.Lock()
    recs = []
    jobs = [(t, k) for k in range(cfg.get("samples", 2)) for t in ids]   # sample-major

    def work(job):
        t, k = job
        rec = {"task_id": t, "sample": k}
        if time.time() > deadline:
            rec["status"] = "skipped: deadline"
        else:
            ts = time.time()
            rec.update(answer(client, ch[t], 1000 * k + 7, deadline,
                              rescue_tokens=cfg.get("rescue_tokens", 8192)))
            rec["wall"] = round(time.time() - ts, 1)
            grids = rec["grids"]
            rec["correct"] = [g == s for g, s in zip(grids, sol[t])] if grids else None
        with lock:
            recs.append(rec)
            with open(jsonl, "a") as f:
                f.write(json.dumps(rec) + "\n")
            print(f"[{time.time() - t0:6.0f}s] {t}#{k} {rec['status']:10s} "
                  f"tok {rec.get('gen_tokens', 0):6d} finish {rec.get('finish')} "
                  f"correct {rec.get('correct')}", flush=True)

    with ThreadPoolExecutor(max_workers=cfg.get("concurrency", 16)) as ex:
        list(ex.map(work, jobs))

    # pass@2 per task: any test output matched by either sample (fractional per test).
    per = {}
    for t in ids:
        n = len(sol[t])
        hit = [False] * n
        for r in recs:
            if r["task_id"] == t and r.get("correct"):
                hit = [h or c for h, c in zip(hit, r["correct"])]
        per[t] = sum(hit) / n
    tok = [r["gen_tokens"] for r in recs if "gen_tokens" in r]
    summary = {"config": cfg, "per_task": per, "score": sum(per.values()), "n_tasks": len(ids),
               "samples": len(recs), "mean_gen_tokens": sum(tok) / max(len(tok), 1),
               "cut_at_max_tokens": sum(1 for r in recs if r.get("finish") == "length"),
               "wall_s": time.time() - t0, "records": recs}
    json.dump(summary, open(out, "w"))
    print(f"\nDIRECT SCORE {summary['score']:.2f} / {len(ids)}  mean tokens {summary['mean_gen_tokens']:.0f}  "
          f"cut {summary['cut_at_max_tokens']}  wall {summary['wall_s']:.0f}s", flush=True)
    print("per task:", {t: round(v, 2) for t, v in per.items()}, flush=True)


if __name__ == "__main__":
    main(json.loads(sys.argv[1]))


Writing /kaggle/working/hybrid/run_direct.py


In [11]:
%%writefile /kaggle/working/hybrid/submit_main.py
"""Everything the competition rerun does after the fallback submission is written:
install the vLLM wheelhouse, start the server, run the agent. It NEVER raises --
an exception in a rerun fails the submission even when submission.json exists --
so every stage is caught and logged, and the fallback stands if a stage fails.

    python submit_main.py '<json config>'
"""
import glob
import json
import os
import shutil
import subprocess
import sys
import time
import traceback
import urllib.request
from pathlib import Path

HERE = os.path.dirname(os.path.abspath(__file__))
WORK = Path("/kaggle/working")
SITE = WORK / "vllm-site-packages"
LOG = WORK / "vllm-server.log"


def one(pattern):
    hits = glob.glob(pattern, recursive=True)
    if not hits:
        raise FileNotFoundError(pattern)
    return Path(hits[0])


def install(cfg):
    wh = one(os.path.join(cfg["wheelhouse_dir"], "**", "requirements.lock")).parent  # q27: from the setup cell
    t = time.time()
    subprocess.run([sys.executable, "-m", "pip", "install", "--no-index", "--find-links", str(wh),
                    "--requirement", str(wh / "requirements.lock"), "--target", str(SITE),
                    "--upgrade", "--ignore-installed", "--only-binary", ":all:", "--no-compile",
                    "--disable-pip-version-check", "--no-warn-conflicts", "-q"], check=True)
    print(f"wheelhouse installed in {time.time() - t:.0f}s", flush=True)
    if cfg.get("overlay"):
        wheels = sorted(glob.glob(cfg["overlay"], recursive=True))
        if not wheels:
            raise FileNotFoundError(cfg["overlay"])
        for pat in ("transformers", "transformers-*.dist-info", "huggingface_hub",
                    "huggingface_hub-*.dist-info"):
            for p in SITE.glob(pat):
                shutil.rmtree(p) if p.is_dir() else p.unlink()
        subprocess.run([sys.executable, "-m", "pip", "install", "--no-index", "--no-deps",
                        "--target", str(SITE), "--upgrade", "--force-reinstall", "--no-compile",
                        "--disable-pip-version-check", "-q", *wheels], check=True)
        print("overlay installed:", [Path(w).name for w in wheels], flush=True)


def cuda_link_dir():
    """FlashInfer JIT-links its kernels with plain `-lcuda` (FP8 KV on vLLM 0.19, the sampler on
    0.30). X1/X6/X8 (2026-10-09) died on `ld: cannot find -lcuda`. Link the container's
    libcuda.so* into one folder for LIBRARY_PATH, wherever the driver mounted it."""
    d = WORK / "cuda-link"
    d.mkdir(parents=True, exist_ok=True)
    hits = []
    for pat in ("/usr/local/nvidia/lib64/libcuda.so*", "/usr/lib/x86_64-linux-gnu/libcuda.so*",
                "/usr/lib64/libcuda.so*", "/usr/local/cuda/compat/libcuda.so*"):
        hits += sorted(glob.glob(pat))
    link = d / "libcuda.so"
    if hits and not os.path.lexists(link):
        os.symlink(next((h for h in hits if h.endswith(".so.1")), hits[0]), link)
    print("libcuda:", hits[:4], flush=True)
    return str(d)


def start_server(cfg, extra):
    model = one(cfg["model_glob"]).parent
    env = dict(os.environ, PYTHONPATH=str(SITE), USE_TF="0", TRANSFORMERS_NO_TF="1",
               TRANSFORMERS_NO_TORCHVISION="1", VLLM_NO_USAGE_STATS="1",
               LIBRARY_PATH=":".join([cuda_link_dir(), "/usr/local/nvidia/lib64",
                                      os.environ.get("LIBRARY_PATH", "")]))
    cmd = [sys.executable, "-m", "vllm.entrypoints.openai.api_server", "--model", str(model),
           "--served-model-name", "qwen", "--host", "127.0.0.1", "--port", "1234",
           "--tensor-parallel-size", "4", "--enable-auto-tool-choice", *cfg["parsers"],
           "--generation-config", "vllm", "--enable-prefix-caching",
           "--max-model-len", str(cfg.get("max_model_len", 65536)), "--gpu-memory-utilization", "0.90",
           "--max-num-seqs", str(cfg.get("max_num_seqs", 32)), *extra]
    print(" ".join(cmd), flush=True)
    t = time.time()
    # "w": a retry must not see the previous attempt's "EngineCore failed".
    proc = subprocess.Popen(cmd, env=env, stdout=open(LOG, "w"), stderr=subprocess.STDOUT,
                            start_new_session=True)
    while time.time() - t < cfg.get("server_wait_s", 1500):  # q27: per option set, from the config
        text = LOG.read_text(errors="replace") if LOG.exists() else ""
        if proc.poll() is not None or "EngineCore failed" in text:
            break
        try:
            urllib.request.urlopen("http://127.0.0.1:1234/v1/models", timeout=5)
            print(f"server ready after {time.time() - t:.0f}s", flush=True)
            return proc
        except Exception:
            time.sleep(10)
    print("server FAILED to start; tail:\n" + "\n".join(
        LOG.read_text(errors="replace").splitlines()[-30:]), flush=True)
    kill(proc)
    return None


def kill(proc):
    """Kill the server's whole process group -- engine workers hold GPU memory -- and wait
    for the GPUs to drain, so a restart does not hit out-of-memory."""
    import signal
    try:
        os.killpg(proc.pid, signal.SIGKILL)
    except Exception:
        proc.kill()
    try:
        proc.wait(timeout=60)
    except Exception:
        pass
    for _ in range(36):
        used = subprocess.run(["nvidia-smi", "--query-gpu=memory.used", "--format=csv,noheader,nounits"],
                              capture_output=True, text=True).stdout.split()
        if used and all(int(u) < 1500 for u in used if u.strip().isdigit()):
            return
        time.sleep(5)


def server_ladder(cfg):
    """Option sets to try in order. An explicit cfg["server_ladder"] wins (plain serving is
    appended if missing); otherwise server_extra + server_try -> server_extra -> plain."""
    if cfg.get("server_ladder"):
        ladder = [list(x) for x in cfg["server_ladder"]]
        return ladder if [] in ladder else ladder + [[]]
    base = cfg.get("server_extra", [])
    ladder = [base + cfg["server_try"]] if cfg.get("server_try") else []
    return ladder + [base] + ([[]] if base else [])


def main(cfg):
    proc = None
    try:
        install(cfg)
        # A ladder of server option sets, most ambitious first; the last is plain serving.
        # Untested-on-L4 options (MTP, --reasoning-config) must never cost the whole run.
        ladder = server_ladder(cfg)
        for i, extra in enumerate(ladder):
            proc = start_server(cfg, extra)
            if proc is not None:
                print(f"server up with option set {i}: {extra}", flush=True)
                break
            print(f"option set {i} failed -> next", flush=True)
            time.sleep(20)
        if proc is None:
            return
        sys.path.insert(0, HERE)
        import run_submit
        run_submit.main(dict(cfg, base_url="http://127.0.0.1:1234/v1", model="qwen"))
    except BaseException:
        print("submit_main caught:\n" + traceback.format_exc(), flush=True)
    finally:
        if proc is not None:
            kill(proc)
        shutil.rmtree(SITE, ignore_errors=True)


if __name__ == "__main__":
    try:
        main(json.loads(sys.argv[1]))
    except BaseException:
        print(traceback.format_exc(), flush=True)
    sys.exit(0)


Writing /kaggle/working/hybrid/submit_main.py


In [12]:
%%writefile /kaggle/working/hybrid/hybrid_core.py
"""Submission, B0 candidate pool, routing and merge for the ARC-AGI-2 hybrid notebook (kaggle/arc2/hybrid).

Runs in the notebook process (phase 1, safety net) and in the phase-2 orchestrator (merge), so it needs only the
standard library; build_b0_pool also imports numpy and B0's arc_decoder (lazily).

  submission     {task_id: [{"attempt_1": grid, "attempt_2": grid} per test input]}; a grid is a non-empty
                 rectangular list of lists of ints 0-9 (B0's placeholder [[0]] included)
  safety net     B0's submission as its final cell builds it: ArcDecoder(n_guesses=2).run_selection_algo()
                 (score_kgmon) over every readable file in /kaggle/inference_outputs, placed by
                 ArcDataset.get_submission's rule (attempt i = i-th selected grid, [[0]] when fewer)
  pool           per task and test output: B0's candidates in score_kgmon order with their scores, the margin
                 (score of rank 1 minus rank 2, None below 2 candidates) and the number of unique candidates
  route          every task, shortest first by len(json.dumps(challenges[t])) (the key of the 27B direct runs'
                 cheap_first, Q27D/Q27F), ties by task id; B0's confidence and whether B0 reached the task play no
                 part. B0's confidence key (an output with fewer than 2 candidates first, 0 before 1, then the
                 smallest kgmon margin) is kept for the record and for picking the commit-run tasks
  plan           the 27B samples in issue order over the route: pass 1 = sample 0 on the band (the shortest
                 ceil(0.25 x n) tasks), pass 2 = sample 1 on the band, pass 3 = sample 0 on the tasks beyond the band
  merge          per test output: attempt_1 is never replaced; attempt_2 = the first valid 27B grid in sample order
                 that differs from attempt_1 (a later sample is used only when the earlier ones gave no such grid)
"""
from __future__ import annotations

import bz2
import copy
import json
import math
import os
import pickle
import time

SCHEMA = "arc2_hybrid.v2"
PLACEHOLDER = [[0]]
BAND_FRACTION = 0.25     # pass 1 and 2 cover the shortest ceil(0.25 x n) tasks
BAND_SAMPLES = 2         # samples per band task (pass 1: sample 0, pass 2: sample 1)


# ----------------------------------------------------------------------------
# grids and submissions

def valid_grid(g) -> bool:
    """A grid the scorer can compare: 1-30 rows of the same length 1-30, ints 0-9 (bool excluded)."""
    return (isinstance(g, list) and 1 <= len(g) <= 30 and all(isinstance(r, list) and r for r in g)
            and len({len(r) for r in g}) == 1 and 1 <= len(g[0]) <= 30
            and all(isinstance(v, int) and not isinstance(v, bool) and 0 <= v <= 9 for r in g for v in r))


def placeholder_submission(challenges: dict) -> dict:
    """B0's ArcDataset.get_submission() with no results: [[0]] for both attempts of every test input."""
    return {t: [{"attempt_1": [[0]], "attempt_2": [[0]]} for _ in challenges[t]["test"]] for t in challenges}


def validate_submission(sub, challenges: dict) -> list:
    """Problems with a submission for `challenges` (empty when valid): every task id, one entry per test input,
    attempt_1 and attempt_2 valid grids, no extra task ids."""
    if not isinstance(sub, dict):
        return [f"submission is {type(sub).__name__}, not a dict"]
    p = []
    missing = [t for t in challenges if t not in sub]
    extra = [t for t in sub if t not in challenges]
    if missing:
        p.append(f"{len(missing)} task ids missing (first {missing[:3]})")
    if extra:
        p.append(f"{len(extra)} unknown task ids (first {extra[:3]})")
    for t, task in challenges.items():
        entries = sub.get(t)
        if t in sub and (not isinstance(entries, list) or len(entries) != len(task["test"])):
            p.append(f"{t}: {len(entries) if isinstance(entries, list) else type(entries).__name__} entries for "
                     f"{len(task['test'])} test inputs")
            continue
        for j, e in enumerate(entries or []):
            if not isinstance(e, dict):
                p.append(f"{t}[{j}]: entry is not a dict")
                continue
            for k in ("attempt_1", "attempt_2"):
                if not valid_grid(e.get(k)):
                    p.append(f"{t}[{j}]: {k} is not a valid grid")
    return p


def write_json_atomic(path: str, obj, indent=None) -> None:
    """Temp file in the same directory, fsync, rename: a kill leaves the old file or the new one, never half."""
    d = os.path.dirname(os.path.abspath(path))
    os.makedirs(d, exist_ok=True)
    tmp = os.path.join(d, f".{os.path.basename(path)}.tmp{os.getpid()}")
    with open(tmp, "w") as f:
        json.dump(obj, f, indent=indent)
        f.flush()
        os.fsync(f.fileno())
    os.replace(tmp, path)


def write_submission(path: str, sub: dict, challenges: dict) -> list:
    """Validate, then write atomically. An invalid submission is not written (the file on disk stays); returns the
    problems (empty when written)."""
    problems = validate_submission(sub, challenges)
    if not problems:
        write_json_atomic(path, sub)
    return problems


def read_submission(path: str, challenges: dict):
    """(submission, problems); (None, [reason]) when the file is missing or unreadable."""
    try:
        with open(path) as f:
            sub = json.load(f)
    except Exception as exc:  # missing, cut, not JSON
        return None, [f"{path}: {type(exc).__name__}: {exc}"[:300]]
    return sub, validate_submission(sub, challenges)


def ensure_valid(path: str, challenges: dict, fallbacks=()) -> dict:
    """Make `path` a valid submission whatever happened before: keep it if valid, else copy the first valid fallback
    file, else write the placeholder. Never raises. Returns what was done."""
    try:
        sub, problems = read_submission(path, challenges)
        if sub is not None and not problems:
            return {"status": "valid", "source": path}
        for fb in fallbacks:
            fsub, fproblems = read_submission(fb, challenges)
            if fsub is not None and not fproblems:
                write_json_atomic(path, fsub)
                return {"status": "restored", "source": fb, "problems": problems[:5]}
        write_json_atomic(path, placeholder_submission(challenges))
        return {"status": "placeholder", "problems": problems[:5]}
    except Exception as exc:
        return {"status": "error", "error": f"{type(exc).__name__}: {exc}"[:300]}


def check_stage(path: str, challenges: dict, stage: str, log: list | None = None) -> dict:
    """Record whether `path` holds a valid submission at a named stage (the notebook keeps these rows)."""
    sub, problems = read_submission(path, challenges)
    row = {"stage": stage, "valid": sub is not None and not problems, "problems": problems[:3],
           "t": round(time.time(), 1)}
    if log is not None:
        log.append(row)
    return row


# ----------------------------------------------------------------------------
# B0's outputs: safe loading, selection, pool

def load_b0_outputs(dir_outputs: str, task_ids=None):
    """ArcDecoder.load_decoded_results without the crash: {base_key: {f"{file}.out{i}": sample}} from every file
    that unpickles (a file cut by a kill is skipped and listed). task_ids: only these tasks."""
    results, skipped = {}, []
    wanted = set(task_ids) if task_ids is not None else None
    if not os.path.isdir(dir_outputs):
        return results, skipped
    for key in os.listdir(dir_outputs):                 # B0's order (score ties keep insertion order)
        base_key = key.split(".")[0]
        if wanted is not None and base_key.rsplit("_", 1)[0] not in wanted:
            continue
        try:
            with bz2.BZ2File(os.path.join(dir_outputs, key)) as f:
                outputs = pickle.load(f)
            samples = {f"{key}.out{i}": s for i, s in enumerate(outputs)}
        except Exception as exc:
            skipped.append({"file": key, "error": f"{type(exc).__name__}: {exc}"[:200]})
            continue
        results.setdefault(base_key, {}).update(samples)
    return results, skipped


def _grid_list(a):
    return [[int(v) for v in row] for row in (a.tolist() if hasattr(a, "tolist") else a)]


def build_b0_pool(challenges: dict, dir_outputs: str):
    """(submission, pool, info): B0's final-cell submission over the readable files and the ranked candidates.

    The selection is B0's: arc_decoder.ArcDecoder(n_guesses=2).run_selection_algo() (score_kgmon); attempts are
    placed as ArcDataset.get_submission / fill_submission place them."""
    import numpy as np
    import arc_decoder

    decoded, skipped = load_b0_outputs(dir_outputs)
    decoder = arc_decoder.ArcDecoder(None, n_guesses=2)
    decoder.decoded_results = decoded
    selected = decoder.run_selection_algo()
    sub = placeholder_submission(challenges)
    unknown = []
    for k, v in selected.items():                       # ArcDataset.fill_submission
        base_id, base_nr = k.rsplit("_", 1)
        if base_id not in sub or int(base_nr) >= len(sub[base_id]):
            unknown.append(k)
            continue
        target = sub[base_id][int(base_nr)]
        for i, g in enumerate(v[:len(target)]):
            grid = _grid_list(g)
            if valid_grid(grid):            # B0's decoder only yields valid grids; a bad one would void the whole file
                target[f"attempt_{i + 1}"] = grid
            else:
                unknown.append(f"{k}: attempt_{i + 1} not a valid grid")
    pool = {}
    for t, task in challenges.items():
        outs = []
        for j in range(len(task["test"])):
            guesses = decoded.get(f"{t}_{j}", {})
            groups = {}
            for k, g in guesses.items():
                h = arc_decoder.hashable(g["solution"])
                groups.setdefault(h, [g["solution"], []])[1].append(g)
            official = arc_decoder.score_kgmon(dict(guesses)) if guesses else []
            cands = []
            for r, grid in enumerate(official):
                members = groups[arc_decoder.hashable(grid)][1]
                cands.append({"grid": _grid_list(grid), "rank": r + 1,
                              "score_kgmon": float(arc_decoder.getter_kgmon(members)), "n_beams": len(members),
                              "score_aug_mean": float(np.mean([np.mean(m["score_aug"]) for m in members]))})
            outs.append({"test_index": j, "n_beams": len(guesses), "n_unique": len(cands),
                         "margin_kgmon": (cands[0]["score_kgmon"] - cands[1]["score_kgmon"]) if len(cands) > 1 else None,
                         "candidates": cands})
        pool[t] = outs
    info = {"n_files_read": len({k.rsplit(".out", 1)[0] for v in decoded.values() for k in v}),
            "skipped_files": skipped, "unknown_keys": unknown,
            "tasks_with_candidates": sum(1 for t in pool if any(o["n_unique"] for o in pool[t])),
            "outputs_with_candidates": sum(1 for t in pool for o in pool[t] if o["n_unique"])}
    return sub, pool, info


# ----------------------------------------------------------------------------
# routing

def output_key(o: dict) -> tuple:
    n = int(o.get("n_unique") or 0)
    if n < 2 or o.get("margin_kgmon") is None:
        return (0, float(min(n, 1)))
    return (1, float(o["margin_kgmon"]))


def task_key(outputs: list) -> tuple:
    return min((output_key(o) for o in outputs), default=(0, 0.0))


def task_size(task: dict) -> int:
    """The route key: characters of the task's JSON (q27d_run.cheap_first's key)."""
    return len(json.dumps(task))


def route_order(challenges: dict) -> list:
    """Task ids, shortest task JSON first, ties by id (see the module docstring)."""
    return sorted(challenges, key=lambda t: (task_size(challenges[t]), t))


def band_size(n: int, fraction: float = BAND_FRACTION) -> int:
    """Tasks in the band: ceil(fraction x n), at most n (0.25: 2 of 5, 30 of 120, 60 of 240)."""
    return min(int(n), int(math.ceil(round(float(fraction) * int(n), 9)))) if n else 0


def issue_plan(route: list, fraction: float = BAND_FRACTION, band_samples: int = BAND_SAMPLES,
               beyond: bool = True) -> list:
    """[[task_id, sample, pass]] in issue order: pass k (k = 1..band_samples) = sample k-1 on the band tasks in route
    order; then pass band_samples + 1 = sample 0 on the tasks beyond the band in route order (when beyond)."""
    nb = band_size(len(route), fraction)
    plan = [[t, k, k + 1] for k in range(int(band_samples)) for t in route[:nb]]
    if beyond:
        plan += [[t, 0, int(band_samples) + 1] for t in route[nb:]]
    return plan


def route_table(pool: dict, challenges: dict, fraction: float = BAND_FRACTION,
                band_samples: int = BAND_SAMPLES) -> list:
    """The route with each task's size, band membership and planned samples and, for the record, B0's confidence key
    and (n_unique, margin) per output."""
    order = route_order(challenges)
    nb = band_size(len(order), fraction)
    return [{"task_id": t, "position": i, "size": task_size(challenges[t]), "band": i < nb,
             "samples_planned": int(band_samples) if i < nb else 1, "b0_key": list(task_key(pool.get(t, []))),
             "outputs": [[o["n_unique"], o["margin_kgmon"]] for o in pool.get(t, [])]}
            for i, t in enumerate(order)]


# ----------------------------------------------------------------------------
# merge

def _by_sample(v) -> dict:
    """{sample: grids}; a bare list of grids (one sample) counts as sample 0."""
    if isinstance(v, dict):
        return {int(k): g for k, g in v.items()}
    return {0: v}


def merge(b0_sub: dict, answers: dict) -> tuple:
    """(merged submission, changes). answers: {task_id: {sample: [grid per test output]}} from the 27B samples that
    returned grids (a bare list counts as sample 0). Per output: attempt_1 is never replaced; attempt_2 = the first
    valid grid in sample order that differs from attempt_1; a later sample is used only when the earlier ones gave
    no valid grid that differs from attempt_1."""
    out = copy.deepcopy(b0_sub)
    changes = []
    for t, per in answers.items():
        if t not in out:
            continue
        per = _by_sample(per)
        for j, entry in enumerate(out[t]):
            cands = [(k, per[k][j]) for k in sorted(per)
                     if isinstance(per[k], list) and j < len(per[k]) and valid_grid(per[k][j])]
            if not cands:
                continue
            use = next(((k, g) for k, g in cands if g != entry.get("attempt_1")), None)
            if use is None:
                changes.append({"task_id": t, "test_index": j, "action": "kept: equals attempt_1", "sample": None,
                                "samples_with_grid": [k for k, _ in cands]})
                continue
            k, g = use
            action = "kept: equals attempt_2" if g == entry.get("attempt_2") else "replaced attempt_2"
            entry["attempt_2"] = g
            changes.append({"task_id": t, "test_index": j, "action": action, "sample": k,
                            "samples_with_grid": [c for c, _ in cands]})
    return out, changes


def read_sample_lines(path: str) -> list:
    """The records of a samples.jsonl (a line cut by a kill is skipped)."""
    lines = []
    if not os.path.isfile(path):
        return lines
    with open(path) as f:
        for raw in f:
            raw = raw.strip()
            if not raw:
                continue
            try:
                lines.append(json.loads(raw))
            except json.JSONDecodeError:
                continue
    return lines


def answers_from_lines(lines: list) -> dict:
    """{task_id: {sample: grids}}; per task and sample the first record with grids wins (no sample field: 0)."""
    answers = {}
    for ln in lines:
        k = int(ln.get("sample") or 0)
        if ln.get("grids") and k not in answers.get(ln.get("task_id"), {}):
            answers.setdefault(ln["task_id"], {})[k] = ln["grids"]
    return answers


def answers_from_samples(path: str) -> dict:
    """{task_id: {sample: grids}} from a samples.jsonl."""
    return answers_from_lines(read_sample_lines(path))


def per_task_log(lines: list, changes: list) -> dict:
    """Per task: the samples that ran ({sample: status}) and, per test output, the sample whose grid is attempt_2
    (None: B0's attempt_2 kept) with the merge action."""
    log = {}
    for ln in lines:
        log.setdefault(ln["task_id"], {"samples": {}, "outputs": {}})["samples"].setdefault(
            str(int(ln.get("sample") or 0)), ln.get("status"))
    for c in changes:
        log.setdefault(c["task_id"], {"samples": {}, "outputs": {}})["outputs"][str(c["test_index"])] = {
            "used_sample": c.get("sample"), "action": c["action"]}
    return log


def merge_files(b0_path: str, samples_path: str, out_path: str, challenges: dict, report_path: str | None = None):
    """Rebuild the merged submission from the safety-net file and samples.jsonl; idempotent. Returns a summary.
    Nothing is written when the safety net is unreadable (the file at out_path stays)."""
    b0, problems = read_submission(b0_path, challenges)
    if b0 is None or problems:
        return {"status": "safety net unreadable", "problems": problems[:3]}
    lines = read_sample_lines(samples_path)
    answers = answers_from_lines(lines)
    merged, changes = merge(b0, answers)
    wp = write_submission(out_path, merged, challenges)
    used = {}
    for c in changes:
        if c["action"] != "kept: equals attempt_1":
            used[str(c["sample"])] = used.get(str(c["sample"]), 0) + 1
    summary = {"status": "written" if not wp else "invalid, not written", "problems": wp[:3],
               "tasks_answered": len(answers),
               "replaced": sum(1 for c in changes if c["action"] == "replaced attempt_2"),
               "kept_equal_attempt_1": sum(1 for c in changes if c["action"] == "kept: equals attempt_1"),
               "kept_equal_attempt_2": sum(1 for c in changes if c["action"] == "kept: equals attempt_2"),
               "attempt_2_from_sample": used,
               "utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime())}
    if report_path:
        write_json_atomic(report_path, dict(summary, changes=changes, per_task=per_task_log(lines, changes)))
    return summary


# ----------------------------------------------------------------------------
# scoring (commit run and tests)

def pass_at_2(sub: dict, solutions: dict, task_ids=None) -> dict:
    """Outputs right in either attempt, and the leaderboard-style task score (each task 1, split over its outputs)."""
    ids = [t for t in (task_ids or solutions) if t in solutions and t in sub]
    hits = {t: [e["attempt_1"] == s or e["attempt_2"] == s for e, s in zip(sub[t], solutions[t])] for t in ids}
    return {"outputs": sum(sum(h) for h in hits.values()), "n_outputs": sum(len(h) for h in hits.values()),
            "task_score": round(sum(sum(h) / len(h) for h in hits.values() if h), 4), "n_tasks": len(ids),
            "per_output": {f"{t}_{j}": bool(v) for t, h in hits.items() for j, v in enumerate(h)}}


Writing /kaggle/working/hybrid/hybrid_core.py


In [13]:
%%writefile /kaggle/working/hybrid/hybrid_p2.py
"""Phase 2 of the ARC-AGI-2 hybrid notebook (kaggle/arc2/hybrid, version 6): serve Qwen3.8-27B-FP8 and answer the
tasks, shortest first, in the public notebook's direct mode; merge every answer into submission.json as it arrives.

    python hybrid_p2.py install <config.json>   wheelhouse + serving overlay into submit_main.SITE (started in the
                                                background during phase 1); status in <out_dir>/_install.json
    python hybrid_p2.py run <config.json>       wait for the install, server ladder, smoke request, direct answers
                                                down cfg["plan"] until the issue deadline, stop the server

Direct mode as madarshbb's run_submit.direct calls run_direct.answer (prompt, grid parser, one rescue request
without thinking when the reply holds no grid; seed 1000 * sample + 7), cfg["concurrency"] samples at a time, in
the order of cfg["plan"] ([task_id, sample, pass]; default v6_plan over cfg["route"], see "Version 6" below; each
pass shortest task JSON first). Within each pass jobs are issued in plan order. A sample k >= 1 is not issued when the task's
sample 0 has already returned a valid grid that differs from B0's attempt_1 on every test output (the merge could
not use it); it is logged as skipped. No request is sent after cfg["issue_end_epoch"]; requests in flight are cut by
their HTTP timeout at cfg["collect_end_epoch"].

Deadline-aware caps (version 5, lever A2-D1 part 1; DeadlineCaps, RateMeter): a sample issued at t gets
max_tokens = min(cfg["max_tokens"], rate x (end_by - t)), where rate is the per-stream decode rate (generated tokens
over seconds of the last finished requests, x 0.85; a conservative fallback before any request has finished) and
end_by the latest end of the answer that leaves its rescue request (cfg["rescue_tokens"] at the same rate) time to be
sent before the issue deadline and run_direct's rescue gate, and to finish before the collection deadline, less a
margin. A sample whose cap stays at or above cfg["max_tokens"] is sent exactly as before ("normal"); one that would get
fewer than min_answer_tokens is not issued (nor is anything after it: the plan entries left count as not started).
The rescue request gets min(cfg["rescue_tokens"], rate x (collect - now - margin)) and is not sent below
min_rescue_tokens; the sample then keeps what its answer produced. The deadlines are the run's own (cfg), so a commit
run uses its short ones. Every sample line records its issue time, the cap applied (kind, why), the rate used and the
rescue's cap; the run ends with the line "deadline caps: N reduced, M zero-token at collection" (also in _p2_run.json).

Version 6 (v6_plan, the issue loop in run_routed): the plan is pass 1 = sample 0 on the shortest quarter ("quarter"),
pass 2 = sample 0 on every task beyond it, shortest first ("band": cfg["band_98k"], max_tokens 98,304 and thinking
budget 90,000 as Q27G, at most 16 band samples in flight), pass 3 = sample 1 on the quarter ("second"). Every quarter
sample is issued first; then a free slot goes to the next band task while fewer than 16 band samples run, else (the
band at its cap) to the next second sample beside it; once the band is done or closed, second samples take every
slot. Band issue rule (band_rule): a band task is started only if the deadline caps would give it at least
min_fit_tokens (default: its full 98,304, i.e. it can run to its cap and still leave its rescue time before the
collection deadline at the planning rate); the first refusal closes the band (later band tasks are larger and time only
shrinks) and the rest are logged as skipped. v5's deadline caps apply to every sample (a band sample's normal cap is
98,304). Per-output grid parsing (parse_outputs, installed into run_direct by install_parser): the last n_tests fenced
digit blocks by position, each kept if rectangular, so a ragged block costs only its own output. Logged per sample:
kind, issue time, cap, band rule, in-flight counts, finish, tokens, and which outputs parsed; at the end the B0 DFS
record (cfg["b0_dfs_log"], written by B0's arc_solver.py), the server's KV use and preemptions (/metrics while running,
the server log at the end) and the line "v6: B0 DFS mean s X (H2 320), band issued N skipped M, deadline caps ..."
with the four commit-run checks (also in _p2_run.json).

Version 6 without the band (v6nb: cfg["band_98k"] = {"on": False}): no band sample is issued; the plan is version 5's
(hybrid_core.issue_plan: pass 1 = sample 0 on the quarter, pass 2 = sample 1 on it ("second"), pass 3 = sample 0 on the
tasks beyond it at the normal cap ("rest")), issued in that order (PASS_KIND_BAND_OFF; the "rest" queue after the
second samples); check 3 reads "band off: N issued" and passes when N is 0.

After each finished sample (under a lock): one line appended to <out_dir>/samples.jsonl, then submission.json
rebuilt from the safety net (cfg["b0_submission"]) and every answer so far (hybrid_core.merge_files: atomic write,
validated first; per output the first grid in sample order that differs from attempt_1). A kill at any moment leaves
the last valid submission.json.

Never raises. Stages go to <out_dir>/_stages.json; the server's process group id to <out_dir>/_server.pid (the
notebook kills it after a hard stop). cfg["server"]: "vllm" (GPU kernel) or "stub" (CPU twin and tests: the
scripted hybrid_stub_server; nothing installed or waited for).
"""
import collections
import json
import os
import re
import shutil
import subprocess
import sys
import threading
import time
import traceback
import urllib.request
from concurrent.futures import ThreadPoolExecutor

HERE = os.path.dirname(os.path.abspath(__file__))
sys.path.insert(0, HERE)
import hybrid_core as HC  # noqa: E402

STATUSES = ("answered", "rescued", "no grid", "error")


def seed_of(sample):
    """run_submit.direct's seed for a sample (sample 0: 7, sample 1: 1007; the rescue request uses seed + 1)."""
    return 1000 * int(sample) + 7


class Stages:
    def __init__(self, out_dir, t0, name="_stages.json"):
        self.path = os.path.join(out_dir, name)
        self.t0 = t0
        self.rows = []
        if os.path.isfile(self.path):
            try:
                self.rows = json.load(open(self.path))
            except Exception:
                self.rows = []

    def add(self, stage, status, detail=None, **extra):
        row = {"stage": stage, "status": status, "s": round(time.time() - self.t0, 1)}
        if detail is not None:
            row["detail"] = detail
        row.update(extra)
        self.rows.append(row)
        print(f"[p2 {row['s']:8.1f}s] {stage}: {status}" + (f" ({detail})" if detail else ""), flush=True)
        HC.write_json_atomic(self.path, self.rows, indent=1)


# ----------------------------------------------------------------------------
# install (submit_main.install, the public code's offline wheelhouse + overlay install)

def install(cfg):
    out = cfg["out_dir"]
    os.makedirs(out, exist_ok=True)
    marker = os.path.join(out, "_install.json")
    t = time.time()
    HC.write_json_atomic(marker, {"status": "running", "start_epoch": t, "python": sys.version.split()[0],
                                  "executable": sys.executable, "pid": os.getpid()})
    try:
        import submit_main as sm
        sm.install(cfg)
        dists = sorted(n[:-len(".dist-info")] for n in os.listdir(sm.SITE) if n.endswith(".dist-info"))
        want = [d for d in dists if d.split("-")[0].lower() in ("vllm", "transformers", "huggingface_hub", "torch",
                                                                 "triton", "flashinfer_python")
                or d.lower().startswith(("nvidia_cuda_runtime", "nvidia_cublas", "nvidia_nccl"))]
        HC.write_json_atomic(marker, {"status": "ok", "s": round(time.time() - t, 1), "python": sys.version.split()[0],
                                      "executable": sys.executable, "site": str(sm.SITE), "n_dists": len(dists),
                                      "key_dists": want, "end_epoch": time.time()})
    except BaseException:
        HC.write_json_atomic(marker, {"status": "failed", "s": round(time.time() - t, 1),
                                      "python": sys.version.split()[0], "executable": sys.executable,
                                      "error": traceback.format_exc()[-2000:], "end_epoch": time.time()})


def _alive(pid):
    try:
        os.kill(int(pid), 0)
    except ProcessLookupError:
        return False
    except Exception:
        return True
    try:                                          # a zombie (exited, not yet reaped) counts as gone
        with open(f"/proc/{int(pid)}/stat") as f:
            return f.read().split(")")[-1].split()[0] != "Z"
    except Exception:
        return True


def wait_install(cfg, st):
    """The phase-1 background install's result; install here when it did not run or failed (time permitting)."""
    marker = os.path.join(cfg["out_dir"], "_install.json")
    give_up = float(cfg["server_give_up_epoch"])
    while True:
        try:
            info = json.load(open(marker)) if os.path.isfile(marker) else {}
        except Exception:
            info = {"status": "running"}          # being replaced
        if info.get("status") == "running" and info.get("pid") and not _alive(info["pid"]):
            info = {"status": "failed", "error": f"install process {info['pid']} gone without a result"}
        if info.get("status") == "running" and time.time() < give_up:
            time.sleep(5)
            continue
        break
    if info.get("status") == "ok":
        st.add("install", "ok", f"background install, {info.get('s')} s", key_dists=info.get("key_dists"))
        return True
    if time.time() >= give_up:
        st.add("install", "gave up", f"server give-up time reached; background install {info.get('status')}")
        return False
    st.add("install", "retry", f"background install {info.get('status') or 'not run'}; installing now",
           error=(info.get("error") or "")[-600:] or None)
    install(cfg)
    info = json.load(open(marker))
    st.add("install", info["status"], f"{info.get('s')} s", error=(info.get("error") or "")[-600:] or None)
    return info["status"] == "ok"


# ----------------------------------------------------------------------------
# torch and the driver (the wheelhouse's CUDA libraries, the host's driver)

PROBE = r"""
import json, sys
out = {"python": sys.version.split()[0], "executable": sys.executable}
try:
    import torch
    out.update({"torch": torch.__version__, "torch_cuda": torch.version.cuda, "cuda_available": torch.cuda.is_available()})
    if out["cuda_available"]:
        out.update({"device_count": torch.cuda.device_count(), "device": torch.cuda.get_device_name(0)})
        x = torch.ones(2, device="cuda")
        out["cuda_op"] = float((x + x).sum().item())
except BaseException as exc:
    out["torch_error"] = (type(exc).__name__ + ": " + str(exc))[:500]
if "--vllm" in sys.argv:
    try:
        import vllm
        out["vllm"] = vllm.__version__
    except BaseException as exc:
        out["vllm_error"] = (type(exc).__name__ + ": " + str(exc))[:500]
print("PROBE " + json.dumps(out))
"""


def probe_torch(python, site, env=None, import_vllm=False, timeout=900):
    """Import torch (and vllm) under `python` with only `site` on PYTHONPATH; one CUDA op when a GPU is visible."""
    env = dict(env if env is not None else os.environ, PYTHONPATH=str(site))
    t = time.time()
    try:
        r = subprocess.run([python, "-c", PROBE] + (["--vllm"] if import_vllm else []), capture_output=True, text=True,
                           timeout=timeout, env=env)
        line = next((ln for ln in r.stdout.splitlines() if ln.startswith("PROBE ")), None)
        out = json.loads(line[6:]) if line else {"error": f"exit {r.returncode}: {(r.stderr or r.stdout)[-500:]}"}
    except Exception as exc:
        out = {"error": repr(exc)[:500]}
    out["s"] = round(time.time() - t, 1)
    return out


def driver_info():
    """Driver version and the highest CUDA version it supports (nvidia-smi), or None without a GPU."""
    try:
        r = subprocess.run(["nvidia-smi"], capture_output=True, text=True, timeout=60)
        head = r.stdout
        drv = re.search(r"Driver Version:\s*([\d.]+)", head)
        cuda = re.search(r"CUDA Version:\s*([\d.]+)", head)
        return {"driver": drv.group(1) if drv else None, "cuda": cuda.group(1) if cuda else None}
    except Exception:
        return None


def without_toolkit_libs(ld_path):
    """LD_LIBRARY_PATH without the image's CUDA toolkit directories (the wheels bring their own CUDA libraries)."""
    return os.pathsep.join(p for p in (ld_path or "").split(os.pathsep) if p and "/usr/local/cuda" not in p)


def check_torch(st, site):
    """torch must see the GPUs from the phase-2 interpreter. If it does not with this environment, retry without the
    image's CUDA toolkit library directories on LD_LIBRARY_PATH and keep that setting for the server when it works."""
    drv = driver_info()
    first = probe_torch(sys.executable, site)
    rows = {"driver": drv, "ld_library_path": os.environ.get("LD_LIBRARY_PATH", ""), "probe": first}
    if not first.get("cuda_available") and os.environ.get("LD_LIBRARY_PATH"):
        alt = without_toolkit_libs(os.environ["LD_LIBRARY_PATH"])
        second = probe_torch(sys.executable, site, dict(os.environ, LD_LIBRARY_PATH=alt))
        rows["probe_without_toolkit_libs"] = second
        if second.get("cuda_available"):
            os.environ["LD_LIBRARY_PATH"] = alt
            rows["ld_library_path_used"] = alt
    ok = bool(first.get("cuda_available") or rows.get("probe_without_toolkit_libs", {}).get("cuda_available"))
    st.add("torch", "ok" if ok else "failed",
           f"torch {first.get('torch')} (CUDA {first.get('torch_cuda')}), driver {(drv or {}).get('driver')} "
           f"(CUDA <= {(drv or {}).get('cuda')})", **rows)
    return ok


# ----------------------------------------------------------------------------
# server

def gpu_memory():
    try:
        r = subprocess.run(["nvidia-smi", "--query-gpu=index,memory.used", "--format=csv,noheader,nounits"],
                           capture_output=True, text=True, timeout=60)
        return [int(x.split(",")[1]) for x in r.stdout.strip().splitlines() if "," in x]
    except Exception:
        return None


def wait_ready(url, proc, seconds):
    t = time.time()
    while time.time() - t < seconds:
        if proc.poll() is not None:
            return False
        try:
            urllib.request.urlopen(url, timeout=5)
            return True
        except Exception:
            time.sleep(1)
    return False


def start_stub(cfg):
    cmd = [sys.executable, os.path.join(HERE, "hybrid_stub_server.py"), "--port", str(cfg.get("port", 1234)),
           "--challenges", cfg["challenges"], "--task-ids", ",".join(cfg["route"]),
           "--log", os.path.join(cfg["out_dir"], "_stub_requests.jsonl")]
    for key, flag in (("stub_solutions", "--solutions"), ("b0_submission", "--b0-submission"),
                      ("stub_slow_s", "--slow-s"), ("stub_tok_s", "--tok-s")):
        if cfg.get(key):
            cmd += [flag, str(cfg[key])]
    # its own log, not this process's stdout: a server left after a kill must not hold the notebook's pipe open
    log = open(os.path.join(cfg["out_dir"], "_stub_server.log"), "w")
    proc = subprocess.Popen(cmd, stdout=log, stderr=subprocess.STDOUT, start_new_session=True)
    if not wait_ready(f"http://127.0.0.1:{cfg.get('port', 1234)}/v1/models", proc, 60):
        proc.kill()
        return None
    return proc


def smoke(base_url):
    """One short request: /v1/models can answer while the engine dies on the first generation."""
    from client import ChatClient
    r = ChatClient(base_url, "qwen", max_tokens=256, thinking=False, timeout=600).chat(
        [{"role": "user", "content": "What is 2+2? Answer with the number only."}])
    return {k: r.get(k) for k in ("error", "content", "reasoning", "finish_reason", "completion_tokens")}


def start_vllm(cfg, st, base):
    import submit_main as sm
    give_up = float(cfg["server_give_up_epoch"])
    for i, extra in enumerate(sm.server_ladder(cfg)):
        left = give_up - time.time()
        if left <= 0:
            st.add("server", "gave up", f"server give-up time reached before option set {i}")
            return None, sm
        ts = time.time()
        proc = sm.start_server(dict(cfg, server_wait_s=max(60, min(float(cfg.get("server_wait_s", 1200)), left))), extra)
        if proc is not None:
            s = smoke(base)
            if not s.get("error") and "4" in (s.get("content") or "") + (s.get("reasoning") or ""):
                cfg["server_option_set"] = {"index": i, "args": extra}
                st.add("server", "ok", f"option set {i} up after {time.time() - ts:.0f} s", option_set=i, args=extra,
                       smoke=s)
                return proc, sm
            st.add("smoke", "failed", json.dumps(s)[:300], option_set=i)
            sm.kill(proc)
        st.add("server", "failed", f"option set {i}: {extra}", option_set=i)
        time.sleep(float(cfg.get("ladder_pause_s", 20)))
    return None, sm


# ----------------------------------------------------------------------------
# deadline-aware caps (version 5, lever A2-D1 part 1)

# run_direct.answer sends its rescue request only while deadline - now > 120 s, and run_routed passes
# deadline = collect_end - 60: a rescue can be sent only until collect_end - 180 s
RESCUE_GATE_S = 60 + 120
DEADLINE_CAPS = {
    "on": True,
    # per-stream decode rate before any request has finished, below every rate measured on the 4 L4 (Q27F direct run,
    # 20 in flight: 18.8-21.2 tok/s per sample; hybrid v4 commit run, 7 in flight: 28-30 tok/s)
    "fallback_tok_s": 12.0,
    "rate_factor": 0.85,          # the measured rate is planned with at this fraction
    "rate_window": 20,            # the estimate sums the last 20 finished requests ...
    "rate_min_tokens": 2048,      # ... of at least 2,048 generated tokens that ended at 'stop' or 'length'
    "margin_s": 120.0,            # left free after the planned end of the rescue request
    "min_answer_tokens": 4096,    # a sample whose cap would be smaller is not issued
    "min_rescue_tokens": 1024,    # a rescue request whose cap would be smaller is not sent
    "zero_token_slack_s": 60.0,   # a 0-token error ending this close to collection (or later) was cut there
}


class RateMeter:
    """Per-stream decode rate: generated tokens over wall seconds, summed over the last `window` finished requests
    with at least `min_tokens` tokens and finish 'stop' or 'length' (errors, cut requests and short replies such as
    most rescues are not measured). estimate() gives the rate to plan with: the measured rate x `factor`, or
    `fallback` until a request has been measured."""

    def __init__(self, fallback_tok_s, factor=0.85, window=20, min_tokens=2048):
        self.fallback = float(fallback_tok_s)
        self.factor = float(factor)
        self.min_tokens = int(min_tokens)
        self.rows = collections.deque(maxlen=int(window))
        self.n_measured = 0
        self.lock = threading.Lock()

    def add(self, tokens, seconds, finish):
        """Record one finished request; returns whether it was measured."""
        tokens, seconds = int(tokens or 0), float(seconds or 0.0)
        if finish not in ("stop", "length") or tokens < self.min_tokens or seconds <= 0:
            return False
        with self.lock:
            self.rows.append((tokens, seconds))
            self.n_measured += 1
        return True

    def estimate(self):
        with self.lock:
            rows = list(self.rows)
        if not rows:
            return {"tok_s": self.fallback, "source": "fallback", "measured_tok_s": None, "n": 0}
        measured = sum(r[0] for r in rows) / sum(r[1] for r in rows)
        return {"tok_s": self.factor * measured, "source": "measured", "measured_tok_s": measured, "n": len(rows)}


class DeadlineCaps:
    """The cap of a sample's answer request (chosen when the sample is issued) and of its rescue request (when the
    rescue is about to be sent), from the run's own deadlines and the RateMeter's planning rate:

        answer:  end_by = min(collect - rescue_tokens / rate, issue_end, collect - RESCUE_GATE_S) - margin
                 budget = rate x (end_by - now); 'normal' (max_tokens untouched) when budget >= the normal cap, else
                 'deadline' with max_tokens = budget (the run does not issue it below min_answer_tokens)
        rescue:  budget = rate x (collect - now - margin); 'normal' when >= rescue_tokens, 'deadline' when
                 >= min_rescue_tokens, else 'skip' (not sent)

    Parameters: DEADLINE_CAPS, updated from cfg["deadline_caps"]."""

    def __init__(self, cfg, issue_end, collect_end, t0):
        self.p = dict(DEADLINE_CAPS, **(cfg.get("deadline_caps") or {}))
        self.on = bool(self.p["on"])
        self.issue_end, self.collect_end, self.t0 = float(issue_end), float(collect_end), float(t0)
        self.rescue_tokens = int(cfg.get("rescue_tokens", 8192))
        self.min_answer_tokens = int(self.p["min_answer_tokens"])
        self.min_rescue_tokens = int(self.p["min_rescue_tokens"])
        self.meter = RateMeter(self.p["fallback_tok_s"], self.p["rate_factor"], self.p["rate_window"],
                               self.p["rate_min_tokens"])

    def _s(self, epoch):
        return round(epoch - self.t0, 1)

    def _rate(self, now):
        r = self.meter.estimate()
        rec = {"at_s": self._s(now), "rate_tok_s": round(r["tok_s"], 2), "rate_source": r["source"],
               "rate_measured_tok_s": None if r["measured_tok_s"] is None else round(r["measured_tok_s"], 2),
               "rate_n": r["n"]}
        return r["tok_s"], rec

    def end_by(self, rate):
        """(epoch by which an answer must end, the binding term)."""
        terms = {"collection less rescue": self.collect_end - self.rescue_tokens / rate,
                 "issue deadline": self.issue_end,
                 "rescue gate": self.collect_end - RESCUE_GATE_S}
        name = min(terms, key=lambda k: terms[k])
        return terms[name] - float(self.p["margin_s"]), name

    def answer_cap(self, now, normal):
        normal = int(normal)
        rate, rec = self._rate(now)
        rec["normal"] = normal
        if not self.on:
            return dict(rec, kind="normal", max_tokens=normal, why="deadline caps off")
        end_by, term = self.end_by(rate)
        budget = int(rate * (end_by - now))
        rec.update(end_by_s=self._s(end_by), end_by_term=term, budget_tokens=budget)
        if budget >= normal:
            return dict(rec, kind="normal", max_tokens=normal,
                        why=f"far from the deadline: {budget} tokens fit before {rec['end_by_s']} s >= cap {normal}")
        return dict(rec, kind="deadline", max_tokens=max(0, budget),
                    why=f"{normal} tokens at {rate:.1f} tok/s would end after {rec['end_by_s']} s ({term}, less "
                        f"{self.p['margin_s']:.0f} s margin): {max(0, budget)} tokens fit")

    def rescue_cap(self, now, normal):
        normal = int(normal)
        rate, rec = self._rate(now)
        rec["normal"] = normal
        if not self.on:
            return dict(rec, kind="normal", max_tokens=normal, why="deadline caps off")
        budget = int(rate * (self.collect_end - float(self.p["margin_s"]) - now))
        rec["budget_tokens"] = budget
        left = f"{self.collect_end - now:.0f} s to the collection deadline"
        if budget >= normal:
            return dict(rec, kind="normal", max_tokens=normal, why=f"{left}: {budget} tokens fit >= {normal}")
        if budget >= self.min_rescue_tokens:
            return dict(rec, kind="deadline", max_tokens=budget, why=f"{left}: {budget} tokens fit < {normal}")
        return dict(rec, kind="skip", max_tokens=0,
                    why=f"{left}: {max(0, budget)} tokens fit < {self.min_rescue_tokens}: rescue cannot finish")


def cut_at_collection(line, collect_end_s, slack_s=DEADLINE_CAPS["zero_token_slack_s"]):
    """A sample that ended as an error with 0 tokens at the collection deadline (cut by its HTTP timeout)."""
    return (line.get("status") == "error" and int(line.get("gen_tokens") or 0) == 0
            and float(line.get("end_s") or 0.0) >= float(collect_end_s) - float(slack_s))


def deadline_summary(lines, collect_end_s, not_issued=None, slack_s=DEADLINE_CAPS["zero_token_slack_s"]):
    """Counts for the pre-set rule from the sample lines (samples.jsonl); 'line' starts with
    'deadline caps: N reduced, M zero-token at collection'."""
    caps = [ln.get("cap") or {} for ln in lines]
    reduced = [ln for ln, c in zip(lines, caps) if c.get("kind") == "deadline"]
    normal = [ln for ln, c in zip(lines, caps) if c.get("kind") == "normal"]
    zero = [ln for ln in lines if cut_at_collection(ln, collect_end_s, slack_s)]
    zero_other = [ln for ln in lines if int(ln.get("gen_tokens") or 0) == 0 and ln not in zero]
    normal_caps = sorted({c.get("normal") for c in caps if c.get("kind") == "normal"})
    untouched = all(c.get("max_tokens") == c.get("normal") for c in caps if c.get("kind") == "normal")
    rescue = [ln.get("rescue_cap") or {} for ln in lines if ln.get("rescue_cap")]
    out = {
        "reduced": len(reduced), "zero_token_at_collection": len(zero), "normal": len(normal),
        "normal_caps": normal_caps, "normal_untouched": untouched,
        "reduced_finish": {f: sum(1 for ln in reduced if ln.get("finish") == f) for f in ("length", "stop", None)},
        "reduced_status": {s: sum(1 for ln in reduced if ln.get("status") == s) for s in STATUSES},
        "reduced_with_grids": sum(1 for ln in reduced if ln.get("grids")),
        "rescues": {k: sum(1 for r in rescue if r.get("kind") == k) for k in ("normal", "deadline", "skip")},
        "zero_token_other": len(zero_other),
        "zero_token_at_collection_samples": [[ln.get("task_id"), ln.get("sample")] for ln in zero],
        "not_issued": not_issued,
    }
    out["line"] = (f"deadline caps: {len(reduced)} reduced, {len(zero)} zero-token at collection; {len(normal)} at "
                   f"the normal cap {normal_caps} (untouched: {untouched}); reduced samples: finish "
                   f"{out['reduced_finish']}, {out['reduced_with_grids']} with grids; rescues {out['rescues']}; "
                   f"zero-token otherwise {len(zero_other)}" + (f"; not issued: {not_issued}" if not_issued else ""))
    return out


# ----------------------------------------------------------------------------
# version 6: the plan (quarter, band, second samples), the band at 98k and its issue rule

PASS_KIND = {1: "quarter", 2: "band", 3: "second"}
PASS_KIND_BAND_OFF = {1: "quarter", 2: "second", 3: "rest"}     # v6nb: version 5's plan (hybrid_core.issue_plan)
BAND_98K = {
    "on": True,                      # v6nb: False (no band sample; version 5's plan)
    "max_tokens": 98304,             # Q27G (experiments/runs/kaggle-A2-Q27G-cap90-v1): 8/26 right on ranks 32-60
    "thinking_token_budget": 90000,  # Q27G
    "max_in_flight": 16,             # band samples running at once (all samples: cfg["concurrency"], 20)
    "min_fit_tokens": None,          # the issue rule's threshold; None: the band's max_tokens (the full cap must fit)
}


def band_settings(cfg):
    """BAND_98K updated from cfg["band_98k"], with min_fit_tokens resolved (None -> max_tokens)."""
    b = dict(BAND_98K, **(cfg.get("band_98k") or {}))
    b["on"] = bool(b["on"])
    b["max_tokens"], b["thinking_token_budget"] = int(b["max_tokens"]), int(b["thinking_token_budget"])
    b["max_in_flight"] = int(b["max_in_flight"])
    b["min_fit_tokens"] = int(b["max_tokens"] if b.get("min_fit_tokens") is None else b["min_fit_tokens"])
    return b


def v6_plan(route, fraction=HC.BAND_FRACTION):
    """[[task_id, sample, pass]]: pass 1 = sample 0 on the shortest ceil(fraction x n) tasks (the quarter), pass 2 =
    sample 0 on the tasks beyond it (the band), pass 3 = sample 1 on the quarter; each in route order."""
    nq = HC.band_size(len(route), fraction)
    return ([[t, 0, 1] for t in route[:nq]] + [[t, 0, 2] for t in route[nq:]] + [[t, 1, 3] for t in route[:nq]])


def kind_of(pass_no, band_on=True):
    return (PASS_KIND if band_on else PASS_KIND_BAND_OFF).get(int(pass_no), "quarter")


def v6_route_table(pool, challenges, fraction=HC.BAND_FRACTION, band=None, normal=None):
    """The route with each task's size, kind ('quarter' or 'band'), samples planned and first-sample token limits
    and, for the record, B0's confidence key and (n_unique, margin) per output."""
    band = band or band_settings({})
    rows = HC.route_table(pool, challenges, fraction, 2)
    out = []
    for r in rows:
        q = bool(r["band"])                     # hybrid_core's "band" is version 2's name for the quarter
        out.append({"task_id": r["task_id"], "position": r["position"], "size": r["size"],
                    "kind": "quarter" if q else "band", "quarter": q, "band": not q,
                    "samples_planned": 2 if q else 1,
                    "max_tokens": normal if q else band["max_tokens"],
                    "thinking_token_budget": None if q else band["thinking_token_budget"],
                    "b0_key": r["b0_key"], "outputs": r["outputs"]})
    return out


def band_rule(cap, band, min_answer_tokens=0):
    """(allowed, record) for a band task about to be issued with deadline cap `cap` (DeadlineCaps.answer_cap at the
    band's max_tokens): allowed when the tokens that fit before the answer's end_by (budget_tokens; the cap itself when
    the caps are off) are at least min_fit_tokens (and min_answer_tokens)."""
    fit = cap.get("budget_tokens")
    fit = int(cap["max_tokens"] if fit is None else fit)
    need = max(int(band["min_fit_tokens"]), int(min_answer_tokens))
    ok = fit >= need
    rate = cap.get("rate_tok_s") or 0
    rec = {"allowed": ok, "fit_tokens": fit, "min_fit_tokens": need, "band_max_tokens": band["max_tokens"],
           "cap": cap["max_tokens"], "cap_kind": cap["kind"], "rate_tok_s": cap.get("rate_tok_s"),
           "rate_source": cap.get("rate_source"), "end_by_s": cap.get("end_by_s"), "at_s": cap.get("at_s"),
           "full_cap_min": round(band["max_tokens"] / rate / 60, 1) if rate else None}
    rec["why"] = (f"{fit} tokens fit before {cap.get('end_by_s')} s at {rate} tok/s "
                  f"{'>=' if ok else '<'} {need}" + ("" if ok else ": the band sample could not finish; band closed"))
    return ok, rec


# ----------------------------------------------------------------------------
# version 6: per-output grid parsing (lever A2-K5), installed into run_direct in place of parse_grids

GRID_BLOCK = re.compile(r"```(?:grid)?[^\n]*\n(.*?)```", re.S)          # run_direct.parse_grids' block pattern


def _block_rows(block):
    """run_direct.parse_grids' row cleaning: non-empty rows, spaces and commas removed."""
    return [r.strip().replace(" ", "").replace(",", "") for r in block.strip().splitlines() if r.strip()]


def parse_outputs(text, n_tests):
    """Per test output, in test order: (state, grid) with state 'grid', 'ragged' or 'missing'. The last n_tests
    fenced blocks whose rows are all digits are matched to the outputs by position (fewer blocks: the first outputs
    are missing); each is kept if its rows have one length."""
    n = int(n_tests)
    digit = [rows for rows in (_block_rows(b) for b in GRID_BLOCK.findall(text or ""))
             if rows and all(r.isdigit() for r in rows)]
    last = ([None] * max(0, n - len(digit)) + digit)[-n:] if n > 0 else []
    out = []
    for rows in last:
        if rows is None:
            out.append(("missing", None))
            continue
        try:
            grid = [[int(c) for c in r] for r in rows]
        except ValueError:                    # a digit character int() does not read (e.g. a superscript)
            grid = None
        out.append(("grid", grid) if grid is not None and len({len(r) for r in rows}) == 1 else ("ragged", None))
    return out


def parse_grids_per_output(text, n_tests):
    """run_direct.parse_grids' contract per output: the list of test grids (None for an output without a
    rectangular block), or None when no output has one."""
    grids = [g for _, g in parse_outputs(text, n_tests)]
    return grids if any(g is not None for g in grids) else None


# one ragged block for output 1 and a valid one for output 2: per output -> [None, grid]; run_direct's original
# parser drops the ragged block and returns None (fewer valid blocks than test outputs)
PARSER_SELF_TEST = ("The rule is applied.\n\n```grid\n12\n345\n```\n\n```grid\n12\n34\n```\n", 2, [None, [[1, 2], [3, 4]]])


def install_parser(cfg):
    """Replace run_direct.parse_grids (looked up at call time by run_direct.answer) by the per-output parser unless
    cfg["parser"] == "all-or-nothing"; returns the record (name, installed, self-test through run_direct)."""
    import run_direct
    want = cfg.get("parser", "per-output")
    if want == "per-output":
        run_direct.parse_grids = parse_grids_per_output
    text, n, expect = PARSER_SELF_TEST
    try:
        got = run_direct.parse_grids(text, n)
    except Exception as exc:
        got = f"error: {exc!r}"
    active = run_direct.parse_grids is parse_grids_per_output
    return {"parser": "per-output" if active else "all-or-nothing", "requested": want, "installed": active,
            "self_test": got == expect if active else got is None, "self_test_result": got}


def parse_record(texts, n_tests, status):
    """Which test outputs parsed, and from which reply: the answer's content ('answered'), the rescue's content or
    reasoning ('rescued'); for a sample without grids, the answer and the rescue as they parse."""
    ans = [x for x in texts if x.get("kind") == "answer"]
    resc = [x for x in texts if x.get("kind") == "rescue"]
    states = lambda t: [s for s, _ in parse_outputs(t, n_tests)]  # noqa: E731
    rec = {"n_tests": int(n_tests)}
    if status == "answered" and ans:
        rec.update({"from": "answer", "outputs": states(ans[-1].get("content"))})
    elif status == "rescued" and resc:
        c = resc[-1].get("content") or ""
        src = "rescue" if parse_grids_per_output(c, n_tests) else "rescue reasoning"
        rec.update({"from": src, "outputs": states(c if src == "rescue" else resc[-1].get("reasoning"))})
    else:
        rec.update({"from": None, "answer": states(ans[-1].get("content")) if ans else None,
                    "rescue": states(resc[-1].get("content")) if resc else None})
    rec["parsed"] = sum(1 for s in rec.get("outputs") or [] if s == "grid")
    rec["ragged"] = sum(1 for k in ("outputs", "answer", "rescue") for s in (rec.get(k) or []) if s == "ragged")
    return rec


# ----------------------------------------------------------------------------
# version 6: the server's KV cache use and preemptions (Prometheus /metrics while running; the server log at the end)

def parse_metrics(text):
    """{kv_cache_usage_pct, preemptions, running, waiting} from vLLM's /metrics text (None when absent)."""
    out = {"kv_cache_usage_pct": None, "preemptions": None, "running": None, "waiting": None}
    for line in (text or "").splitlines():
        if not line or line.startswith("#"):
            continue
        try:
            head, value = line.rsplit(" ", 1)
            v = float(value)
        except ValueError:
            continue
        name = head.split("{", 1)[0]
        if "cache_usage_perc" in name:
            pct = 100.0 * v if v <= 1.0 else v
            out["kv_cache_usage_pct"] = max(out["kv_cache_usage_pct"] or 0.0, pct)
        elif "preempt" in name and not name.endswith("_created"):
            out["preemptions"] = (out["preemptions"] or 0.0) + v
        elif name.endswith("num_requests_running"):
            out["running"] = (out["running"] or 0.0) + v
        elif name.endswith("num_requests_waiting"):
            out["waiting"] = (out["waiting"] or 0.0) + v
    return out


class ServerMonitor:
    """Polls <server>/metrics every `every_s` seconds in a daemon thread: peak KV cache use, preemptions (the
    counter's last value), peak running and waiting requests, with times; a failed poll is counted, never raised."""

    def __init__(self, url, t0, every_s=30.0):
        self.url, self.t0, self.every_s = url, float(t0), float(every_s)
        self.stop_event = threading.Event()
        self.rec = {"url": url, "polls": 0, "errors": 0, "last_error": None, "kv_cache_usage_pct_max": None,
                    "kv_peak_s": None, "preemptions": None, "running_max": None, "waiting_max": None, "series": []}
        self.thread = threading.Thread(target=self._loop, daemon=True)

    def poll(self):
        try:
            with urllib.request.urlopen(self.url, timeout=5) as r:
                m = parse_metrics(r.read().decode(errors="replace"))
        except Exception as exc:
            self.rec["errors"] += 1
            self.rec["last_error"] = repr(exc)[:200]
            return None
        s = round(time.time() - self.t0, 1)
        rec = self.rec
        rec["polls"] += 1
        if m["kv_cache_usage_pct"] is not None and m["kv_cache_usage_pct"] > (rec["kv_cache_usage_pct_max"] or -1):
            rec["kv_cache_usage_pct_max"], rec["kv_peak_s"] = round(m["kv_cache_usage_pct"], 2), s
        if m["preemptions"] is not None:
            rec["preemptions"] = int(m["preemptions"])
        for k in ("running", "waiting"):
            if m[k] is not None:
                rec[f"{k}_max"] = max(rec[f"{k}_max"] or 0, int(m[k]))
        if rec["polls"] % 10 == 1:                    # every 5 min at the default interval
            rec["series"].append([s, m["kv_cache_usage_pct"], m["running"], m["waiting"], m["preemptions"]])
        return m

    def _loop(self):
        while not self.stop_event.is_set():
            self.poll()
            self.stop_event.wait(self.every_s)

    def start(self):
        self.thread.start()
        return self

    def stop(self):
        self.stop_event.set()
        self.thread.join(timeout=10)
        self.poll()
        return self.rec


LOG_STATS = re.compile(r"Running: (\d+) reqs,.*?Waiting: (\d+) reqs,.*?GPU KV cache usage: ([\d.]+)%")
LOG_CONCURRENCY = re.compile(r"Maximum concurrency for ([\d,]+) tokens per request: ([\d.]+)x")


def parse_server_log(path):
    """KV cache use, running/waiting peaks, preemption mentions and the KV capacity line from a vLLM server log."""
    out = {"path": path, "found": bool(path) and os.path.isfile(path)}
    if not out["found"]:
        return out
    kv, run_max, wait_max, pre, n, conc = 0.0, 0, 0, [], 0, None
    with open(path, errors="replace") as f:
        for line in f:
            m = LOG_STATS.search(line)
            if m:
                n += 1
                run_max, wait_max = max(run_max, int(m.group(1))), max(wait_max, int(m.group(2)))
                kv = max(kv, float(m.group(3)))
            c = LOG_CONCURRENCY.search(line)
            if c:
                conc = {"tokens_per_request": int(c.group(1).replace(",", "")), "max_concurrency": float(c.group(2))}
            if "preempt" in line.lower():
                pre.append(line.strip()[-200:])
    out.update(stats_lines=n, kv_cache_usage_pct_max=kv if n else None, running_max=run_max if n else None,
               waiting_max=wait_max if n else None, preemption_lines=len(pre), preemption_examples=pre[:3],
               kv_capacity=conc)
    return out


# ----------------------------------------------------------------------------
# version 6: B0's DFS record (one JSON line per task from arc_solver.py) and the run summary with the four checks

H2_DFS_MEAN_S = 320.3        # H2 (experiments/runs/kaggle-A2-H2-v1/pool, 40 dev tasks): mean DFS seconds per task


def read_b0_dfs(path):
    """{task_id: record}, the last line per task (a line cut by a kill is skipped)."""
    recs = {}
    for ln in HC.read_sample_lines(path) if path else []:
        if isinstance(ln, dict) and ln.get("task_id"):
            recs[ln["task_id"]] = ln
    return recs


def b0_dfs_summary(path, reference=None):
    """B0's per-task DFS seconds and batches (version 6: 2 batches per test output, no transposed view) against H2:
    the mean over all tasks with a record and over those that decoded every batch, H2's mean and, with
    reference {"h2_dfs_s": {task: s}}, H2 on the same complete tasks."""
    recs = read_b0_dfs(path)
    ref = (reference or {}).get("h2_dfs_s") or {}
    h2_mean = float((reference or {}).get("h2_dfs_mean_s") or H2_DFS_MEAN_S)
    rows = list(recs.values())
    complete = [r for r in rows if r.get("batches_decoded") == r.get("batches_built")]
    mean = lambda xs: round(sum(xs) / len(xs), 1) if xs else None  # noqa: E731
    same = [r for r in complete if r["task_id"] in ref]
    out = {"path": path, "tasks": len(rows), "complete": len(complete),
           "dfs_mean_s": mean([float(r.get("dfs_s") or 0) for r in rows]),
           "dfs_mean_s_complete": mean([float(r.get("dfs_s") or 0) for r in complete]),
           "h2_dfs_mean_s": h2_mean,
           "batches_per_output": sorted({round(r["batches_built"] / r["outputs"], 2) for r in rows if r.get("outputs")}),
           "transposed_batches": sum(int(r.get("transposed_batches") or 0) for r in rows),
           "same_tasks": len(same),
           "dfs_s_same_tasks": round(sum(float(r["dfs_s"]) for r in same), 1) if same else None,
           "h2_dfs_s_same_tasks": round(sum(float(ref[r["task_id"]]) for r in same), 1) if same else None,
           "per_task": {r["task_id"]: [r.get("dfs_s"), r.get("batches_decoded"), r.get("batches_built"),
                                       r.get("outputs")] for r in rows}}
    if same:
        out["ratio_same_tasks"] = round(out["dfs_s_same_tasks"] / max(out["h2_dfs_s_same_tasks"], 1e-9), 3)
    m = out["dfs_mean_s_complete"] if out["dfs_mean_s_complete"] is not None else out["dfs_mean_s"]
    out["ratio_to_h2_mean"] = round(m / h2_mean, 3) if m is not None else None
    return out


def v6_checks(dfs, dsum, band, parser, lines):
    """The owner's four commit-run checks, each (verdict, detail); verdict True, False or None (no data)."""
    m = dfs.get("dfs_mean_s_complete") if dfs.get("dfs_mean_s_complete") is not None else dfs.get("dfs_mean_s")
    c1 = (None if m is None else m <= 0.6 * dfs["h2_dfs_mean_s"],
          "no B0 DFS record" if m is None else f"{m:.0f} s vs H2 {dfs['h2_dfs_mean_s']:.0f} s "
          f"({100 * (m / dfs['h2_dfs_mean_s'] - 1):+.0f}%; {dfs['complete']} of {dfs['tasks']} tasks with every batch)")
    z = dsum["zero_token_at_collection"]
    c2 = (z <= 2, f"{z} zero-token at collection")
    bad = sorted({d.get("plan_index") for d in band["decisions"]
                  if d.get("issued") and not (d.get("allowed") and d["fit_tokens"] >= d["min_fit_tokens"])}
                 | {ln.get("plan_index") for ln in lines
                    if ln.get("kind") == "band" and not (ln.get("band_rule") or {}).get("allowed")}, key=str)
    c3 = (not bad, f"{band['issued']} issued, all allowed; {band['skipped_by_rule']} skipped by the rule" if not bad
          else f"plan entries issued against the rule: {bad}")
    if not band.get("on", True):                 # v6nb: the band is off, so none may be issued
        c3 = (not bad and band["issued"] == 0, f"band off: {band['issued']} issued")
    parsed = [ln.get("parse") for ln in lines if ln.get("parse")]
    c4 = (bool(parser.get("installed") and parser.get("self_test")) and len(parsed) == len(lines),
          f"run_direct.parse_grids per-output: {parser.get('installed')}, self-test {parser.get('self_test')}, "
          f"parse records {len(parsed)} of {len(lines)} samples")
    return {"1_b0_dfs_40pct_below_h2": c1, "2_zero_token_at_collection_le_2": c2,
            "3_band_issued_only_when_allowed": c3, "4_per_output_parser_active": c4}


def v6_summary(lines, events, collect_end_s, dfs, parser, band_cfg, server=None, not_issued=None):
    """The run's version-6 record and the line 'v6: B0 DFS mean s X (H2 320), band issued N skipped M, deadline caps
    ...' with the four checks, from the sample lines and the plan log (also usable after a kill)."""
    dsum = deadline_summary(lines, collect_end_s, not_issued=not_issued)
    dec = [dict(e["band_rule"], plan_index=e.get("plan_index"), task_id=e.get("task_id"),
                issued=e.get("event") == "issued") for e in events if e.get("band_rule")]
    issued = [e for e in events if e.get("kind") == "band" and e.get("event") == "issued"]
    skipped = [e for e in events if e.get("kind") == "band" and e.get("event") == "skipped"]
    not_started = [e for e in events if e.get("kind") == "band" and e.get("event") == "not started"]
    band = {"on": band_cfg["on"], "issued": len(issued), "skipped_by_rule": len(skipped), "not_started": len(not_started),
            "max_in_flight_seen": max((int((e.get("in_flight") or {}).get("band", 0)) for e in issued), default=0),
            "max_in_flight": band_cfg["max_in_flight"], "min_fit_tokens": band_cfg["min_fit_tokens"],
            "max_tokens": band_cfg["max_tokens"], "thinking_token_budget": band_cfg["thinking_token_budget"],
            "decisions": dec}
    kinds = {k: [ln for ln in lines if ln.get("kind") == k] for k in ("quarter", "band", "second", "rest")}
    par = [ln.get("parse") or {} for ln in lines]
    parse = {"samples": len(lines), "outputs_parsed": sum(p.get("parsed", 0) for p in par),
             "outputs": sum(p.get("n_tests", 0) for p in par if p.get("from")),
             "samples_partial": sum(1 for p in par if p.get("from") and 0 < p.get("parsed", 0) < p.get("n_tests", 0)),
             "ragged": sum(p.get("ragged", 0) for p in par)}
    checks = v6_checks(dfs, dsum, band, parser, lines)
    srv = server or {}
    mon, log = srv.get("metrics") or {}, srv.get("log") or {}
    kv = [x for x in (mon.get("kv_cache_usage_pct_max"), log.get("kv_cache_usage_pct_max")) if x is not None]
    pre = [x for x in (mon.get("preemptions"), log.get("preemption_lines")) if x is not None]
    dfs_m = dfs.get("dfs_mean_s_complete") if dfs.get("dfs_mean_s_complete") is not None else dfs.get("dfs_mean_s")
    line = (f"v6: B0 DFS mean s {'n/a' if dfs_m is None else f'{dfs_m:.0f}'} (H2 {dfs['h2_dfs_mean_s']:.0f}; "
            f"{dfs['complete']} of {dfs['tasks']} tasks with every batch, batches per output "
            f"{dfs['batches_per_output']}, transposed {dfs['transposed_batches']}"
            + (f", H2 on the same tasks {dfs['h2_dfs_s_same_tasks'] / max(dfs['same_tasks'], 1):.0f}"
               if dfs.get("same_tasks") else "") + "), "
            + (f"band issued {band['issued']} skipped {band['skipped_by_rule']} (rule: fit >= {band['min_fit_tokens']} "
               f"tokens; {band['not_started']} not started; band in flight max {band['max_in_flight_seen']} of "
               f"{band['max_in_flight']})" if band["on"] else f"band off: {band['issued']} issued")
            + f", {dsum['line']}; samples by kind "
            f"{ {k: len(v) for k, v in kinds.items()} }; parser {parser.get('parser')} (self-test "
            f"{parser.get('self_test')}): {parse['outputs_parsed']} outputs parsed, {parse['samples_partial']} samples "
            f"partial, {parse['ragged']} ragged blocks; server KV peak {max(kv) if kv else 'n/a'}%, preemptions "
            f"{max(pre) if pre else 'n/a'}")
    verdict = lambda v: {True: "PASS", False: "FAIL", None: "NO DATA"}[v]  # noqa: E731
    checks_line = "v6 checks: " + "; ".join(f"({k.split('_', 1)[0]}) {k.split('_', 1)[1].replace('_', ' ')}: "
                                             f"{verdict(v)} ({d})" for k, (v, d) in checks.items())
    return {"line": line, "checks_line": checks_line, "checks": {k: {"pass": v, "detail": d} for k, (v, d) in checks.items()},
            "b0_dfs": dfs, "band": band, "parse": parse, "parser": parser, "server": srv, "deadline_caps": dsum,
            "samples_by_kind": {k: len(v) for k, v in kinds.items()}}


# ----------------------------------------------------------------------------
# direct answers down the route

class SampleClient:
    """The shared ChatClient as one sample sees it (run_direct.answer reads .max_tokens and calls .chat): no request
    is sent after the issue deadline; every request's tokens, finish reason and seconds are kept. With `caps`
    (version 5): every finished request feeds the rate meter, and the rescue request is capped or not sent
    (DeadlineCaps.rescue_cap); the answer request's cap was chosen when the sample was issued (`cap`)."""

    def __init__(self, client, issue_end, caps=None, cap=None):
        self.client = client
        self.issue_end = float(issue_end)
        self.max_tokens = client.max_tokens
        self.requests = []
        self.texts = []
        self.caps = caps
        self.cap = cap
        self.rescue_cap = None

    def chat(self, messages, **kw):
        rescue = (kw.get("template_kwargs") or {}).get("enable_thinking") is False
        kind = "rescue" if rescue else "answer"
        if time.time() >= self.issue_end:
            self.requests.append({"kind": kind, "not_sent": "issue deadline", "max_tokens": kw.get("max_tokens"),
                                  "completion_tokens": 0, "prompt_tokens": 0, "finish": None, "s": 0.0})
            return {"error": "issue deadline passed: request not sent", "context_overflow": False}
        extra = {}
        if self.caps is not None:
            extra["cap"] = (self.cap or {}).get("kind", "normal")
            if rescue:
                rc = self.rescue_cap = self.caps.rescue_cap(time.time(), kw.get("max_tokens") or self.client.max_tokens)
                extra["cap"] = rc["kind"]
                if rc["kind"] == "skip":
                    self.requests.append({"kind": kind, "not_sent": "deadline cap: rescue cannot finish",
                                          "max_tokens": kw.get("max_tokens"), "completion_tokens": 0,
                                          "prompt_tokens": 0, "finish": None, "s": 0.0, "cap": "skip"})
                    return {"error": "deadline cap: the rescue cannot finish before the collection deadline: request "
                                     "not sent", "context_overflow": False}
                if rc["kind"] == "deadline":
                    kw["max_tokens"] = rc["max_tokens"]
        t = time.time()
        r = self.client.chat(messages, **kw)
        s = time.time() - t
        if self.caps is not None:
            self.caps.meter.add(r.get("completion_tokens"), s, r.get("finish_reason"))
        self.requests.append(dict({"kind": kind, "max_tokens": kw.get("max_tokens"), "seed": kw.get("seed"),
                                   "completion_tokens": int(r.get("completion_tokens") or 0),
                                   "prompt_tokens": int(r.get("prompt_tokens") or 0), "finish": r.get("finish_reason"),
                                   "s": round(s, 2), "error": (r.get("error") or "")[:300] or None,
                                   "context_overflow": bool(r.get("context_overflow")),
                                   "thinking_token_budget": getattr(self.client, "thinking_token_budget", None)},
                                  **extra))
        self.texts.append({"kind": kind, "content": r.get("content") or "", "reasoning": r.get("reasoning") or ""})
        return r


def status_of(raw):
    raw = raw or ""
    return raw if raw in ("answered", "rescued", "no grid") else "error"


def sample_line(task_id, sample, pass_no, plan_index, position, seq, rec, requests, t0, ts, now, issued=None,
                cap=None, rescue_cap=None, collect_end=None, kind=None, band_rule_rec=None, in_flight=None, parse=None):
    grids = rec.get("grids")
    line = {"task_id": task_id, "sample": int(sample), "pass": int(pass_no), "plan_index": plan_index,
            "kind": kind or kind_of(pass_no), "route_position": position, "seq": seq, "seed": seed_of(sample),
            "status": status_of(rec.get("status")), "status_detail": str(rec.get("status", ""))[:300],
            "finish": rec.get("finish"), "rescue_finish": rec.get("rescue_finish"),
            "gen_tokens": int(rec.get("gen_tokens") or 0),
            "prompt_tokens": sum(int(r.get("prompt_tokens") or 0) for r in requests),
            "n_requests": sum(1 for r in requests if not r.get("not_sent")), "requests": requests,
            "grids": grids if grids else None, "wall_s": round(now - ts, 2),
            "start_s": round(ts - t0, 2), "end_s": round(now - t0, 2)}
    if cap is not None:                  # version 5: the deadline-cap record (issue time, cap, why, rate)
        line.update(issue_s=round((issued if issued is not None else ts) - t0, 2), cap=cap, rescue_cap=rescue_cap,
                    rescue_error=rec.get("rescue_error"))
        if collect_end is not None:
            line["cut_at_collection"] = cut_at_collection(line, collect_end - t0)
    # version 6: the band issue rule's decision (band samples), the samples in flight when it was issued (itself
    # included) and which test outputs parsed
    line.update(band_rule=band_rule_rec, in_flight=in_flight, parse=parse)
    return line


def plan_of(cfg, ch):
    """[[task_id, sample, pass]] in issue order: cfg["plan"], else v6_plan over cfg["route"]."""
    plan = cfg.get("plan")
    if plan is None:
        route = [t for t in cfg["route"] if t in ch]
        plan = v6_plan(route, cfg.get("band_fraction", HC.BAND_FRACTION))
    return [[t, int(k), int(p)] for t, k, p in plan if t in ch]


def differs_everywhere(grids, b0_entries):
    """True when `grids` holds, for every test output, a valid grid that differs from B0's attempt_1."""
    if not isinstance(grids, list) or not isinstance(b0_entries, list) or len(grids) < len(b0_entries):
        return False
    return all(HC.valid_grid(g) and g != e.get("attempt_1") for g, e in zip(grids, b0_entries))


def run_routed(cfg, st=None):
    """Direct answers down the plan (version 6: quarter, then the band beside second samples, then second samples);
    returns the summary (also <out_dir>/_p2_run.json)."""
    from client import ChatClient
    from run_direct import answer

    t0 = float(cfg["run_t0"])
    out = cfg["out_dir"]
    os.makedirs(os.path.join(out, "transcripts"), exist_ok=True)
    with open(cfg["challenges"]) as f:
        ch = json.load(f)
    route = [t for t in cfg["route"] if t in ch]
    position = {t: i for i, t in enumerate(route)}
    plan = plan_of(cfg, ch)
    b0, _ = HC.read_submission(cfg["b0_submission"], ch)
    b0 = b0 or {}
    issue_end, collect_end = float(cfg["issue_end_epoch"]), float(cfg["collect_end_epoch"])
    jsonl = os.path.join(out, "samples.jsonl")
    band = band_settings(cfg)
    kind_p = lambda p: kind_of(p, band["on"])  # noqa: E731  (v6nb: version 5's passes when the band is off)
    # as run_submit.main builds the client; the band's own client carries Q27G's limits (98,304 tokens, thinking
    # budget 90,000), every other setting the same
    common = dict(temperature=cfg.get("temperature", 0.6), top_p=cfg.get("top_p", 0.95), top_k=cfg.get("top_k", 20),
                  thinking=cfg.get("thinking", True), reasoning_effort=cfg.get("reasoning_effort"),
                  template_kwargs=cfg.get("template_kwargs"))
    client = ChatClient(cfg["base_url"], cfg["model"], max_tokens=cfg.get("max_tokens", 16384),
                        thinking_token_budget=cfg.get("thinking_token_budget"), **common)
    band_client = ChatClient(cfg["base_url"], cfg["model"], max_tokens=band["max_tokens"],
                             thinking_token_budget=band["thinking_token_budget"], **common)
    parser = install_parser(cfg)
    lock = threading.Lock()
    cond = threading.Condition(lock)
    concurrency = int(cfg.get("concurrency", 16))
    in_flight = {"total": 0, "band": 0}
    seq = [0]
    count = {}
    not_started, skipped, issued, band_skipped = [], [], [], []
    first_grids = {}                     # task -> sample 0's grids (once finished)
    merge_last = [{}]
    plan_log = os.path.join(out, "_plan_log.jsonl")
    # version 5: deadline-aware caps on the run's own deadlines
    caps = DeadlineCaps(cfg, issue_end, collect_end, t0)
    normal_cap = int(cfg.get("max_tokens") or client.max_tokens)
    stopped = {}                         # why issuing stopped before the issue deadline (a cap below the minimum)
    events = []
    base = cfg["base_url"].rstrip("/")
    monitor = ServerMonitor((base[:-3] if base.endswith("/v1") else base) + "/metrics", t0,
                            float(cfg.get("metrics_every_s", 30))).start()

    def log_event(i, t, k, p, event, **extra):
        row = dict({"plan_index": i, "task_id": t, "sample": k, "pass": p, "kind": kind_p(p), "event": event,
                    "s": round(time.time() - t0, 2)}, **extra)
        events.append(row)
        with open(plan_log, "a") as f:
            f.write(json.dumps(row) + "\n")

    def work(i, t, k, p, cap, issued_at, rule, flight):
        try:
            sample(i, t, k, p, cap, issued_at, rule, flight)
        finally:
            with cond:
                in_flight["total"] -= 1
                if kind_p(p) == "band":
                    in_flight["band"] -= 1
                cond.notify_all()

    def sample(i, t, k, p, cap, issued_at, rule, flight):
        ts = time.time()
        kind = kind_p(p)
        sc = SampleClient(band_client if kind == "band" else client, issue_end, caps, cap)
        # a 'normal' cap leaves the request as version 4 sends it (the band: its own 98,304); a 'deadline' cap
        # replaces max_tokens
        mt = cap["max_tokens"] if cap["kind"] == "deadline" else (band["max_tokens"] if kind == "band"
                                                                  else cfg.get("max_tokens"))
        try:
            # deadline = collect_end - 60: run_direct.answer's request timeout is deadline - now + 60
            rec = answer(sc, ch[t], seed_of(k), collect_end - 60, max_tokens=mt,
                         rescue_tokens=cfg.get("rescue_tokens", 8192))
        except Exception as exc:
            rec = {"status": f"error: {type(exc).__name__}: {exc}"[:300], "grids": None,
                   "gen_tokens": sum(r["completion_tokens"] for r in sc.requests)}
        now = time.time()
        try:
            prs = parse_record(sc.texts, len(ch[t]["test"]), status_of(rec.get("status")))
        except Exception as exc:
            prs = {"error": repr(exc)[:200]}
        with lock:
            line = sample_line(t, k, p, i, position.get(t), seq[0], rec, sc.requests, t0, ts, now, issued=issued_at,
                               cap=cap, rescue_cap=sc.rescue_cap, collect_end=collect_end, kind=kind,
                               band_rule_rec=rule, in_flight=flight, parse=prs)
            seq[0] += 1
            if k == 0 and line["grids"]:
                first_grids[t] = line["grids"]
            with open(jsonl, "a") as f:
                f.write(json.dumps(line) + "\n")
                f.flush()
            HC.write_json_atomic(os.path.join(out, "transcripts", f"{t}.{k}.json"),
                                 {"task_id": t, "sample": k, "pass": p, "kind": kind, "status": rec.get("status"),
                                  "requests": sc.requests, "texts": sc.texts})
            merge_last[0] = HC.merge_files(cfg["b0_submission"], jsonl, cfg["submission"], ch,
                                           report_path=os.path.join(out, "_merge.json"))
            count[line["status"]] = count.get(line["status"], 0) + 1
            rc = sc.rescue_cap
            print(f"[{now - t0:7.0f}s] plan #{i} {kind} pass {p} {t} s{k} {line['status']:8s} tok "
                  f"{line['gen_tokens']:6d} finish {line['finish']} wall {line['wall_s']:7.1f}s | issued "
                  f"{line['issue_s']:.0f}s cap {cap['max_tokens']} {cap['kind']} ({cap['rate_source']} "
                  f"{cap['rate_tok_s']} tok/s)"
                  + (f" band rule {'allowed' if rule['allowed'] else 'REFUSED'} (fit {rule['fit_tokens']} >= "
                     f"{rule['min_fit_tokens']})" if rule else "")
                  + (f" rescue {rc['kind']} {rc['max_tokens']} finish {line['rescue_finish']}" if rc else "")
                  + (" CUT AT COLLECTION" if line.get("cut_at_collection") else "")
                  + f" | parse {prs.get('from')} {prs.get('outputs') or [prs.get('answer'), prs.get('rescue')]}"
                  + f" | {count} | merge {merge_last[0].get('status')}, replaced {merge_last[0].get('replaced')}",
                  flush=True)

    passes = sorted({p for _, _, p in plan})
    queues = {k: collections.deque(i for i, (_, _, p) in enumerate(plan) if kind_p(p) == k)
              for k in ("quarter", "band", "second", "rest")}
    print(f"phase 2: {len(route)} tasks (shortest first: {route[:5]}), {len(plan)} samples planned "
          f"({', '.join(f'pass {p} ({kind_p(p)}): {sum(1 for j in plan if j[2] == p)}' for p in passes)}), "
          f"concurrency {concurrency}, max_tokens {cfg.get('max_tokens')} (thinking {cfg.get('thinking_token_budget')}),"
          + (f" band max_tokens {band['max_tokens']} (thinking {band['thinking_token_budget']}), at most "
             f"{band['max_in_flight']} band samples in flight, band issue rule: fit >= {band['min_fit_tokens']} tokens; "
             if band["on"] else " band off (version 5's plan); ")
          + f"rescue {cfg.get('rescue_tokens')}; max_model_len {cfg.get('max_model_len')}, max_num_seqs "
          f"{cfg.get('max_num_seqs')}; issue deadline in {issue_end - time.time():.0f} s, collect in "
          f"{collect_end - time.time():.0f} s", flush=True)
    print(f"deadline caps: {'on' if caps.on else 'off'} {caps.p}; issue {issue_end - t0:.0f} s, collect "
          f"{collect_end - t0:.0f} s, rescue gate {collect_end - RESCUE_GATE_S - t0:.0f} s after start", flush=True)
    print(f"parser: {parser['parser']} (run_direct.parse_grids replaced: {parser['installed']}, self-test "
          f"{parser['self_test']}: {parser['self_test_result']})", flush=True)
    ta = time.time()

    def take_next():
        """(plan index, kind) of the next entry to issue, or None when the next one must wait for a slot of its own
        kind (the band at its cap and no second sample left). Quarter first; then the band while it has a slot,
        second samples beside it otherwise; second samples alone once the band is done or closed; then the rest
        (v6nb, band off: version 5's pass 3)."""
        if queues["quarter"]:
            return queues["quarter"].popleft(), "quarter"
        if queues["band"]:
            if in_flight["band"] < band["max_in_flight"]:
                return queues["band"].popleft(), "band"
            if queues["second"]:
                return queues["second"].popleft(), "second"
            return None
        if queues["second"]:
            return queues["second"].popleft(), "second"
        if queues["rest"]:
            return queues["rest"].popleft(), "rest"
        return None

    # issue: a free slot (concurrency) goes to the next entry by take_next; at the issue deadline, or when a cap
    # falls below the minimum, every entry left counts as not started
    with ThreadPoolExecutor(max_workers=concurrency) as ex:
        while any(queues.values()):
            with cond:
                while in_flight["total"] >= concurrency and time.time() < issue_end:
                    cond.wait(timeout=2.0)
                now = time.time()
                if now >= issue_end:
                    break
                nxt = take_next()
                if nxt is None:
                    cond.wait(timeout=2.0)          # a band slot frees when a band sample ends
                    continue
                i, kind = nxt
                t, k, p = plan[i]
                if k >= 1 and differs_everywhere(first_grids.get(t), b0.get(t)):
                    skipped.append({"task_id": t, "sample": k, "pass": p, "plan_index": i, "s": round(now - t0, 1)})
                    log_event(i, t, k, p, "skipped", reason="sample 0 already gave a valid grid that differs "
                                                            "from attempt_1 on every output")
                    print(f"[{now - t0:7.0f}s] plan #{i} {t} s{k} (pass {p}) skipped: sample 0 already "
                          f"differs from attempt_1 on every output", flush=True)
                    continue
                cap = caps.answer_cap(now, band["max_tokens"] if kind == "band" else normal_cap)
                rule = None
                if kind == "band":
                    ok, rule = band_rule(cap, band, caps.min_answer_tokens)
                    if not ok:
                        # the band closes: later band tasks are larger and the time left only shrinks
                        left = [i] + list(queues["band"])
                        queues["band"].clear()
                        for j in left:
                            band_skipped.append(j)
                            log_event(j, *plan[j], "skipped", reason="band issue rule", band_rule=rule if j == i else None)
                        print(f"[{now - t0:7.0f}s] plan #{i} band {t} s{k} (pass {p}) skipped: band issue rule: "
                              f"{rule['why']}; {len(left)} band tasks skipped, second samples take the slots",
                              flush=True)
                        continue
                elif cap["kind"] == "deadline" and cap["max_tokens"] < caps.min_answer_tokens:
                    # no time left for an answer of min_answer_tokens and its rescue; later entries have less
                    stopped.update(plan_index=i, s=round(now - t0, 1), cap=cap,
                                   reason=f"deadline cap {cap['max_tokens']} < {caps.min_answer_tokens} tokens")
                    print(f"[{now - t0:7.0f}s] plan #{i} {t} s{k} (pass {p}) not issued: {stopped['reason']} "
                          f"({cap['why']}); issuing stops", flush=True)
                    queues[kind].appendleft(i)
                    break
                in_flight["total"] += 1
                if kind == "band":
                    in_flight["band"] += 1
                flight = dict(in_flight)
                issued.append(i)
                log_event(i, t, k, p, "issued", cap=cap["max_tokens"], cap_kind=cap["kind"],
                          rate_tok_s=cap["rate_tok_s"], rate_source=cap["rate_source"], in_flight=flight,
                          band_rule=rule)
                ex.submit(work, i, t, k, p, cap, now, rule, flight)
        with cond:
            left = sorted(j for q in queues.values() for j in q)
            for j in left:
                not_started.append(plan[j])
                log_event(j, *plan[j], "not started", reason=stopped.get("reason", "issue deadline"))
            for q in queues.values():
                q.clear()
    wall = time.time() - ta
    lines = HC.read_sample_lines(jsonl)
    final = HC.merge_files(cfg["b0_submission"], jsonl, cfg["submission"], ch,
                           report_path=os.path.join(out, "_merge.json"))
    by_pass = {}
    for p in passes:
        by_pass[str(p)] = {"kind": kind_p(p), "planned": sum(1 for j in plan if j[2] == p),
                           "finished": sum(1 for ln in lines if ln.get("pass") == p),
                           "with_grids": sum(1 for ln in lines if ln.get("pass") == p and ln.get("grids")),
                           "skipped": sum(1 for x in skipped if x["pass"] == p)
                           + sum(1 for j in band_skipped if plan[j][2] == p),
                           "not_started": sum(1 for x in not_started if x[2] == p)}
    summary = {
        "schema": HC.SCHEMA, "n_route": len(route), "n_plan": len(plan), "plan_head": plan[:12],
        "started": len(issued), "finished": len(lines),
        "last_started_plan_index": max(issued, default=None),
        "last_started_position": max((ln["route_position"] for ln in lines if ln.get("route_position") is not None),
                                     default=None),
        "not_started": len(not_started), "skipped": skipped, "band_skipped": [plan[j] for j in band_skipped],
        "by_pass": by_pass,
        "status": {k: sum(1 for ln in lines if ln["status"] == k) for k in STATUSES},
        "answered_tasks": len({ln["task_id"] for ln in lines if ln.get("grids")}),
        "generated_tokens": sum(ln["gen_tokens"] for ln in lines), "agent_wall_s": round(wall, 1),
        "gen_tok_per_s": round(sum(ln["gen_tokens"] for ln in lines) / max(wall, 1e-9), 1),
        "agent_start_s": round(ta - t0, 1), "issue_end_s": round(issue_end - t0, 1),
        "collect_end_s": round(collect_end - t0, 1), "end_s": round(time.time() - t0, 1), "merge": final,
        "settings": {k: cfg.get(k) for k in ("band_fraction", "band_samples", "beyond_band", "concurrency",
                                             "max_num_seqs", "max_model_len", "max_tokens", "rescue_tokens",
                                             "thinking_token_budget", "temperature", "top_p", "top_k",
                                             "template_kwargs", "server", "server_option_set", "engine")},
    }
    # version 5: the pre-set rule's counts (at most 2 samples with 0 tokens at the collection deadline; samples far
    # from the deadline at the normal cap, untouched)
    not_issued = ({"count": len(not_started), "from_plan_index": stopped["plan_index"], "at_s": stopped["s"],
                   "reason": stopped["reason"]} if stopped else None)
    dsum = deadline_summary(lines, collect_end - t0, not_issued=not_issued)
    dsum.update(params=caps.p, rescue_gate_s=round(collect_end - RESCUE_GATE_S - t0, 1),
                rate_at_end=caps._rate(time.time())[1], rate_measured_requests=caps.meter.n_measured)
    summary["deadline_caps"] = dsum
    summary["settings"]["deadline_caps"] = caps.p
    summary["settings"]["band_98k"] = band
    # version 6: the server's KV use and preemptions, B0's DFS record, the summary line and the four checks
    server = {"metrics": monitor.stop(), "log": parse_server_log(cfg.get("server_log"))}
    dfs = b0_dfs_summary(cfg.get("b0_dfs_log"), cfg.get("v6_reference"))
    v6 = v6_summary(lines, events, collect_end - t0, dfs, parser, band, server, not_issued=not_issued)
    summary["v6"] = v6
    HC.write_json_atomic(os.path.join(out, "_p2_run.json"), summary, indent=1)
    print(dsum["line"], flush=True)
    print(f"server: KV cache peak {server['metrics'].get('kv_cache_usage_pct_max')}% at "
          f"{server['metrics'].get('kv_peak_s')} s (metrics, {server['metrics'].get('polls')} polls, "
          f"{server['metrics'].get('errors')} failed), preemptions {server['metrics'].get('preemptions')}, running max "
          f"{server['metrics'].get('running_max')}, waiting max {server['metrics'].get('waiting_max')}; log: "
          f"KV peak {server['log'].get('kv_cache_usage_pct_max')}%, running max {server['log'].get('running_max')}, "
          f"waiting max {server['log'].get('waiting_max')}, preemption lines {server['log'].get('preemption_lines')}, "
          f"capacity {server['log'].get('kv_capacity')}", flush=True)
    print(f"B0 DFS per task (s, batches decoded, built, outputs): {dfs['per_task']}", flush=True)
    print(v6["line"], flush=True)
    print(v6["checks_line"], flush=True)
    print(f"phase 2 done: {summary['started']} of {len(plan)} planned samples started (last plan index "
          f"{summary['last_started_plan_index']}), {len(skipped)} skipped, {len(band_skipped)} band skipped by the "
          f"rule, {summary['not_started']} not started; by pass {by_pass}; {summary['answered_tasks']} tasks with "
          f"grids, {summary['generated_tokens']} tokens in {wall:.0f} s ({summary['gen_tok_per_s']} tok/s); merge "
          f"{final}", flush=True)
    return summary


# ----------------------------------------------------------------------------

def run(cfg):
    t0 = float(cfg["run_t0"])
    out = cfg["out_dir"]
    os.makedirs(out, exist_ok=True)
    st = Stages(out, t0)
    kind = cfg.get("server", "vllm")
    base = f"http://127.0.0.1:{cfg.get('port', 1234)}/v1"
    proc, sm = None, None
    st.add("start", "ok", f"server={kind}", python=sys.version.split()[0], gpu_memory_mib=gpu_memory(),
           route=len(cfg.get("route", [])), plan=len(cfg.get("plan") or []))
    try:
        if kind == "vllm":
            if not wait_install(cfg, st):
                st.add("agent", "skipped", "no serving install")
                return
            import submit_main
            check_torch(st, submit_main.SITE)      # recorded; the server ladder runs either way
            proc, sm = start_vllm(cfg, st, base)
            cfg.setdefault("server_log", str(submit_main.LOG))   # version 6: KV use and preemptions read at the end
        else:
            proc = start_stub(cfg)
            cfg["server_option_set"] = {"index": None, "args": "stub"}
            s = smoke(base) if proc else {}
            st.add("server", "ok" if proc and "4" in (s.get("content") or "") else "failed", "stub server", smoke=s)
        if proc is None:
            st.add("agent", "skipped", "no server")
            return
        with open(os.path.join(out, "_server.pid"), "w") as f:
            f.write(str(proc.pid))
        if time.time() >= float(cfg["issue_end_epoch"]):
            st.add("agent", "skipped", "issue deadline passed before the server was up")
            return
        summary = run_routed(dict(cfg, base_url=base, model="qwen"), st)
        st.add("agent", "ok", f"{summary['started']} samples started, {len(summary['skipped'])} skipped, "
                              f"{summary['answered_tasks']} tasks answered, "
                              f"{summary['merge'].get('replaced')} attempt_2 replaced",
               agent_wall_s=summary["agent_wall_s"], gen_tok_per_s=summary["gen_tok_per_s"])
    except BaseException:
        st.add("error", "caught", traceback.format_exc()[-3000:])
    finally:
        if proc is not None:
            if kind == "vllm" and sm is not None:
                sm.kill(proc)
            else:
                proc.kill()
            st.add("server", "stopped", gpu_memory_mib=gpu_memory())
        if kind == "vllm":
            try:
                import submit_main
                shutil.rmtree(submit_main.SITE, ignore_errors=True)
            except Exception:
                pass
        if os.path.isfile(os.path.join(out, "_server.pid")):
            os.remove(os.path.join(out, "_server.pid"))
        st.add("end", "ok")


if __name__ == "__main__":
    try:
        mode, path = sys.argv[1], sys.argv[2]
        with open(path) as f:
            config = json.load(f)
        if mode == "install":
            install(config)
        else:
            run(config)
    except BaseException:
        print(traceback.format_exc(), flush=True)
    sys.exit(0)


Writing /kaggle/working/hybrid/hybrid_p2.py


In [14]:
# The verify-and-refine harness (arc_harness; MIT-0, its Poetiq-derived parts MIT: arc_harness/NOTICES.md) and
# its phase-2 driver hybrid_vr.py, written into the hybrid directory; VR_CONFIG is phase 2's harness settings.
import hashlib
import os
import sys

VR_FILES = {
    'arc_harness/__init__.py': '"""arc_harness: a verify-and-refine harness around an open-weights reasoning model for ARC-AGI-2.\n\nLayout\n  data.py        task loading (Kaggle json, GitHub dirs, ARC-AGI-1, ConceptARC), holdout split\n  grids.py       grid text formatting and grid parsing from model output\n  sandbox.py     run the model\'s transform() in a subprocess; grade on demonstrations (Poetiq feedback)\n  validator.py   cheap hard checks on a candidate output (shape, palette) from the demonstrations\n  prompts.py     prompts for direct, direct+program, repair and Poetiq\n  llm.py         LLM backends: vLLM (OpenAI-compatible), scripted, oracle simulator\n  budget.py      token/time budget accounting and per-stage caps\n  solvers.py     direct, poetiq, vr (verify-and-refine) solvers; all emit Sample records\n  selection.py   rank candidates into attempt_1 / attempt_2, merge with an external pool\n  submission.py  always-valid submission.json writer\n  score.py       local scorer\n  loss.py        attribute each missed output to a loss category\n  run.py         CLI entry point\n  ablate.py      replay-based component ablation\n"""\n\n__version__ = "0.1.0"\n',
    'arc_harness/ablate.py': '"""Replay-based ablation: what each component adds, from a run\'s saved samples (no model calls).\n\n  python -m arc_harness.ablate runs/x [--external pool.json] [--tasks dev]\n\nStage ablations keep only the samples of some stages and re-run selection:\n  s1            first samples (plus their rescues): "answer + program, verification as the selector"\n  s1+repair     adds the repair round\n  s1+repair+s2  adds the second samples (the full run)\nSelection ablations re-rank the full sample set:\n  grid_only     votes over written grids, no programs, no validator\n  vr-no-validator\n  vr            the harness policy\nMerge ablations, when an external pool is given: R0 | verified_first | harness_first.\nAlso reports tokens by stage, the verified-but-wrong rate, and right outputs per million tokens for each arm.\n"""\nfrom __future__ import annotations\n\nimport argparse\nimport json\nimport os\nfrom collections import Counter\nfrom typing import Dict, List\n\nfrom .data import load_dir\nfrom .grids import hardcoded_risk\nfrom .run import ROOT, select_tasks\nfrom .score import score_submission\nfrom .selection import two_attempts\nfrom .solvers import load_samples\n\nSTAGE_SETS = {\n    "s1": {"s1", "rescue"},\n    "s1+repair": {"s1", "rescue", "repair", "rescue_repair"},\n    "s1+repair+s2": {"s1", "rescue", "repair", "rescue_repair", "s2", "rescue_s2"},\n}\n\n\ndef replay(tasks: Dict[str, dict], samples: Dict[str, List[dict]], policy: str, stages=None,\n           external=None, merge="R0", use_validator=True) -> dict:\n    sub = {}\n    tokens = 0\n    for tid, task in tasks.items():\n        ss = samples.get(tid, [])\n        if stages is not None:\n            keep = set(s["sample_id"] for s in ss if s["stage"] in stages)\n            # a rescue\'s token cost belongs with its cut parent; keep parents of kept rescues for accounting\n            ss_keep = [s for s in ss if s["sample_id"] in keep]\n            parents = {s["parent"] for s in ss_keep if s.get("parent")}\n            tokens += sum(s["gen_tokens"] for s in ss if s["sample_id"] in keep or s["sample_id"] in parents)\n        else:\n            ss_keep = ss\n            tokens += sum(s["gen_tokens"] for s in ss)\n        ext = (external or {}).get(tid)\n        sub[tid] = two_attempts(task, ss_keep, policy, ext, merge, use_validator)\n    sc = score_submission(sub, tasks)\n    return {"right": sc["right"], "n": sc["n_outputs"], "score": sc["score"], "tokens": tokens,\n            "per_M": 1e6 * sc["right"] / max(1, tokens), "attempt1_right": sc["attempt1_right"]}\n\n\ndef verified_stats(tasks, samples) -> dict:\n    """How precise the verification bit is: verified programs right / verified programs."""\n    n_ver = n_ver_right = n_prog = n_prog_right = n_hard = 0\n    for tid, task in tasks.items():\n        for s in samples.get(tid, []):\n            v = s.get("verify") or {}\n            if not v.get("ran"):\n                continue\n            for j, te in enumerate(task["test"]):\n                if "output" not in te:\n                    continue\n                po = (v.get("program_outputs") or [None] * (j + 1))[j] if j < len(v.get("program_outputs") or []) else None\n                if po is None:\n                    continue\n                n_prog += 1\n                n_prog_right += po == te["output"]\n                if v.get("passed"):\n                    n_ver += 1\n                    n_ver_right += po == te["output"]\n                    n_hard += hardcoded_risk(s.get("program") or "")\n    return {"program_outputs": n_prog, "program_outputs_right": n_prog_right, "verified": n_ver,\n            "verified_right": n_ver_right, "verified_with_literals": n_hard,\n            "verified_precision": n_ver_right / n_ver if n_ver else None,\n            "verified_but_wrong_rate": (n_ver - n_ver_right) / n_ver if n_ver else None}\n\n\ndef tokens_by_stage(samples) -> dict:\n    c = Counter()\n    k = Counter()\n    for ss in samples.values():\n        for s in ss:\n            c[s["stage"]] += s["gen_tokens"]\n            k[s["stage"]] += 1\n    return {st: {"calls": k[st], "tokens": c[st]} for st in c}\n\n\ndef main(argv=None):\n    ap = argparse.ArgumentParser()\n    ap.add_argument("run_dir")\n    ap.add_argument("--tasks", default="dev")\n    ap.add_argument("--data-root", default=os.path.join(ROOT, "data"))\n    ap.add_argument("--external", default=None)\n    args = ap.parse_args(argv)\n    tasks = select_tasks(args.tasks, args.data_root)\n    samples = load_samples(os.path.join(args.run_dir, "samples.jsonl"))\n    tasks = {t: tasks[t] for t in tasks if t in samples} or tasks\n    external = None\n    if args.external:\n        with open(args.external) as f:\n            external = json.load(f)\n    rows = []\n    for name, stages in STAGE_SETS.items():\n        r = replay(tasks, samples, "vr", stages)\n        rows.append(("stages", name) + (r["right"], r["n"], r["tokens"], r["per_M"]))\n    for name, pol, val in (("grid_only", "grid_only", False), ("vr-no-validator", "vr", False), ("vr", "vr", True),\n                           ("poetiq-rank", "poetiq", False)):\n        r = replay(tasks, samples, pol, None, use_validator=val)\n        rows.append(("selection", name) + (r["right"], r["n"], r["tokens"], r["per_M"]))\n    if external:\n        for m in ("R0", "verified_first", "harness_first", "verified_then_margin"):\n            r = replay(tasks, samples, "vr", None, external, m)\n            rows.append(("merge", m) + (r["right"], r["n"], r["tokens"], r["per_M"]))\n        def _att(e):\n            return e["attempts"] if isinstance(e, dict) else e\n        ext_only = {tid: [{"attempt_1": e[0], "attempt_2": e[1] if len(e) > 1 else [[0]]}\n                          for e in _att(external.get(tid, []))] for tid in tasks}\n        sc = score_submission(ext_only, tasks)\n        rows.append(("merge", "external_only", sc["right"], sc["n_outputs"], 0, 0.0))\n    out = {"rows": [dict(zip(("kind", "arm", "right", "n", "tokens", "right_per_M"), r)) for r in rows],\n           "verified": verified_stats(tasks, samples), "tokens_by_stage": tokens_by_stage(samples)}\n    with open(os.path.join(args.run_dir, "ablation.json"), "w") as f:\n        json.dump(out, f, indent=1)\n    print(f"{\'kind\':10s} {\'arm\':18s} {\'right\':>6s} {\'n\':>4s} {\'tokens\':>10s} {\'right/M\':>8s}")\n    for r in rows:\n        print(f"{r[0]:10s} {r[1]:18s} {r[2]:6d} {r[3]:4d} {r[4]:10d} {r[5]:8.1f}")\n    print("verification:", json.dumps(out["verified"]))\n    print("tokens by stage:", json.dumps(out["tokens_by_stage"]))\n    return 0\n\n\nif __name__ == "__main__":\n    main()\n',
    'arc_harness/augment.py': '"""Task augmentations for independent second samples: a dihedral transform plus a colour permutation (0 fixed).\n\nA sample on an augmented view is solved in the transformed frame; its grids and program outputs are mapped back\nwith the inverse, so every candidate lives in the original frame and votes across views are over the same grids.\nThe program is verified on the augmented demonstrations (the frame it was written for). The same trick is what\nARChitects/NVARC use for augmentation voting; here it buys diversity for the few second samples the budget allows.\n"""\nfrom __future__ import annotations\n\nimport random\nfrom dataclasses import dataclass\nfrom typing import List, Optional\n\nfrom .grids import Grid\n\n\ndef _rot90(g: Grid) -> Grid:          # counter-clockwise\n    return [list(r) for r in zip(*g)][::-1]\n\n\ndef _rot(g: Grid, k: int) -> Grid:\n    for _ in range(k % 4):\n        g = _rot90(g)\n    return g\n\n\ndef _flip(g: Grid) -> Grid:           # left-right\n    return [r[::-1] for r in g]\n\n\n@dataclass(frozen=True)\nclass Aug:\n    rot: int = 0                      # quarter turns, counter-clockwise, applied after the flip\n    flip: bool = False\n    perm: tuple = tuple(range(10))    # colour map, perm[0] == 0\n\n    @property\n    def inv_perm(self) -> tuple:\n        inv = [0] * 10\n        for a, b in enumerate(self.perm):\n            inv[b] = a\n        return tuple(inv)\n\n    def apply(self, g: Grid) -> Grid:\n        g = [[self.perm[c] for c in r] for r in g]\n        if self.flip:\n            g = _flip(g)\n        return _rot(g, self.rot)\n\n    def invert(self, g: Optional[Grid]) -> Optional[Grid]:\n        if g is None:\n            return None\n        g = _rot(g, -self.rot)\n        if self.flip:\n            g = _flip(g)\n        inv = self.inv_perm\n        return [[inv[c] for c in r] for r in g]\n\n    def apply_task(self, task: dict) -> dict:\n        return {"id": task["id"], "aug": self.key(),\n                "train": [{"input": self.apply(p["input"]), "output": self.apply(p["output"])} for p in task["train"]],\n                "test": [{"input": self.apply(t["input"])} for t in task["test"]]}\n\n    def key(self) -> str:\n        return f"r{self.rot}{\'f\' if self.flip else \'\'}p{\'\'.join(map(str, self.perm))}"\n\n    @property\n    def is_identity(self) -> bool:\n        return self.rot == 0 and not self.flip and self.perm == tuple(range(10))\n\n\nIDENTITY = Aug()\n\n\ndef random_aug(seed: int, geometry: bool = True, colours: bool = True) -> Aug:\n    rng = random.Random(seed)\n    rot, flip = (rng.randrange(4), rng.random() < 0.5) if geometry else (0, False)\n    perm = list(range(10))\n    if colours:\n        tail = perm[1:]\n        rng.shuffle(tail)\n        perm = [0] + tail\n    a = Aug(rot, flip, tuple(perm))\n    return a if not a.is_identity else Aug(1, False, tuple(perm))\n\n\ndef invert_sample(aug: Aug, sample: dict) -> dict:\n    """Map a sample\'s grids and program outputs back to the original frame (in place)."""\n    if aug.is_identity:\n        return sample\n    sample["grids"] = [aug.invert(g) for g in sample["grids"]]\n    v = sample.get("verify") or {}\n    if v.get("program_outputs"):\n        v["program_outputs"] = [aug.invert(g) for g in v["program_outputs"]]\n    sample["aug"] = aug.key()\n    return sample\n',
    'arc_harness/budget.py': '"""Token and wall-clock budget.\n\nThe budget unit is generated tokens (reasoning + answer), summed over every model call. A run is given a mean\nbudget per task B; the pool is B x n_tasks. Per-stage caps are derived from B so that one knob gives the 15k /\n30k / 60k / 120k curve:\n\n  think1   first-sample thinking cap     clamp(0.68 B,  8k, 90k)   (41k at B=60k, the measured sweet spot)\n  think2   repair-round thinking cap     clamp(0.30 B,  6k, 24k)   (repair is skipped below B=20k)\n  answer   answer part after thinking    6k  (grids for up to 2 tests plus a program)\n  rescue   thinking-off rescue cap       8k\n\nA deadline guard clips every grant so that the request can finish before the deadline minus a reserve at the\nmeasured per-stream rate (aggregate rate / requests in flight).\n"""\nfrom __future__ import annotations\n\nimport threading\nimport time\nfrom dataclasses import dataclass, field\nfrom typing import Optional\n\n\ndef clamp(x, lo, hi):\n    return max(lo, min(hi, int(x)))\n\n\n@dataclass\nclass Caps:\n    think1: int\n    think2: int\n    answer: int = 6000\n    rescue: int = 8000\n    repair_enabled: bool = True\n\n    @staticmethod\n    def from_mean(b: int, overrides: Optional[dict] = None) -> "Caps":\n        c = Caps(think1=clamp(0.68 * b, 8000, 90000), think2=clamp(0.30 * b, 6000, 24000),\n                 repair_enabled=b >= 20000)\n        for k, v in (overrides or {}).items():\n            setattr(c, k, v)\n        return c\n\n\n@dataclass\nclass Budget:\n    mean_tokens_per_task: int\n    n_tasks: int\n    deadline: Optional[float] = None          # epoch seconds; None = no wall-clock limit\n    reserve_s: float = 900.0                  # kept free before the deadline for writing the submission\n    tokens_per_s: float = 334.0               # aggregate, initial estimate; updated from observations\n    max_in_flight: int = 20\n    caps: Caps = field(default_factory=lambda: Caps(41000, 18000))\n    spent: int = 0\n    prefill: int = 0\n    calls: int = 0\n    _obs_tokens: int = 0\n    _obs_seconds: float = 0.0\n    _lock: threading.Lock = field(default_factory=threading.Lock, repr=False)\n    in_flight: int = 0\n    committed: int = 0                        # tokens granted to requests still in flight\n\n    def __post_init__(self):\n        self.pool = int(self.mean_tokens_per_task) * int(self.n_tasks)\n        self.start = time.time()\n\n    def remaining_tokens(self) -> int:\n        return max(0, self.pool - self.spent)\n\n    def remaining_seconds(self) -> Optional[float]:\n        if self.deadline is None:\n            return None\n        return self.deadline - time.time() - self.reserve_s\n\n    def per_stream_rate(self) -> float:\n        """Tokens/s one request can expect: the aggregate shared by a full batch. Conservative in the tail of a\n        run (fewer requests go faster), right at saturation, never optimistic at the start."""\n        return self.tokens_per_s / max(1, self.max_in_flight)\n\n    def grant(self, want: int, min_useful: int = 2000, reserve: int = 0) -> int:\n        """Tokens this request may generate now: the cap, clipped by the pool and by the deadline.\n        `reserve` is kept back (e.g. for the rescue a cut sample will need), in tokens and in time.\n        Returns 0 when fewer than min_useful tokens can be afforded."""\n        with self._lock:\n            g = min(int(want), self.remaining_tokens() - self.committed - int(reserve))\n            rs = self.remaining_seconds()\n            if rs is not None:\n                g = min(g, int(rs * self.per_stream_rate()) - int(reserve))\n            return g if g >= min_useful else 0\n\n    def begin(self, granted: int = 0):\n        with self._lock:\n            self.in_flight += 1\n            self.committed += int(granted)\n\n    def charge(self, tokens_out: int, tokens_in: int = 0, seconds: float = 0.0, granted: int = 0):\n        with self._lock:\n            self.spent += int(tokens_out)\n            self.prefill += int(tokens_in)\n            self.calls += 1\n            self.in_flight = max(0, self.in_flight - 1)\n            self.committed = max(0, self.committed - int(granted))\n            elapsed = time.time() - self.start\n            if seconds > 1.0 and elapsed > 600 and self.spent > 200_000:   # real servers only, once saturated\n                self.tokens_per_s = self.spent / elapsed                 # the observed aggregate rate\n\n    def timeout_for(self, granted: int, floor_s: float = 600.0) -> float:\n        """A read timeout for one request: three times the expected stream time at a full batch, at least floor_s."""\n        return max(floor_s, 3.0 * granted / max(1.0, self.per_stream_rate()))\n\n    def summary(self) -> dict:\n        return {"pool": self.pool, "spent": self.spent, "prefill": self.prefill, "calls": self.calls,\n                "mean_per_task": self.spent / max(1, self.n_tasks), "elapsed_s": time.time() - self.start,\n                "tokens_per_s_est": self.tokens_per_s}\n',
    'arc_harness/data.py': '"""Task loading and splits.\n\nA task is {"id": str, "train": [{"input": grid, "output": grid}, ...], "test": [{"input": grid, "output": grid?}, ...]}.\nGrids are lists of lists of ints 0-9.\n"""\nfrom __future__ import annotations\n\nimport hashlib\nimport json\nimport os\nfrom typing import Dict, Iterable, List, Optional\n\nGrid = List[List[int]]\n\nSPLIT_SALT = "arc2-split-v1"\nHOLDOUT_SIZE = 40\n\n\ndef load_dir(path: str) -> Dict[str, dict]:\n    """Load every *.json task file in a directory (GitHub ARC-AGI / ARC-AGI-2 layout)."""\n    tasks = {}\n    for name in sorted(os.listdir(path)):\n        if not name.endswith(".json"):\n            continue\n        tid = name[:-5]\n        with open(os.path.join(path, name)) as f:\n            t = json.load(f)\n        tasks[tid] = {"id": tid, "train": t["train"], "test": t["test"]}\n    return tasks\n\n\ndef load_kaggle(challenges_path: str, solutions_path: Optional[str] = None) -> Dict[str, dict]:\n    """Load the Kaggle challenge file (and optionally the solutions file) into tasks."""\n    with open(challenges_path) as f:\n        ch = json.load(f)\n    sol = {}\n    if solutions_path and os.path.exists(solutions_path):\n        with open(solutions_path) as f:\n            sol = json.load(f)\n    tasks = {}\n    for tid, t in ch.items():\n        tests = [{"input": x["input"]} for x in t["test"]]\n        if tid in sol:\n            for j, out in enumerate(sol[tid]):\n                tests[j]["output"] = out\n        tasks[tid] = {"id": tid, "train": t["train"], "test": tests}\n    return tasks\n\n\ndef load_conceptarc(root: str) -> Dict[str, dict]:\n    """ConceptARC corpus: corpus/<Concept>/<Concept><n>.json, same task format."""\n    tasks = {}\n    corpus = os.path.join(root, "corpus")\n    if not os.path.isdir(corpus):\n        return tasks\n    for concept in sorted(os.listdir(corpus)):\n        d = os.path.join(corpus, concept)\n        if not os.path.isdir(d):\n            continue\n        for name in sorted(os.listdir(d)):\n            if not name.endswith(".json"):\n                continue\n            with open(os.path.join(d, name)) as f:\n                t = json.load(f)\n            tid = f"conceptarc/{name[:-5]}"\n            tasks[tid] = {"id": tid, "train": t["train"], "test": t["test"]}\n    return tasks\n\n\ndef public_sets(data_root: str) -> Dict[str, Dict[str, dict]]:\n    """All public task sets under data/: arc2_train, arc2_eval, arc1_train, arc1_eval, conceptarc."""\n    out = {}\n    pairs = {\n        "arc2_train": "ARC-AGI-2/data/training",\n        "arc2_eval": "ARC-AGI-2/data/evaluation",\n        "arc1_train": "ARC-AGI/data/training",\n        "arc1_eval": "ARC-AGI/data/evaluation",\n    }\n    for k, rel in pairs.items():\n        p = os.path.join(data_root, rel)\n        if os.path.isdir(p):\n            out[k] = load_dir(p)\n    ca = os.path.join(data_root, "ConceptARC")\n    if os.path.isdir(ca):\n        out["conceptarc"] = load_conceptarc(ca)\n    return out\n\n\ndef split_key(task_id: str, salt: str = SPLIT_SALT) -> str:\n    return hashlib.sha256((task_id + salt).encode()).hexdigest()\n\n\ndef holdout_split(task_ids: Iterable[str], n_holdout: int = HOLDOUT_SIZE, salt: str = SPLIT_SALT):\n    """Sort ids by sha256(task_id + salt); the last n are the untouched holdout.\n\n    Returns (dev_ids, holdout_ids), each in the sorted order.\n    """\n    ordered = sorted(task_ids, key=lambda t: split_key(t, salt))\n    return ordered[:-n_holdout], ordered[-n_holdout:]\n\n\ndef task_size(task: dict) -> int:\n    """Cells in the task text: the proxy for prompt length used for shortest-first ordering."""\n    n = 0\n    for p in task["train"]:\n        n += _cells(p["input"]) + _cells(p["output"])\n    for t in task["test"]:\n        n += _cells(t["input"])\n    return n\n\n\ndef _cells(g: Grid) -> int:\n    return sum(len(r) for r in g)\n\n\ndef n_test_outputs(tasks: Dict[str, dict]) -> int:\n    return sum(len(t["test"]) for t in tasks.values())\n',
    'arc_harness/grids.py': '"""Grid text formatting and parsing of grids from model output."""\nfrom __future__ import annotations\n\nimport json\nimport re\nfrom typing import List, Optional\n\nGrid = List[List[int]]\n\n\ndef grid_text(g: Grid) -> str:\n    return "\\n".join(" ".join(str(c) for c in row) for row in g)\n\n\ndef grid_shape(g) -> Optional[tuple]:\n    if not isinstance(g, list) or not g or not all(isinstance(r, list) for r in g):\n        return None\n    w = len(g[0])\n    if w == 0 or any(len(r) != w for r in g):\n        return None\n    return (len(g), w)\n\n\ndef is_grid(g) -> bool:\n    s = grid_shape(g)\n    if s is None or s[0] > 30 or s[1] > 30:\n        return False\n    return all(isinstance(c, int) and not isinstance(c, bool) and 0 <= c <= 9 for r in g for c in r)\n\n\ndef grids_equal(a, b) -> bool:\n    return is_grid(a) and is_grid(b) and a == b\n\n\ndef cell_accuracy(pred: Grid, truth: Grid) -> float:\n    """Share of right cells; 0 if the shapes differ."""\n    if grid_shape(pred) != grid_shape(truth):\n        return 0.0\n    n = sum(len(r) for r in truth)\n    hit = sum(1 for pr, tr in zip(pred, truth) for p, t in zip(pr, tr) if p == t)\n    return hit / n if n else 0.0\n\n\n_ROW_RE = re.compile(r"^\\s*[0-9](?:[ ,]+[0-9])*\\s*$")\n\n\ndef _parse_block(text: str) -> Optional[Grid]:\n    """Parse a block of digit rows (\'0 1 2\' per line, or \'012\', or JSON-ish) into a grid."""\n    text = text.strip()\n    if not text:\n        return None\n    # JSON list of lists\n    if text.startswith("["):\n        try:\n            g = json.loads(text)\n            return g if is_grid(g) else None\n        except json.JSONDecodeError:\n            pass\n    rows = []\n    for line in text.splitlines():\n        line = line.strip().strip("|").strip()\n        if not line:\n            continue\n        if _ROW_RE.match(line):\n            rows.append([int(c) for c in re.findall(r"[0-9]", line)])\n        elif re.fullmatch(r"[0-9]{1,30}", line):\n            rows.append([int(c) for c in line])\n        else:\n            return None\n    return rows if is_grid(rows) else None\n\n\n# Answer layouts the prompts ask for:  <output_1> ... </output_1>  (one tag per test input)\n_TAG_RE = re.compile(r"<output_(\\d+)>\\s*(.*?)\\s*</output_\\1>", re.S)\n_FENCE_RE = re.compile(r"```(?:[a-zA-Z]*)\\n(.*?)```", re.S)\n\n\ndef parse_outputs(text: str, n_test: int) -> List[Optional[Grid]]:\n    """Parse the test output grids from a reply. Each test output is parsed on its own.\n\n    Order of preference: <output_k> tags, then the last n_test digit blocks (fenced or bare).\n    """\n    out: List[Optional[Grid]] = [None] * n_test\n    tags = {int(k): _parse_block(v) for k, v in _TAG_RE.findall(text)}\n    if tags:\n        for k in range(n_test):\n            out[k] = tags.get(k + 1)\n        if any(o is not None for o in out):\n            return out\n    blocks = [b for b in _FENCE_RE.findall(text) if _parse_block(b) is not None]\n    if not blocks:\n        blocks = _bare_blocks(text)\n    grids = [_parse_block(b) for b in blocks]\n    grids = [g for g in grids if g is not None]\n    if grids:\n        tail = grids[-n_test:]\n        for k, g in enumerate(tail):\n            out[k] = g\n    return out\n\n\ndef _bare_blocks(text: str) -> List[str]:\n    """Maximal runs of lines that look like grid rows."""\n    blocks, cur = [], []\n    for line in text.splitlines():\n        s = line.strip()\n        if s and (_ROW_RE.match(s) or re.fullmatch(r"[0-9]{1,30}", s)):\n            cur.append(s)\n        else:\n            if cur:\n                blocks.append("\\n".join(cur))\n            cur = []\n    if cur:\n        blocks.append("\\n".join(cur))\n    return blocks\n\n\n_PY_RE = re.compile(r"```(?:python|py|python3)?[ \\t]*\\n(.*?)```", re.S | re.I)\n\n\ndef parse_program(text: str) -> Optional[str]:\n    """The last fenced block (```python, ```py or bare ```) that defines transform()."""\n    cands = [m for m in _PY_RE.findall(text) if "def transform" in m]\n    return cands[-1] if cands else None\n\n\n_LITERAL_RE = re.compile(r"\\[\\s*(?:-?\\d+\\s*,\\s*)+-?\\d+\\s*\\]")\n\n\ndef hardcoded_risk(program: str, min_ints: int = 40) -> bool:\n    """A program that carries long integer-list literals is probably a lookup table of the demonstrations: it\n    passes them without encoding the rule. Verification by such a program is not trusted as evidence."""\n    n = sum(len(re.findall(r"-?\\d+", m)) for m in _LITERAL_RE.findall(program))\n    return n >= min_ints\n',
    'arc_harness/llm.py': '"""LLM backends.\n\n  VLLMChat      an OpenAI-compatible vLLM server (the Kaggle path); thinking on/off via chat_template_kwargs\n  ScriptedLLM   a python callable, for tests\n  OracleLLM     a simulator driven by the solutions, for CPU end-to-end checks of the pipeline and of the\n                budget curve plumbing; it does NOT measure anything about a real model\n\nAll return a Completion. tokens_out counts reasoning + answer tokens; that is the budget unit.\n"""\nfrom __future__ import annotations\n\nimport hashlib\nimport json\nimport random\nimport re\nimport socket\nimport threading\nimport time\nimport urllib.error\nimport urllib.request\nfrom dataclasses import dataclass, field\nfrom typing import Callable, Dict, List, Optional\n\nfrom .augment import IDENTITY, Aug\nfrom .grids import grid_text\n\nTHINK_RE = re.compile(r"<think>(.*?)</think>", re.S)\n\n\n@dataclass\nclass Completion:\n    text: str\n    reasoning: str\n    tokens_out: int\n    tokens_in: int\n    finish: str                      # "stop" | "length" | "error"\n    cut_in_thinking: bool\n    latency_s: float = 0.0\n    error: Optional[str] = None\n    meta: dict = field(default_factory=dict)\n\n\nclass LLM:\n    name = "base"\n\n    def generate(self, messages: List[dict], max_tokens: int, thinking: bool = True,\n                 temperature: float = 1.0, seed: Optional[int] = None, timeout_s: Optional[float] = None) -> Completion:\n        raise NotImplementedError\n\n\nclass VLLMChat(LLM):\n    """OpenAI-compatible chat completions against a local vLLM server.\n\n    Qwen3-family thinking is switched with chat_template_kwargs={"enable_thinking": ...}. If the server runs with a\n    reasoning parser, reasoning arrives in message.reasoning_content; otherwise it is split from <think> tags here.\n    The thinking budget is enforced by the caller as max_tokens = thinking_cap + answer_cap, and a reply that ends\n    with finish_reason=="length" and no answer content is reported as cut_in_thinking (the caller then rescues).\n    """\n    name = "vllm"\n\n    def __init__(self, base_url: str = "http://127.0.0.1:8000/v1", model: str = "qwen", timeout_s: float = 3600,\n                 top_p: float = 0.95, top_k: int = 20, extra: Optional[dict] = None, retries: int = 3,\n                 system: Optional[str] = None, template_kwargs: Optional[dict] = None):\n        self.base_url = base_url.rstrip("/")\n        self.model = model\n        self.timeout_s = timeout_s\n        self.top_p = top_p\n        self.top_k = top_k\n        self.extra = extra or {}              # extra body fields, e.g. {"reasoning_effort": "low"}\n        self.template_kwargs = template_kwargs or {}   # extra chat_template_kwargs (model-specific effort knobs)\n        self.system = system                  # optional system prompt\n        self.retries = retries\n\n    def generate(self, messages, max_tokens, thinking=True, temperature=1.0, seed=None, timeout_s=None) -> Completion:\n        if self.system:\n            messages = [{"role": "system", "content": self.system}] + list(messages)\n        body = {\n            "model": self.model, "messages": messages, "max_tokens": int(max_tokens),\n            "temperature": temperature, "top_p": self.top_p, "top_k": self.top_k,\n            "chat_template_kwargs": {"enable_thinking": bool(thinking), **self.template_kwargs},\n        }\n        if seed is not None:\n            body["seed"] = int(seed)\n        body.update(self.extra)\n        t0 = time.time()\n        last_err = None\n        timeout = timeout_s or self.timeout_s\n        for attempt in range(self.retries + 1):\n            try:\n                req = urllib.request.Request(self.base_url + "/chat/completions", data=json.dumps(body).encode(),\n                                             headers={"Content-Type": "application/json"})\n                with urllib.request.urlopen(req, timeout=timeout) as r:\n                    resp = json.load(r)\n                return self._parse(resp, time.time() - t0)\n            except urllib.error.HTTPError as e:\n                detail = ""\n                try:\n                    detail = e.read().decode()[:500]\n                except Exception:\n                    pass\n                last_err = f"HTTP {e.code}: {detail}"\n                if e.code == 400 and "context length" in detail and body["max_tokens"] > 2048:\n                    # prompt + max_tokens over the server\'s max_model_len (large tasks at the band\'s cap): retry at\n                    # 0.75x, as the public direct mode does\n                    body["max_tokens"] = int(body["max_tokens"] * 0.75)\n                    continue\n                if 400 <= e.code < 500:\n                    break                        # a bad request will not get better by retrying\n                time.sleep(min(30, 2 ** attempt))\n            except (TimeoutError, socket.timeout) as e:\n                last_err = f"timeout after {timeout:.0f}s"\n                break                            # the server is still generating; do not pile a second request on\n            except (urllib.error.URLError, OSError, json.JSONDecodeError) as e:\n                last_err = f"{type(e).__name__}: {e}"\n                time.sleep(min(30, 2 ** attempt))\n        return Completion(text="", reasoning="", tokens_out=0, tokens_in=0, finish="error",\n                          cut_in_thinking=False, latency_s=time.time() - t0, error=last_err)\n\n    @staticmethod\n    def _parse(resp: dict, latency: float) -> Completion:\n        ch = resp["choices"][0]\n        msg = ch.get("message", {})\n        content = msg.get("content") or ""\n        reasoning = msg.get("reasoning_content") or msg.get("reasoning") or ""\n        if not reasoning and "<think>" in content:\n            m = THINK_RE.search(content)\n            if m:\n                reasoning = m.group(1)\n                content = content[m.end():]\n            else:  # cut inside the think block\n                reasoning = content.split("<think>", 1)[1]\n                content = ""\n        usage = resp.get("usage", {})\n        finish = ch.get("finish_reason") or "stop"\n        cut = finish == "length" and not content.strip()\n        return Completion(text=content, reasoning=reasoning, tokens_out=int(usage.get("completion_tokens", 0)),\n                          tokens_in=int(usage.get("prompt_tokens", 0)), finish=finish, cut_in_thinking=cut,\n                          latency_s=latency, meta={"id": resp.get("id")})\n\n    def ping(self) -> bool:\n        try:\n            with urllib.request.urlopen(self.base_url + "/models", timeout=10) as r:\n                return r.status == 200\n        except Exception:\n            return False\n\n\nclass ScriptedLLM(LLM):\n    """fn(messages, thinking, max_tokens) -> str | (text, reasoning). Tokens are counted as len/4."""\n    name = "scripted"\n\n    def __init__(self, fn: Callable):\n        self.fn = fn\n        self.calls = 0\n\n    def generate(self, messages, max_tokens, thinking=True, temperature=1.0, seed=None, timeout_s=None) -> Completion:\n        self.calls += 1\n        out = self.fn(messages, thinking, max_tokens)\n        text, reasoning = (out, "") if isinstance(out, str) else out\n        n = (len(text) + len(reasoning)) // 4\n        return Completion(text=text, reasoning=reasoning, tokens_out=n, tokens_in=len(messages[-1]["content"]) // 4,\n                          finish="stop", cut_in_thinking=False)\n\n\nclass OracleLLM(LLM):\n    """A deterministic simulator of a reasoning model, keyed on the test input shown in the prompt.\n\n    Each task gets a hidden "tokens needed" T from its id. With a thinking cap C >= T the reply finishes on its own\n    and is right with probability p_own; with C < T the reply is cut (the caller rescues it, and the rescue is right\n    with a probability that grows with C/T). A repair turn (the prompt carries example feedback) succeeds with\n    probability p_repair. Wrong grids are near misses (a few cells) or wrong shapes. Programs are lookup tables so\n    that the sandbox path, verification and the verified-but-wrong case are all exercised.\n\n    This is for exercising the harness on CPU. Its numbers say nothing about any real model.\n    """\n    name = "oracle"\n\n    def __init__(self, tasks: Dict[str, dict], p_own: float = 0.8, p_rescue_max: float = 0.45,\n                 p_repair: float = 0.35, p_program: float = 0.75, p_verified_wrong: float = 0.08, seed: int = 0,\n                 t_min: int = 4000, t_max: int = 160000):\n        self.index = {}\n        for t in tasks.values():\n            self.index[grid_text(t["test"][0]["input"])] = t\n        self.p_own, self.p_rescue_max, self.p_repair = p_own, p_rescue_max, p_repair\n        self.p_program, self.p_verified_wrong = p_program, p_verified_wrong\n        self.seed = seed\n        self.t_min, self.t_max = t_min, t_max\n        self.lock = threading.Lock()\n        self.calls = 0\n\n    def tokens_needed(self, tid: str) -> int:\n        h = int(hashlib.sha256((tid + "oracle").encode()).hexdigest(), 16) % 10_000 / 10_000\n        return int(self.t_min + (h ** 1.6) * (self.t_max - self.t_min))\n\n    def _find(self, prompt: str):\n        """The task shown in the prompt and the view (Aug) it is shown in, or (None, None)."""\n        m = re.search(r"(?:Test input #1|Challenge #1)\\n(?:<Diagram>\\n)?(.*?)(?:\\n</Diagram>)?\\n(?:\\n|$)", prompt, re.S)\n        if not m:\n            return None, None\n        key = m.group(1).strip()\n        if key in self.index:\n            return self.index[key], IDENTITY\n        seen = [[int(c) for c in row.split()] for row in key.splitlines()]\n        for t in self.index.values():\n            for rot in range(4):\n                for flip in (False, True):\n                    geo = Aug(rot, flip)\n                    g = geo.apply(t["test"][0]["input"])\n                    if len(g) != len(seen) or len(g[0]) != len(seen[0]):\n                        continue\n                    perm, ok = {}, True\n                    for ra, rb in zip(g, seen):\n                        for a, b in zip(ra, rb):\n                            if perm.setdefault(a, b) != b:\n                                ok = False\n                                break\n                        if not ok:\n                            break\n                    if ok and perm.get(0, 0) == 0 and len(set(perm.values())) == len(perm):\n                        full = list(range(10))\n                        used = set(perm.values())\n                        free = [c for c in range(10) if c not in used]\n                        for c in range(10):\n                            if c in perm:\n                                full[c] = perm[c]\n                            elif c != 0 or 0 not in used:\n                                full[c] = free.pop(0) if c != 0 else 0\n                        return t, Aug(rot, flip, tuple(full))\n        return None, None\n\n    def generate(self, messages, max_tokens, thinking=True, temperature=1.0, seed=None, timeout_s=None) -> Completion:\n        with self.lock:\n            self.calls += 1\n            n = self.calls\n        prompt = messages[-1]["content"]\n        task, aug = self._find(prompt)\n        if task is None:\n            return Completion(text="I cannot parse this task.", reasoning="", tokens_out=20, tokens_in=len(prompt) // 4,\n                              finish="stop", cut_in_thinking=False)\n        rng = random.Random(f"{self.seed}:{task[\'id\']}:{n}:{seed}")\n        T = self.tokens_needed(task["id"])\n        is_rescue = "<your_reasoning_so_far>" in prompt\n        is_repair = "Results on the examples" in prompt or "EXISTING PARTIAL" in prompt\n        wants_program = "transform(grid" in prompt\n        answer_cap = 6000\n        think_cap = max_tokens - answer_cap if thinking else 0\n\n        if thinking and not is_repair and think_cap < T:\n            # cut inside thinking: no answer at all\n            return Completion(text="", reasoning="partial reasoning " * (think_cap // 8), tokens_out=max_tokens,\n                              tokens_in=len(prompt) // 4, finish="length", cut_in_thinking=True)\n        if is_repair:\n            p_right = self.p_repair if rng.random() < 0.9 else 0.0\n            spent = min(think_cap, T // 3) if thinking else 500\n        elif is_rescue:\n            frac = min(1.0, len(prompt) / 1.0 / max(1, T))  # crude: more of the tail, more progress\n            p_right = self.p_rescue_max * (0.5 + 0.5 * frac)\n            spent = 1500\n        else:\n            p_right = self.p_own\n            spent = min(think_cap, T) if thinking else 800\n        right = rng.random() < p_right\n        grids = []\n        for te in task["test"]:\n            truth = aug.apply(te["output"])          # the answer in the frame the prompt shows\n            grids.append(truth if right else self._wrong(truth, rng))\n        text = "\\n".join(f"<output_{k}>\\n{grid_text(g)}\\n</output_{k}>" for k, g in enumerate(grids, 1))\n        tokens = spent + sum(len(grid_text(g)) // 3 for g in grids)\n        if wants_program and rng.random() < self.p_program:\n            passes = rng.random() < (0.9 if right else 0.25)\n            prog_test = grids\n            if passes and right and rng.random() < self.p_verified_wrong:\n                prog_test = [self._wrong(g, rng) for g in grids]  # passes the examples, wrong on the test\n                text = "\\n".join(f"<output_{k}>\\n{grid_text(g)}\\n</output_{k}>" for k, g in enumerate(prog_test, 1))\n            text += "\\n\\n```python\\n" + self._lookup_program(aug.apply_task(task), passes, prog_test, rng) + "\\n```"\n            tokens += 900\n        return Completion(text=text, reasoning="reasoning " * (spent // 10), tokens_out=tokens,\n                          tokens_in=len(prompt) // 4, finish="stop", cut_in_thinking=False)\n\n    @staticmethod\n    def _wrong(truth, rng):\n        g = [row[:] for row in truth]\n        if rng.random() < 0.25 and len(g) > 1:\n            return g[:-1]  # wrong shape\n        for _ in range(rng.randint(1, 3)):\n            i, j = rng.randrange(len(g)), rng.randrange(len(g[0]))\n            g[i][j] = (g[i][j] + rng.randint(1, 9)) % 10\n        return g\n\n    @staticmethod\n    def _lookup_program(task, passes, test_grids, rng):\n        table = {}\n        for k, p in enumerate(task["train"]):\n            out = p["output"]\n            if not passes and k == 0:\n                out = OracleLLM._wrong(out, rng)\n            table[json.dumps(p["input"])] = out\n        for te, g in zip(task["test"], test_grids):\n            table[json.dumps(te["input"])] = g\n        return ("import json\\nimport numpy as np\\nTABLE = " + json.dumps(table) +\n                "\\ndef transform(grid):\\n    return np.array(TABLE[json.dumps(grid.tolist())])")\n',
    'arc_harness/loss.py': '"""Where each missed test output was lost.\n\nFor every test output that the final attempts miss, the first matching category (in this order):\n  parse_failure            a sample finished (own / rescued) but produced no candidate grid at all for this output\n  not_in_top_two           a right grid exists among the candidates (written or program output) but was not picked\n  verified_but_wrong       a program passed every demonstration and its test output is wrong; nothing right exists\n  right_rule_wrong_output  a near miss: a candidate has the right shape and >= 90% right cells (or <= 2 wrong\n                           cells), or a program\n                           matched all but one demonstration (the rule was essentially found, the execution was off)\n  ran_out_of_budget        no sample for the task finished on its own (every sample was cut; rescues do not count)\n  no_right_rule            samples finished on their own and nothing came close\nCategories are heuristics: \'right rule\' is read from the artefacts, not from the reasoning.\n"""\nfrom __future__ import annotations\n\nfrom collections import Counter\nfrom typing import Dict, List\n\nfrom .grids import cell_accuracy, grid_shape, is_grid\n\nCATEGORIES = ["parse_failure", "not_in_top_two", "verified_but_wrong", "right_rule_wrong_output",\n              "ran_out_of_budget", "no_right_rule"]\n\n\ndef _cands(samples: List[dict], j: int):\n    for s in samples:\n        g = s["grids"][j] if j < len(s["grids"]) else None\n        if is_grid(g):\n            yield g, s, "grid"\n        po = (s.get("verify") or {}).get("program_outputs") or []\n        if j < len(po) and is_grid(po[j]):\n            yield po[j], s, "program"\n\n\ndef classify_output(task: dict, samples: List[dict], j: int, attempts: dict) -> dict:\n    truth = task["test"][j]["output"]\n    a1, a2 = attempts.get("attempt_1"), attempts.get("attempt_2")\n    hit = (is_grid(a1) and a1 == truth) or (is_grid(a2) and a2 == truth)\n    cands = list(_cands(samples, j))\n    finished = [s for s in samples if s["finish"] in ("own", "rescued", "length")]\n    own = [s for s in samples if s["finish"] == "own"]\n    row = {"task_id": task["id"], "test": j, "hit": hit, "n_samples": len(samples), "n_candidates": len(cands),\n           "n_own_finish": len(own), "category": None}\n    if hit:\n        return row\n    if not cands:\n        row["category"] = "parse_failure" if finished else "ran_out_of_budget"\n        return row\n    if any(g == truth for g, _, _ in cands):\n        row["category"] = "not_in_top_two"\n        return row\n    verified = [s for s in samples if (s.get("verify") or {}).get("passed")]\n    if verified:\n        row["category"] = "verified_but_wrong"\n        return row\n    best_acc = max(cell_accuracy(g, truth) if grid_shape(g) == grid_shape(truth) else 0.0 for g, _, _ in cands)\n    n_cells = sum(len(r) for r in truth)\n    min_wrong = min((round((1 - cell_accuracy(g, truth)) * n_cells) if grid_shape(g) == grid_shape(truth) else n_cells)\n                    for g, _, _ in cands)\n    n_train = len(task["train"])\n    near_prog = any((s.get("verify") or {}).get("ran") and n_train >= 2 and s["verify"]["n_pass"] >= n_train - 1\n                    for s in samples)\n    row["best_cell_accuracy"] = round(best_acc, 3)\n    row["min_wrong_cells"] = min_wrong\n    if best_acc >= 0.9 or min_wrong <= 2 or near_prog:\n        row["category"] = "right_rule_wrong_output"\n        return row\n    if not own:\n        row["category"] = "ran_out_of_budget"\n        return row\n    row["category"] = "no_right_rule"\n    return row\n\n\ndef attribute(tasks: Dict[str, dict], samples: Dict[str, List[dict]], submission: dict) -> dict:\n    rows = []\n    for tid, task in tasks.items():\n        for j, te in enumerate(task["test"]):\n            if "output" not in te:\n                continue\n            att = (submission.get(tid) or [{}] * (j + 1))[j] if tid in submission and j < len(submission[tid]) else {}\n            rows.append(classify_output(task, samples.get(tid, []), j, att))\n    counts = Counter(r["category"] for r in rows if r["category"])\n    n = len(rows)\n    right = sum(r["hit"] for r in rows)\n    return {"n_outputs": n, "right": right, "missed": n - right,\n            "by_category": {c: counts.get(c, 0) for c in CATEGORIES}, "rows": rows}\n\n\ndef format_report(rep: dict) -> str:\n    lines = [f"right {rep[\'right\']}/{rep[\'n_outputs\']} ({rep[\'right\']/max(1,rep[\'n_outputs\']):.1%}); missed {rep[\'missed\']}:"]\n    for c in CATEGORIES:\n        k = rep["by_category"][c]\n        lines.append(f"  {c:26s} {k:4d}  ({k/max(1,rep[\'missed\']):.0%} of misses)")\n    return "\\n".join(lines)\n',
    'arc_harness/prompts.py': '"""Prompts. Every reply layout the harness asks for is parsed by grids.parse_outputs / grids.parse_program.\n\nLayouts\n  direct            reason, then <output_k> grids\n  direct+program    reason, then <output_k> grids AND one ```python block with transform(grid)\n  rescue            thinking off; the task plus the tail of the cut reasoning; same answer layout\n  repair            new user turn: the program\'s results on the demonstrations (Poetiq feedback); revise\n  poetiq            Poetiq\'s solver prompt (condensed, MIT, Copyright 2025 Poetiq, Inc.)\n"""\nfrom __future__ import annotations\n\nfrom typing import List, Optional\n\nfrom .grids import grid_text\n\n\ndef format_task(task: dict, seed: Optional[int] = None, style: str = "diagram") -> str:\n    """The task as text. seed shuffles the demonstration order (Poetiq does this every round);\n    the numbering in the text is the shuffled numbering, and feedback uses the same numbering."""\n    train, test = task["train"], task["test"]\n    order = list(range(len(train)))\n    if seed is not None:\n        import numpy as np            # only when shuffling: the serving Python may run without numpy\n        order = list(np.random.default_rng(seed).permutation(len(train)))\n    s = []\n    for n, i in enumerate(order, 1):\n        s.append(f"Example #{n}\\nInput:\\n{_g(train[i][\'input\'], style)}\\nOutput:\\n{_g(train[i][\'output\'], style)}\\n")\n    for n, t in enumerate(test, 1):\n        s.append(f"Test input #{n}\\n{_g(t[\'input\'], style)}\\n")\n    return "\\n".join(s)\n\n\ndef _g(grid, style):\n    if style == "diagram":\n        return f"<Diagram>\\n{grid_text(grid)}\\n</Diagram>"\n    return grid_text(grid)\n\n\nANSWER_GRIDS = """Write each test output grid as rows of digits separated by spaces, inside <output_1> ... </output_1>{more}.\nEvery row must have the same length. Do not write anything else inside the tags."""\n\n\ndef _more(n_test: int) -> str:\n    return "" if n_test == 1 else " and " + ", ".join(f"<output_{k}> ... </output_{k}>" for k in range(2, n_test + 1))\n\n\ndef direct_prompt(task: dict, seed: Optional[int] = None, style: str = "diagram") -> str:\n    n = len(task["test"])\n    return (\n        "You are solving an ARC-AGI puzzle. Each example shows an input grid and the output grid produced by the "\n        "same hidden rule. Grids are rows of digits 0-9 (0 is usually the background). Work out the rule from the "\n        "examples, check it against every example, then apply it to the test input"\n        + ("s" if n > 1 else "") + ".\\n\\n" + format_task(task, seed, style) + "\\n\\n"\n        + ANSWER_GRIDS.format(more=_more(n))\n    )\n\n\nPROGRAM_SPEC = """Then write the rule as a Python function `transform(grid: np.ndarray) -> np.ndarray` in ONE\n```python block (numpy and scipy are available; return a 2-D array of ints 0-9). The function must reproduce every\nexample output from its input and must give the same test output as the grid you wrote. If you cannot write a\nfaithful program, still write the grid; a partial program is better than none."""\n\n\ndef think_hint(tokens: int) -> str:\n    words = int(tokens * 0.6)\n    return (f"\\n\\nYou have room for about {words:,} words of reasoning before you must answer. Spend them on "\n            "checking a candidate rule against every example, not on re-reading the grids; when the rule fits all "\n            "examples, apply it and answer.")\n\n\ndef direct_program_prompt(task: dict, seed: Optional[int] = None, style: str = "diagram") -> str:\n    n = len(task["test"])\n    return (\n        "You are solving an ARC-AGI puzzle. Each example shows an input grid and the output grid produced by the "\n        "same hidden rule. Grids are rows of digits 0-9 (0 is usually the background). Work out the rule from the "\n        "examples, check it against every example, then apply it to the test input"\n        + ("s" if n > 1 else "") + ".\\n\\n" + format_task(task, seed, style) + "\\n\\n"\n        + "First, w" + ANSWER_GRIDS.format(more=_more(n))[1:] + "\\n\\n" + PROGRAM_SPEC\n    )\n\n\ndef rescue_prompt(task: dict, reasoning_tail: str, with_program: bool, style: str = "diagram") -> str:\n    """Thinking off. The model\'s own cut reasoning is shown so it can finish from where it stopped."""\n    n = len(task["test"])\n    body = (\n        "You were solving this ARC-AGI puzzle and your reasoning was cut off by a length limit. The end of your "\n        "reasoning is shown below. Do not start over: use what you had found and write the answer now.\\n\\n"\n        + format_task(task, None, style)\n        + "\\n\\n<your_reasoning_so_far>\\n...\\n" + reasoning_tail + "\\n</your_reasoning_so_far>\\n\\n"\n        + ANSWER_GRIDS.format(more=_more(n))\n    )\n    if with_program:\n        body += "\\n\\n" + PROGRAM_SPEC.replace("Then write", "After the grids, write")\n    return body\n\n\ndef repair_prompt(task: dict, program: Optional[str], feedback: str, score: float,\n                  prev_grids: List[Optional[list]], style: str = "diagram") -> str:\n    """One Poetiq feedback round on the model\'s own program, as a new user turn."""\n    n = len(task["test"])\n    parts = [\n        "You proposed a rule for this ARC-AGI puzzle, written as a Python program. It was run on the examples and "\n        "it does NOT reproduce all of them. Study the feedback, work out what the real rule is, then write a "\n        "corrected answer.\\n",\n        format_task(task, None, style),\n    ]\n    if program:\n        parts.append(f"\\nYour previous program:\\n```python\\n{program}\\n```")\n    else:\n        parts.append("\\n(Your previous reply had no runnable program.)")\n    parts.append(f"\\nResults on the examples (score {score:.2f}, 1.0 = all examples correct):\\n{feedback}")\n    shown = [(k, g) for k, g in enumerate(prev_grids, 1) if g is not None]\n    if shown:\n        parts.append("\\nThe test output grid(s) you wrote last time (numbered as the test inputs):\\n" + "\\n\\n".join(\n            f"<previous_output_{k}>\\n{grid_text(g)}\\n</previous_output_{k}>" for k, g in shown))\n    parts.append(\n        "\\nThe examples are the ground truth. If your previous grid is consistent with the corrected rule, keep it; "\n        "otherwise change it. Write a program that encodes the rule; do not hard-code the example outputs.\\n\\n"\n        "First, w" + ANSWER_GRIDS.format(more=_more(n))[1:] + "\\n\\n" + PROGRAM_SPEC\n    )\n    return "\\n".join(parts)\n\n\n# --- Poetiq (condensed from poetiq-ai/poetiq-arc-agi-solver, MIT, Copyright 2025 Poetiq, Inc.) ---\n\nPOETIQ_SOLVER = """You solve ARC tasks by writing Python. Study the examples, find the rule, and write\n`transform(grid: np.ndarray) -> np.ndarray` that turns every input into its output.\nPrefer simple rules. Return the complete function in one ```python block.\n\n{problem}"""\n\nPOETIQ_FEEDBACK = """\n\nEXISTING PARTIAL/INCORRECT SOLUTIONS, each with its code, feedback on the examples and a\nscore from 0 (worst) to 1 (best). Study them and write a new solution that fixes all the issues.\n\n{feedback}"""\n\n\ndef poetiq_prompt(task: dict, seed: int, tries: List[dict], max_shown: int = 5) -> str:\n    problem = format_task(task, seed, "diagram").replace("Test input #", "Challenge #")\n    prompt = POETIQ_SOLVER.format(problem=problem)\n    if tries:\n        shown = sorted(tries, key=lambda t: t["score"])[-max_shown:]  # best ones, worst to best\n        prompt += POETIQ_FEEDBACK.format(feedback="\\n".join(\n            f"<solution_{k}>\\n```python\\n{t[\'code\']}\\n```\\n{t[\'feedback\']}\\nScore: {t[\'score\']:.2f}\\n</solution_{k}>"\n            for k, t in enumerate(shown, 1)))\n    return prompt\n',
    'arc_harness/run.py': '"""CLI.\n\n  python -m arc_harness.run --solver vr --budget 60000 --tasks dev --backend oracle --out runs/x\n  python -m arc_harness.run --solver vr --budget 33000 --kaggle /kaggle/input/... --backend vllm --deadline-s 25000\n\n--tasks: dev | holdout | eval (all 120) | train:N (first N training tasks by size) | a json list of ids | a path\n--backend: vllm (needs --url) | oracle (CPU simulator, keyed on solutions) | fixture (a jsonl of canned replies)\nWrites out/samples.jsonl, out/submission.json, out/run.json, and when solutions exist out/score.json and\nout/loss.json.\n"""\nfrom __future__ import annotations\n\nimport argparse\nimport json\nimport os\nimport sys\nimport time\n\nfrom .budget import Budget, Caps\nfrom .data import holdout_split, load_dir, load_kaggle, task_size\nfrom .llm import OracleLLM, VLLMChat\nfrom .loss import attribute, format_report\nfrom .score import score_submission\nfrom .solvers import Config, Runner\n\nROOT = os.path.join(os.path.dirname(os.path.abspath(__file__)), "..")\n\n\ndef select_tasks(spec: str, data_root: str, kaggle_dir: str = None) -> dict:\n    if kaggle_dir:\n        ch = os.path.join(kaggle_dir, "arc-agi_test_challenges.json")\n        sol = os.path.join(kaggle_dir, "arc-agi_test_solutions.json")\n        if spec == "kaggle_test":\n            return load_kaggle(ch, sol)\n        if spec in ("kaggle_eval",):\n            return load_kaggle(os.path.join(kaggle_dir, "arc-agi_evaluation_challenges.json"),\n                               os.path.join(kaggle_dir, "arc-agi_evaluation_solutions.json"))\n    ev = load_dir(os.path.join(data_root, "ARC-AGI-2/data/evaluation"))\n    if spec == "eval":\n        return ev\n    if spec in ("dev", "holdout"):\n        dev, hold = holdout_split(ev.keys())\n        ids = dev if spec == "dev" else hold\n        return {t: ev[t] for t in ids}\n    if spec.startswith("train:"):\n        tr = load_dir(os.path.join(data_root, "ARC-AGI-2/data/training"))\n        n = int(spec.split(":")[1])\n        ids = sorted(tr, key=lambda t: task_size(tr[t]))[:n]\n        return {t: tr[t] for t in ids}\n    if spec.startswith("dev:"):\n        n = int(spec.split(":")[1])\n        dev, _ = holdout_split(ev.keys())\n        ids = sorted(dev, key=lambda t: task_size(ev[t]))[:n]\n        return {t: ev[t] for t in ids}\n    if os.path.exists(spec):\n        with open(spec) as f:\n            ids = json.load(f)\n        pool = dict(ev)\n        pool.update(load_dir(os.path.join(data_root, "ARC-AGI-2/data/training")))\n        return {t: pool[t] for t in ids}\n    raise SystemExit(f"unknown task spec {spec}")\n\n\ndef make_llm(args, tasks):\n    if args.backend == "vllm":\n        return VLLMChat(args.url, args.model, timeout_s=args.timeout_s, system=args.system or None,\n                        extra=json.loads(args.extra_body) if args.extra_body else None,\n                        template_kwargs=json.loads(args.template_kwargs) if args.template_kwargs else None)\n    if args.backend == "oracle":\n        return OracleLLM(tasks, seed=args.seed)\n    raise SystemExit(args.backend)\n\n\ndef main(argv=None):\n    ap = argparse.ArgumentParser()\n    ap.add_argument("--solver", default="vr", choices=["direct", "poetiq", "vr"])\n    ap.add_argument("--budget", type=int, default=60000, help="mean generated tokens per task")\n    ap.add_argument("--tasks", default="dev")\n    ap.add_argument("--data-root", default=os.path.join(ROOT, "data"))\n    ap.add_argument("--kaggle", default=None, help="Kaggle competition input dir")\n    ap.add_argument("--backend", default="oracle", choices=["vllm", "oracle"])\n    ap.add_argument("--url", default="http://127.0.0.1:8000/v1")\n    ap.add_argument("--model", default="qwen")\n    ap.add_argument("--timeout-s", type=float, default=3600)\n    ap.add_argument("--system", default="", help="system prompt (e.g. an effort instruction)")\n    ap.add_argument("--extra-body", default="", help=\'extra JSON fields for the request, e.g. {"reasoning_effort": "low"}\')\n    ap.add_argument("--template-kwargs", default="", help="extra chat_template_kwargs as JSON")\n    ap.add_argument("--think-hint", action="store_true", help="tell the model its thinking budget in the prompt")\n    ap.add_argument("--out", required=True)\n    ap.add_argument("--deadline-s", type=float, default=None, help="wall-clock seconds for the whole run")\n    ap.add_argument("--reserve-s", type=float, default=900)\n    ap.add_argument("--in-flight", type=int, default=20)\n    ap.add_argument("--tokens-per-s", type=float, default=334)\n    ap.add_argument("--think1", type=int, default=None)\n    ap.add_argument("--think2", type=int, default=None)\n    ap.add_argument("--no-repair", action="store_true")\n    ap.add_argument("--resamples", type=int, default=3)\n    ap.add_argument("--shuffle-demos", action="store_true")\n    ap.add_argument("--no-augment-s2", action="store_true", help="second samples on the plain view")\n    ap.add_argument("--style", default="diagram", choices=["diagram", "plain"])\n    ap.add_argument("--temperature", type=float, default=1.0)\n    ap.add_argument("--selection", default="")\n    ap.add_argument("--merge", default="R0", choices=["R0", "verified_first", "harness_first", "verified_then_margin"])\n    ap.add_argument("--external", default=None, help="json {task_id: [[attempt_1, attempt_2] per test]}")\n    ap.add_argument("--no-validator", action="store_true")\n    ap.add_argument("--seed", type=int, default=0)\n    ap.add_argument("--quiet", action="store_true")\n    args = ap.parse_args(argv)\n\n    tasks = select_tasks(args.tasks, args.data_root, args.kaggle)\n    llm = make_llm(args, tasks)\n    overrides = {k: v for k, v in (("think1", args.think1), ("think2", args.think2)) if v}\n    budget = Budget(args.budget, len(tasks), deadline=(time.time() + args.deadline_s) if args.deadline_s else None,\n                    reserve_s=args.reserve_s, tokens_per_s=args.tokens_per_s, max_in_flight=args.in_flight,\n                    caps=Caps.from_mean(args.budget, overrides))\n    cfg = Config(solver=args.solver, temperature=args.temperature, style=args.style, shuffle_demos=args.shuffle_demos,\n                 max_resamples=args.resamples, repair=not args.no_repair, augment_s2=not args.no_augment_s2, selection=args.selection, merge=args.merge,\n                 use_validator=not args.no_validator, seed=args.seed, log=not args.quiet, think_hint=args.think_hint)\n    external = None\n    if args.external:\n        with open(args.external) as f:\n            external = json.load(f)\n    print(f"[run] solver={args.solver} budget={args.budget} tasks={len(tasks)} outputs="\n          f"{sum(len(t[\'test\']) for t in tasks.values())} backend={llm.name} caps={budget.caps}", flush=True)\n    runner = Runner(tasks, llm, budget, cfg, args.out, external)\n    summary = runner.run()\n    print("[run] budget:", json.dumps(summary["budget"]), flush=True)\n    with open(runner.sub_path) as f:\n        sub = json.load(f)\n    if any("output" in te for t in tasks.values() for te in t["test"]):\n        sc = score_submission(sub, tasks)\n        rep = attribute(tasks, runner.samples, sub)\n        with open(os.path.join(args.out, "score.json"), "w") as f:\n            json.dump({k: v for k, v in sc.items() if k != "rows"} | {"rows": sc["rows"]}, f, indent=1)\n        with open(os.path.join(args.out, "loss.json"), "w") as f:\n            json.dump(rep, f, indent=1)\n        gen = summary["budget"]["spent"]\n        print(f"[score] {sc[\'right\']}/{sc[\'n_outputs\']} = {sc[\'score\']:.4f}; attempt_1 right {sc[\'attempt1_right\']}; "\n              f"tasks fully right {sc[\'tasks_fully_right\']}/{sc[\'n_tasks\']}; "\n              f"mean tokens/task {gen/len(tasks):.0f}; right per M tokens {1e6*sc[\'right\']/max(1,gen):.1f}")\n        print(format_report(rep))\n    return 0\n\n\nif __name__ == "__main__":\n    sys.exit(main())\n',
    'arc_harness/sandbox.py': '"""Run the model\'s transform(grid) in a subprocess and grade it on the demonstrations.\n\nFeedback follows Poetiq\'s format (MIT, Copyright 2025 Poetiq, Inc.): per example correct/incorrect, a shape\nmessage, a diff grid with wrong cells as \'prediction/correct\', the cell accuracy, and any error text.\nAll grids of one program run in one subprocess (one interpreter start per program instead of per grid).\nThe subprocess is not a security sandbox.\n"""\nfrom __future__ import annotations\n\nimport json\nimport os\nimport subprocess\nimport sys\nimport tempfile\nfrom dataclasses import dataclass, field\nfrom typing import List, Optional\n\nfrom .grids import Grid, cell_accuracy, grid_shape, is_grid\n\nPER_GRID_TIMEOUT_S = 5.0\nMAX_TIMEOUT_S = 40.0\n# The interpreter that runs the model\'s programs. In the hybrid notebook the harness runs under the serving Python\n# (3.12, no numpy outside the vLLM site, which -I hides), so the notebook points this at the image\'s own Python.\nSANDBOX_PYTHON_ENV = "ARC_SANDBOX_PYTHON"\n\n\ndef sandbox_python() -> str:\n    return os.environ.get(SANDBOX_PYTHON_ENV) or sys.executable\n\nRUNNER = r\'\'\'\nimport json, sys, signal\nimport numpy as np\ntry:\n    import scipy  # noqa: F401\nexcept Exception:\n    pass\ncode = sys.stdin.readline()\ngrids = json.loads(sys.stdin.readline())\nns = {}\ntry:\n    exec(compile(json.loads(code), "<model>", "exec"), ns)\n    fn = ns["transform"]\nexcept Exception as e:\n    print(json.dumps([{"out": None, "err": "definition failed: %s: %s" % (type(e).__name__, e)}] * len(grids)))\n    sys.exit(0)\nres = []\nclass _TO(Exception):\n    pass\ndef _alarm(*a):\n    raise _TO()\nsignal.signal(signal.SIGALRM, _alarm)\nfor g in grids:\n    signal.alarm(__ALARM__)\n    try:\n        out = fn(np.array(g))\n        signal.alarm(0)\n        if hasattr(out, "tolist"):\n            out = out.tolist()\n        out = [[int(c) for c in row] for row in out]\n        res.append({"out": out, "err": None})\n    except _TO:\n        res.append({"out": None, "err": "timeout"})\n    except Exception as e:\n        signal.alarm(0)\n        res.append({"out": None, "err": "%s: %s" % (type(e).__name__, str(e)[:300])})\nprint(json.dumps(res))\n\'\'\'.replace("__ALARM__", str(int(PER_GRID_TIMEOUT_S)))\n\n\ndef run_program(code: str, grids: List[Grid]) -> List[dict]:\n    """Run transform on each grid. Returns [{"out": grid|None, "err": str|None}, ...]."""\n    if not grids:\n        return []\n    with tempfile.TemporaryDirectory() as td:\n        path = os.path.join(td, "runner.py")\n        with open(path, "w") as f:\n            f.write(RUNNER)\n        stdin = json.dumps(code) + "\\n" + json.dumps(grids) + "\\n"\n        timeout = min(MAX_TIMEOUT_S, PER_GRID_TIMEOUT_S * len(grids) + 3)\n        try:\n            p = subprocess.run([sandbox_python(), "-I", path], input=stdin, capture_output=True, text=True,\n                               timeout=timeout, cwd=td)\n        except subprocess.TimeoutExpired:\n            return [{"out": None, "err": "timeout"}] * len(grids)\n    if p.returncode != 0:\n        return [{"out": None, "err": (p.stderr or p.stdout).strip()[-500:]}] * len(grids)\n    try:\n        res = json.loads(p.stdout.strip().splitlines()[-1])\n    except (json.JSONDecodeError, IndexError):\n        return [{"out": None, "err": "runner produced no result"}] * len(grids)\n    for r in res:\n        if r["out"] is not None and not is_grid(r["out"]):\n            r["err"] = (r["err"] or "") + " output is not a valid grid (rectangular, 1-30 per side, values 0-9)"\n            r["bad"] = r["out"]\n            r["out"] = None\n    return res\n\n\ndef diff_text(pred: Grid, truth: Grid) -> str:\n    """Right cells as-is, wrong cells as \'prediction/correct\'."""\n    return "\\n".join(" ".join(str(p) if p == t else f"{p}/{t}" for p, t in zip(pr, tr))\n                     for pr, tr in zip(pred, truth))\n\n\n@dataclass\nclass Grade:\n    passed: bool                     # right on every demonstration\n    score: float                     # mean cell accuracy over demonstrations\n    n_pass: int                      # demonstrations matched exactly\n    feedback: str                    # Poetiq-style text for the model\n    test_outputs: List[Optional[Grid]]\n    errors: List[Optional[str]] = field(default_factory=list)\n\n\ndef grade(code: str, train: List[dict], test: List[dict]) -> Grade:\n    grids = [p["input"] for p in train] + [t["input"] for t in test]\n    res = run_program(code, grids)\n    scores, notes, errs = [], [], []\n    for i, (p, r) in enumerate(zip(train, res[:len(train)]), 1):\n        truth = p["output"]\n        pred = r["out"]\n        errs.append(r["err"])\n        if pred is not None and pred == truth:\n            scores.append(1.0)\n            notes.append(f"Example #{i}: correct.")\n            continue\n        note = f"Example #{i}: incorrect."\n        if pred is None:\n            scores.append(0.0)\n            note += "\\nThe output has to be a rectangular grid of integers 0-9."\n        elif grid_shape(pred) != grid_shape(truth):\n            scores.append(0.0)\n            note += f"\\nShape mismatch: your output shape was {grid_shape(pred)}, the correct shape is {grid_shape(truth)}."\n        else:\n            acc = cell_accuracy(pred, truth)\n            scores.append(acc)\n            note += (f"\\nWrong cells shown as \'prediction/correct\':\\n```\\n{diff_text(pred, truth)}\\n```"\n                     f"\\nCell accuracy: {acc:.2f} (0 worst, 1 best).")\n        if r["err"]:\n            note += f"\\nYour code produced this error:\\n{r[\'err\']}"\n        notes.append(note)\n    test_outputs = [r["out"] for r in res[len(train):]]\n    n_pass = sum(1 for s in scores if s == 1.0)\n    return Grade(passed=bool(scores) and n_pass == len(scores), score=float(sum(scores) / max(1, len(scores))),\n                 n_pass=n_pass, feedback="\\n\\n".join(notes), test_outputs=test_outputs, errors=errs)\n',
    'arc_harness/score.py': '"""Local scorer: Kaggle\'s rule (1 if either attempt matches exactly; mean over all test outputs)."""\nfrom __future__ import annotations\n\nfrom typing import Dict, List, Optional\n\nfrom .grids import is_grid\n\n\ndef score_submission(sub: dict, tasks: Dict[str, dict]) -> dict:\n    rows = []\n    for tid, t in tasks.items():\n        for j, te in enumerate(t["test"]):\n            if "output" not in te:\n                continue\n            truth = te["output"]\n            e = (sub.get(tid) or [{}] * (j + 1))[j] if tid in sub and j < len(sub[tid]) else {}\n            a1, a2 = e.get("attempt_1"), e.get("attempt_2")\n            hit1 = is_grid(a1) and a1 == truth\n            hit2 = is_grid(a2) and a2 == truth\n            rows.append({"task_id": tid, "test": j, "hit": hit1 or hit2, "hit1": hit1, "hit2": hit2})\n    n = len(rows)\n    right = sum(r["hit"] for r in rows)\n    tasks_full = {}\n    for r in rows:\n        tasks_full.setdefault(r["task_id"], True)\n        tasks_full[r["task_id"]] &= r["hit"]\n    return {"n_outputs": n, "right": right, "score": right / n if n else 0.0,\n            "attempt1_right": sum(r["hit1"] for r in rows), "n_tasks": len(tasks_full),\n            "tasks_fully_right": sum(tasks_full.values()), "rows": rows}\n\n\ndef candidate_is_right(grid: Optional[list], truth: list) -> bool:\n    return grid is not None and is_grid(grid) and grid == truth\n\n\ndef per_output_table(sub: dict, tasks: Dict[str, dict]) -> List[dict]:\n    return score_submission(sub, tasks)["rows"]\n',
    'arc_harness/selection.py': '"""Rank the candidate outputs of a task into attempt_1 / attempt_2, and merge with an external pool.\n\nA Sample (see solvers.py) carries for each test input a written grid and, if it had a program, the program\'s\noutput on that test input, plus the program\'s verification result on the demonstrations.\n\nPolicies\n  grid_only       votes over written grids (the direct baseline\'s selection)\n  poetiq          Poetiq\'s ranking: passers (by votes) before failures (by votes, then demo score)\n  vr              verified programs first (by votes), then grids by self-consistency, validator, demo score,\n                  finish state, stage order\nMerge with an external pool (e.g. the NVARC phase\'s top-2):\n  R0              external attempt_1 kept; the harness\'s best differing answer takes attempt_2\n  verified_first  a verified harness answer takes attempt_1, the external attempt_1 moves to attempt_2; else R0\n  harness_first   harness top-1, then the external attempt_1\n  verified_then_margin  verified harness answer first; otherwise the external attempt_1 first only when its\n                  confidence margin (external[tid] may be {"attempts": [...], "margin": x}) is >= margin_threshold,\n                  else the harness answer first (the 4B\'s margin predicts its correctness, AUC 0.94 [brief])\n"""\nfrom __future__ import annotations\n\nimport json\nfrom dataclasses import dataclass, field\nfrom typing import Dict, List, Optional\n\nfrom .grids import Grid, hardcoded_risk, is_grid\nfrom .validator import check\n\nSTAGE_ORDER = {"s1": 0, "rescue": 0, "repair": 1, "rescue_repair": 1, "s2": 2, "rescue_s2": 2}\n\n\n@dataclass\nclass Group:\n    grid: Grid\n    verified_votes: int = 0        # samples whose program passed every demonstration and outputs this grid\n    verified_clean: int = 0        # ...and whose program carries no long literal lists (not a lookup table)\n    support: int = 0               # samples whose written grid or program output is this grid\n    double: int = 0                # samples where both the written grid and the program output are this grid\n    own_finish: int = 0            # supporting samples that finished on their own (not cut, not rescued)\n    best_score: float = 0.0        # best demonstration cell-accuracy of a supporting program\n    best_npass: int = 0\n    hard_ok: bool = True\n    earliest_stage: int = 9\n    sources: List[str] = field(default_factory=list)\n\n    def key_vr(self):\n        return (self.verified_clean > 0, self.verified_votes > 0, self.verified_votes, self.hard_ok, self.support,\n                self.double, self.best_npass, self.best_score, self.own_finish, -self.earliest_stage)\n\n    def key_grid_only(self):\n        return (self.support, self.own_finish, self.hard_ok, -self.earliest_stage)\n\n    def key_poetiq(self):\n        return (self.verified_votes > 0, self.support, self.best_score)\n\n\ndef _k(g) -> str:\n    return json.dumps(g)\n\n\ndef group_candidates(task: dict, samples: List[dict], j: int, use_validator: bool = True) -> Dict[str, Group]:\n    """Groups of identical candidate grids for test input j."""\n    groups: Dict[str, Group] = {}\n    test_in = task["test"][j]["input"]\n    for s in samples:\n        g_written = (s.get("grids") or [None] * (j + 1))[j] if j < len(s.get("grids") or []) else None\n        v = s.get("verify") or {}\n        prog_outs = v.get("program_outputs") or []\n        g_prog = prog_outs[j] if j < len(prog_outs) else None\n        if g_written is not None and not is_grid(g_written):\n            g_written = None\n        if g_prog is not None and not is_grid(g_prog):\n            g_prog = None\n        seen = set()\n        for g, kind in ((g_written, "grid"), (g_prog, "program")):\n            if g is None:\n                continue\n            k = _k(g)\n            grp = groups.get(k)\n            if grp is None:\n                grp = Group(grid=g)\n                if use_validator:\n                    grp.hard_ok = check(task["train"], test_in, g)["hard_ok"]\n                groups[k] = grp\n            if k not in seen:\n                grp.support += 1\n                seen.add(k)\n                if s.get("finish") == "own":\n                    grp.own_finish += 1\n                grp.earliest_stage = min(grp.earliest_stage, STAGE_ORDER.get(s.get("stage", ""), 5))\n                grp.sources.append(f"{s.get(\'sample_id\')}:{kind}")\n            if kind == "program" and v.get("ran"):\n                grp.best_score = max(grp.best_score, float(v.get("score", 0.0)))\n                grp.best_npass = max(grp.best_npass, int(v.get("n_pass", 0)))\n                if v.get("passed"):\n                    grp.verified_votes += 1\n                    if not hardcoded_risk(s.get("program") or ""):\n                        grp.verified_clean += 1\n        if g_written is not None and g_prog is not None and _k(g_written) == _k(g_prog):\n            groups[_k(g_written)].double += 1\n    return groups\n\n\ndef rank(task: dict, samples: List[dict], j: int, policy: str = "vr", use_validator: bool = True) -> List[Group]:\n    groups = group_candidates(task, samples, j, use_validator)\n    keyf = {"vr": Group.key_vr, "grid_only": Group.key_grid_only, "poetiq": Group.key_poetiq}[policy]\n    return sorted(groups.values(), key=keyf, reverse=True)\n\n\ndef fallback_grid(task: dict, j: int) -> Grid:\n    """A valid grid when nothing else exists: the test input itself (identity is right on ~0% but valid)."""\n    return [row[:] for row in task["test"][j]["input"]]\n\n\ndef two_attempts(task: dict, samples: List[dict], policy: str = "vr", external=None,\n                 merge: str = "R0", use_validator: bool = True, margin_threshold: float = 0.5) -> List[dict]:\n    """attempt_1 / attempt_2 for every test input of the task.\n\n    external: [[attempt_1, attempt_2] per test input] or {"attempts": that, "margin": float | [float per test]}."""\n    out = []\n    ext_margin = None\n    if isinstance(external, dict):\n        ext_margin = external.get("margin")\n        external = external.get("attempts")\n    for j in range(len(task["test"])):\n        ranked = rank(task, samples, j, policy, use_validator)\n        ours = [g.grid for g in ranked]\n        top_verified = bool(ranked) and ranked[0].verified_clean > 0\n        ext = (external[j] if external and j < len(external) else []) or []\n        ext = [g for g in ext if is_grid(g)]\n        picks: List[Grid] = []\n\n        def add(g):\n            if g is not None and is_grid(g) and all(g != p for p in picks) and len(picks) < 2:\n                picks.append(g)\n\n        if not ext:\n            for g in ours:\n                add(g)\n        elif merge == "R0":\n            add(ext[0])\n            for g in ours:\n                add(g)\n            for g in ext[1:]:\n                add(g)\n        elif merge == "verified_first":\n            if top_verified:\n                add(ours[0])\n                add(ext[0])\n                for g in ours[1:]:\n                    add(g)\n                for g in ext[1:]:\n                    add(g)\n            else:\n                add(ext[0])\n                for g in ours:\n                    add(g)\n                for g in ext[1:]:\n                    add(g)\n        elif merge == "harness_first":\n            for g in ours[:1]:\n                add(g)\n            for g in ext:\n                add(g)\n            for g in ours[1:]:\n                add(g)\n        elif merge == "verified_then_margin":\n            m = ext_margin[j] if isinstance(ext_margin, list) else ext_margin\n            ext_confident = m is not None and float(m) >= margin_threshold\n            if top_verified or (ours and not ext_confident):\n                for g in ours[:1]:\n                    add(g)\n                add(ext[0])\n                for g in ours[1:]:\n                    add(g)\n                for g in ext[1:]:\n                    add(g)\n            else:\n                add(ext[0])\n                for g in ours:\n                    add(g)\n                for g in ext[1:]:\n                    add(g)\n        else:\n            raise ValueError(merge)\n        while len(picks) < 2:\n            fb = fallback_grid(task, j)\n            if picks and picks[0] == fb:\n                fb = [[0]]\n            picks.append(fb)\n        out.append({"attempt_1": picks[0], "attempt_2": picks[1]})\n    return out\n',
    'arc_harness/solvers.py': '"""Solvers. Each turns a set of tasks into Sample records and a submission, under a Budget.\n\n  direct   the model writes the grids; cut samples are rescued; spare budget buys second samples (shortest first)\n  poetiq   the Appendix A loop (one expert, up to 10 rounds) with each task given the mean budget B\n  vr       verify-and-refine:\n             s1      grid + program in one reply; the program is run on the demonstrations (the verification bit)\n             rescue  a cut sample gets a thinking-off turn with the tail of its reasoning\n             repair  an unverified sample gets one Poetiq feedback round (new user turn, capped thinking)\n             s2      spare budget buys independent second samples on still-unverified tasks, shortest first\n           selection: verified programs first, then self-consistent grids (selection.py, policy "vr")\n\nA Sample record (one per model reply that could carry an answer):\n  task_id, sample_id, stage, parent, finish ("own" | "cut" | "rescued" | "length" | "error"),\n  gen_tokens, prompt_tokens, latency_s, grids (per test input), program, verify {ran, passed, score, n_pass,\n  program_outputs, feedback}, text (answer), reasoning_tail (last 16k chars), seed\nCorrectness is never computed here; the solvers do not see solutions.\n\nScheduling: a priority heap of jobs run by a thread pool (up to budget.max_in_flight in flight). Each task is a\nchain (s1 -> repair -> s2 -> s2 ...), one request in flight per task, so no two samples of one task race; the heap\norders chains across tasks by (stage, task size), shortest first within a stage.\n"""\nfrom __future__ import annotations\n\nimport hashlib\nimport heapq\nimport json\nimport os\nimport threading\nimport time\nfrom concurrent.futures import ThreadPoolExecutor, wait, FIRST_COMPLETED\nfrom dataclasses import dataclass, field\nfrom typing import Callable, Dict, List, Optional\n\nfrom .augment import IDENTITY, Aug, invert_sample, random_aug\nfrom .budget import Budget\nfrom .data import task_size\nfrom .grids import hardcoded_risk, parse_outputs, parse_program\nfrom .llm import LLM, Completion\nfrom .prompts import direct_program_prompt, direct_prompt, poetiq_prompt, repair_prompt, rescue_prompt, think_hint\nfrom .sandbox import grade\nfrom .selection import two_attempts\nfrom .submission import identity_submission, write_submission\n\nREASONING_TAIL_CHARS = 16000\n\n\n@dataclass\nclass Config:\n    solver: str = "vr"                   # direct | poetiq | vr\n    temperature: float = 1.0\n    style: str = "diagram"               # task text style\n    shuffle_demos: bool = False          # new demonstration order per sample (Poetiq does this every round)\n    augment_s2: bool = True              # second samples see a dihedral + colour-permuted view, mapped back\n    max_resamples: int = 3               # extra independent samples a task may get from spare budget\n    repair: bool = True                  # vr: one feedback round for unverified samples\n    repair_only_with_program: bool = False\n    poetiq_rounds: int = 10\n    poetiq_experts: int = 1\n    selection: str = ""                  # "" = the solver\'s own policy\n    merge: str = "R0"                    # with an external pool\n    use_validator: bool = True\n    seed: int = 0\n    write_every_s: float = 20.0\n    max_errors_in_a_row: int = 40\n    think_hint: bool = False             # tell the model its thinking budget in the prompt\n    log: bool = True\n    # hybrid scheduling (all optional; the defaults keep the single-group behaviour)\n    only_ids: Optional[List[str]] = None     # sample only these tasks; every task is still written (external pool)\n    quarter_ids: Optional[set] = None        # group 0: every stage of these tasks goes before group 1\'s first samples\n    think1_band: Optional[int] = None        # first-sample thinking cap for tasks outside quarter_ids\n    min_useful_band: Optional[int] = None    # smallest grant for their first sample (default: half its cap)\n    extra: dict = field(default_factory=dict)\n\n\nclass Runner:\n    def __init__(self, tasks: Dict[str, dict], llm: LLM, budget: Budget, cfg: Config, out_dir: str,\n                 external: Optional[Dict[str, List[List[list]]]] = None):\n        self.tasks = tasks\n        self.llm = llm\n        self.budget = budget\n        self.cfg = cfg\n        self.out_dir = out_dir\n        self.external = external or {}\n        os.makedirs(out_dir, exist_ok=True)\n        self.samples: Dict[str, List[dict]] = {tid: [] for tid in tasks}\n        self.sizes = {tid: task_size(t) for tid, t in tasks.items()}\n        self.lock = threading.Lock()\n        self.seq = 0\n        self.heap: List[tuple] = []\n        self.sub_path = os.path.join(out_dir, "submission.json")\n        self.samples_path = os.path.join(out_dir, "samples.jsonl")\n        self._last_write = 0.0\n        self.events: List[dict] = []\n        self.errors_in_a_row = 0\n        self.policy = cfg.selection or {"direct": "grid_only", "poetiq": "poetiq", "vr": "vr"}[cfg.solver]\n        write_submission(identity_submission(tasks), tasks, self.sub_path)   # valid from the first second\n        if os.path.exists(self.samples_path):                                 # a rerun into the same dir: keep\n            os.replace(self.samples_path, self.samples_path + f".prev-{int(time.time())}")   # the old file aside\n        open(self.samples_path, "w").close()\n\n    # ------------------------------------------------------------------ scheduling\n    def push(self, priority: tuple, fn: Callable):\n        with self.lock:\n            self.seq += 1\n            heapq.heappush(self.heap, (priority, self.seq, fn))\n\n    def pop(self):\n        with self.lock:\n            return heapq.heappop(self.heap) if self.heap else None\n\n    def run(self) -> dict:\n        self.seed_jobs()\n        futures = set()\n        with ThreadPoolExecutor(max_workers=self.budget.max_in_flight) as pool:\n            while True:\n                while len(futures) < self.budget.max_in_flight:\n                    item = self.pop()\n                    if item is None:\n                        break\n                    futures.add(pool.submit(self._guard, item[2]))\n                if not futures:\n                    break\n                done, futures = wait(futures, return_when=FIRST_COMPLETED)\n                for f in done:\n                    for pr, fn in (f.result() or []):\n                        self.push(pr, fn)\n                self.maybe_write(force=False)\n                rs = self.budget.remaining_seconds()\n                if (rs is not None and rs <= 0) or self.budget.remaining_tokens() <= 0:\n                    with self.lock:\n                        self.heap.clear()\n        self.maybe_write(force=True)\n        summary = {"budget": self.budget.summary(), "n_samples": sum(len(v) for v in self.samples.values()),\n                   "config": self.cfg.__dict__, "policy": self.policy}\n        with open(os.path.join(self.out_dir, "run.json"), "w") as f:\n            json.dump(summary, f, indent=1, default=str)\n        return summary\n\n    def _guard(self, fn):\n        try:\n            return fn()\n        except Exception as e:  # a failed job must never kill the run\n            self.log_event({"error": f"{type(e).__name__}: {e}"})\n            return []\n\n    def log_event(self, ev: dict):\n        ev["t"] = round(time.time() - self.budget.start, 1)\n        with self.lock:\n            self.events.append(ev)\n        if self.cfg.log:\n            print(json.dumps(ev), flush=True)\n\n    # ------------------------------------------------------------------ model calls\n    def call(self, tid: str, stage: str, prompt: str, think_cap: int, answer_cap: int, thinking: bool,\n             seed: int, reserve: int = 0, min_useful: int = 1500) -> Optional[Completion]:\n        want = (think_cap + answer_cap) if thinking else answer_cap\n        granted = self.budget.grant(want, min_useful=min_useful, reserve=reserve)\n        if granted <= 0:\n            self.log_event({"task": tid, "stage": stage, "skipped": "no budget"})\n            return None\n        self.budget.begin(granted)\n        t0 = time.time()\n        comp = self.llm.generate([{"role": "user", "content": prompt}], granted, thinking=thinking,\n                                 temperature=self.cfg.temperature, seed=seed,\n                                 timeout_s=self.budget.timeout_for(granted))\n        comp.latency_s = time.time() - t0\n        self.budget.charge(comp.tokens_out, comp.tokens_in, comp.latency_s, granted)\n        comp.meta["granted"] = granted\n        if comp.finish == "error":\n            self.log_event({"task": tid, "stage": stage, "error": comp.error})\n            self._on_error()\n        else:\n            self.errors_in_a_row = 0\n        return comp\n\n    def _on_error(self):\n        """A server hiccup must not drain the job heap in seconds: back off, and stop after many in a row."""\n        with self.lock:\n            self.errors_in_a_row += 1\n            n = self.errors_in_a_row\n        time.sleep(min(60, 5 * n))\n        if n >= self.cfg.max_errors_in_a_row:\n            self.log_event({"fatal": f"{n} model errors in a row; stopping new work"})\n            with self.lock:\n                self.heap.clear()\n\n    def record(self, task: dict, stage: str, parent: Optional[str], comp: Completion, with_program: bool,\n               seed: int, aug: Aug = IDENTITY) -> dict:\n        """task is the view the model saw (possibly augmented); the record is stored in the original frame."""\n        tid = task["id"]\n        n_test = len(task["test"])\n        grids = parse_outputs(comp.text, n_test) if comp.text else [None] * n_test\n        program = parse_program(comp.text) if (with_program and comp.text) else None\n        verify = {"ran": False, "passed": False, "score": 0.0, "n_pass": 0, "program_outputs": [None] * n_test,\n                  "feedback": ""}\n        if program:\n            g = grade(program, task["train"], task["test"])\n            verify = {"ran": True, "passed": g.passed, "score": g.score, "n_pass": g.n_pass,\n                      "program_outputs": g.test_outputs, "feedback": g.feedback, "errors": g.errors}\n        if comp.finish == "error":\n            finish = "error"\n        elif comp.cut_in_thinking:\n            finish = "cut"\n        elif comp.finish == "length":\n            finish = "length"\n        elif stage.startswith("rescue"):\n            finish = "rescued"\n        else:\n            finish = "own"\n        with self.lock:\n            sid = f"{tid}-{stage}-{len(self.samples[tid])}"\n            s = {"task_id": tid, "sample_id": sid, "stage": stage, "parent": parent, "finish": finish,\n                 "gen_tokens": comp.tokens_out, "prompt_tokens": comp.tokens_in, "granted": comp.meta.get("granted"),\n                 "latency_s": round(comp.latency_s, 1), "grids": grids, "program": program, "verify": verify,\n                 "text": comp.text[-20000:], "reasoning_tail": comp.reasoning[-REASONING_TAIL_CHARS:],\n                 "reasoning_chars": len(comp.reasoning), "seed": seed, "with_program": with_program}\n            invert_sample(aug, s)\n            self.samples[tid].append(s)\n            with open(self.samples_path, "a") as f:\n                f.write(json.dumps(s) + "\\n")\n        self.log_event({"task": tid, "stage": stage, "finish": finish, "tokens": comp.tokens_out,\n                        "grids": sum(g is not None for g in grids), "program": bool(program),\n                        "verified": verify["passed"], "score": round(verify["score"], 2)})\n        return s\n\n    def maybe_write(self, force: bool):\n        now = time.time()\n        if not force and now - self._last_write < self.cfg.write_every_s:\n            return\n        self._last_write = now\n        sub = {}\n        with self.lock:\n            snapshot = {tid: list(v) for tid, v in self.samples.items()}\n        for tid, task in self.tasks.items():\n            sub[tid] = two_attempts(task, snapshot[tid], self.policy, self.external.get(tid), self.cfg.merge,\n                                    self.cfg.use_validator)\n        write_submission(sub, self.tasks, self.sub_path)\n\n    # ------------------------------------------------------------------ job graphs\n    def _group(self, tid: str) -> int:\n        q = self.cfg.quarter_ids\n        return 0 if q is None or tid in q else 1\n\n    def _prio(self, tid: str, stage_rank: int) -> tuple:\n        """Heap priority: (group, stage, task size). One group (the default) orders stage-major, shortest first."""\n        return (self._group(tid), stage_rank, self.sizes[tid])\n\n    def seed_jobs(self):\n        ids = self.tasks if self.cfg.only_ids is None else [t for t in self.cfg.only_ids if t in self.tasks]\n        order = sorted(ids, key=lambda t: self.sizes[t])\n        for tid in order:\n            if self.cfg.solver == "poetiq":\n                self.push(self._prio(tid, 0), lambda tid=tid: self.job_poetiq_round(tid, 0, [], 0))\n            else:\n                self.push(self._prio(tid, 0), lambda tid=tid: self.job_sample(tid, "s1", 0))\n\n    def _next_s2(self, tid: str, r: int):\n        """The next link of a task\'s chain: second sample r+1, if allowed."""\n        nr = r + 1\n        if nr > self.cfg.max_resamples:\n            return []\n        return [(self._prio(tid, 2 + nr), lambda: self.job_sample(tid, "s2", nr))]\n\n    @staticmethod\n    def _aug_seed(tid: str, seed: int) -> int:\n        return int(hashlib.sha256(f"{tid}:{seed}".encode()).hexdigest()[:8], 16)\n\n    def _task_done(self, tid: str) -> bool:\n        """vr: a verified answer exists. direct: two samples agree on every test output."""\n        with self.lock:\n            ss = list(self.samples[tid])\n        if self.cfg.solver == "vr":\n            # done = a verified program whose test output the model\'s own written grid agrees with (or no grid\n            # was written); a verified program that contradicts the written grid is not trusted on its own\n            for s in ss:\n                if not s["verify"]["passed"] or hardcoded_risk(s.get("program") or ""):\n                    continue\n                po = s["verify"]["program_outputs"]\n                if all(g is None or (j < len(po) and g == po[j]) for j, g in enumerate(s["grids"])):\n                    return True\n            return False\n        n_test = len(self.tasks[tid]["test"])\n        for j in range(n_test):\n            grids = [json.dumps(s["grids"][j]) for s in ss if s["grids"][j] is not None]\n            if not grids or max(grids.count(g) for g in set(grids)) < 2:\n                return False\n        return True\n\n    def job_sample(self, tid: str, stage: str, r: int, retry: int = 0):\n        """One independent sample (s1 or s2) and the next link of the task\'s chain."""\n        task = self.tasks[tid]\n        if stage == "s2" and self._task_done(tid):\n            return []\n        with_program = self.cfg.solver == "vr"\n        seed = self.cfg.seed * 1000 + r * 10 + retry\n        dseed = seed if (self.cfg.shuffle_demos and r > 0) else None\n        aug = random_aug(self._aug_seed(tid, seed)) if (stage == "s2" and self.cfg.augment_s2) else IDENTITY\n        view = aug.apply_task(task) if not aug.is_identity else task\n        prompt = (direct_program_prompt if with_program else direct_prompt)(view, dseed, self.cfg.style)\n        caps = self.budget.caps\n        think1 = caps.think1 if self._group(tid) == 0 else (self.cfg.think1_band or caps.think1)\n        if self.cfg.think_hint:\n            prompt += think_hint(think1)\n        # a sample that might be cut needs its rescue affordable too; a second sample, or a first sample outside\n        # the quarter, is not worth starting with less than half its cap\n        min_useful = 1500 if (stage == "s1" and self._group(tid) == 0) else (think1 + caps.answer) // 2\n        if stage == "s1" and self._group(tid) == 1 and self.cfg.min_useful_band is not None:\n            min_useful = self.cfg.min_useful_band\n        comp = self.call(tid, stage, prompt, think1, caps.answer, True, seed, reserve=caps.rescue,\n                         min_useful=min_useful)\n        if comp is None:\n            return []\n        s = self.record(view, stage, None, comp, with_program, seed, aug)\n        if s["finish"] == "error":\n            if stage == "s1" and retry < 2:     # a server error is not a sample: try the first sample again\n                return [(self._prio(tid, 0), lambda: self.job_sample(tid, "s1", r, retry + 1))]\n            return self._next_s2(tid, r)\n        if s["finish"] == "cut":\n            rs = self.rescue(view, s, with_program, seed, aug)\n            if rs is None:\n                return []\n            s = rs\n        if stage == "s1":\n            follow = self.after_sample(task, s)   # the repair job continues the chain itself\n            return follow if follow else self._next_s2(tid, 0)\n        return self._next_s2(tid, r)\n\n    def rescue(self, task: dict, cut: dict, with_program: bool, seed: int, aug: Aug = IDENTITY) -> Optional[dict]:\n        prompt = rescue_prompt(task, cut["reasoning_tail"], with_program, self.cfg.style)\n        comp = self.call(task["id"], "rescue", prompt, 0, self.budget.caps.rescue, False, seed)\n        if comp is None:\n            return None\n        stage = {"s1": "rescue", "repair": "rescue_repair", "s2": "rescue_s2"}.get(cut["stage"], "rescue")\n        return self.record(task, stage, cut["sample_id"], comp, with_program, seed, aug)\n\n    def after_sample(self, task: dict, s: dict):\n        """vr: queue a repair round if the sample is not verified."""\n        caps = self.budget.caps\n        if self.cfg.solver != "vr" or not self.cfg.repair or not caps.repair_enabled:\n            return []\n        if s["verify"]["passed"] or s["finish"] == "error" or s["stage"] in ("repair", "rescue_repair"):\n            return []\n        if self.cfg.repair_only_with_program and not s["program"]:\n            return []\n        if s["stage"] in ("s2", "rescue_s2"):\n            return []   # one repair per task, on the first sample\n        tid = task["id"]\n        return [(self._prio(tid, 1), lambda: self.job_repair(tid, s))]\n\n    def job_repair(self, tid: str, s: dict):\n        task = self.tasks[tid]\n        if self._task_done(tid):\n            return []\n        v = s["verify"]\n        feedback = v["feedback"] if v["ran"] else "(no runnable program was found in your reply)"\n        prompt = repair_prompt(task, s["program"], feedback, v["score"] if v["ran"] else 0.0, s["grids"],\n                               self.cfg.style)\n        caps = self.budget.caps\n        seed = s["seed"] + 500\n        comp = self.call(tid, "repair", prompt, caps.think2, caps.answer, True, seed, reserve=caps.rescue,\n                         min_useful=max(1500, caps.think2 // 2))\n        if comp is None:\n            return []\n        rs = self.record(task, "repair", s["sample_id"], comp, True, seed)\n        if rs["finish"] == "cut":\n            self.rescue(task, rs, True, seed)\n        return self._next_s2(tid, 0)\n\n    def job_poetiq_round(self, tid: str, it: int, tries: List[dict], spent: int):\n        """One round of the Poetiq loop for one task; the task\'s own budget is the mean B."""\n        task = self.tasks[tid]\n        per_task = self.budget.mean_tokens_per_task\n        if it >= self.cfg.poetiq_rounds or spent >= per_task:\n            return []\n        seed = self.cfg.seed * 1000 + it\n        prompt = poetiq_prompt(task, seed, tries)\n        caps = self.budget.caps\n        think = min(caps.think1, max(1500, per_task - spent - caps.answer))\n        comp = self.call(tid, f"poetiq_r{it}", prompt, think, caps.answer, True, seed, reserve=caps.rescue)\n        if comp is None:\n            return []\n        s = self.record(task, f"poetiq_r{it}", None, comp, True, seed)\n        spent += comp.tokens_out\n        if s["finish"] == "cut":\n            rs = self.rescue(task, s, True, seed)\n            if rs is None:\n                return []\n            spent += rs["gen_tokens"]\n            s = rs\n        if s["verify"]["passed"]:\n            return []\n        if s["program"]:\n            tries = tries + [{"code": s["program"], "feedback": s["verify"]["feedback"], "score": s["verify"]["score"]}]\n        return [(self._prio(tid, 0), lambda: self.job_poetiq_round(tid, it + 1, tries, spent))]\n\n\ndef load_samples(path: str) -> Dict[str, List[dict]]:\n    out: Dict[str, List[dict]] = {}\n    if not os.path.exists(path):\n        return out\n    with open(path) as f:\n        for line in f:\n            if line.strip():\n                s = json.loads(line)\n                out.setdefault(s["task_id"], []).append(s)\n    return out\n',
    'arc_harness/submission.py': '"""An always-valid submission.json.\n\nvalidate_submission() enforces the Kaggle shape: every task id, one entry per test input, attempt_1 and attempt_2\nboth present and valid grids. write_submission() writes atomically (temp file + rename), so a crash mid-write\nnever leaves a half file. The harness writes an identity submission first thing, then rewrites after every task.\n"""\nfrom __future__ import annotations\n\nimport json\nimport os\nimport tempfile\nfrom typing import Dict, List\n\nfrom .grids import is_grid\n\n\ndef identity_submission(tasks: Dict[str, dict]) -> Dict[str, List[dict]]:\n    sub = {}\n    for tid, t in tasks.items():\n        sub[tid] = [{"attempt_1": [r[:] for r in te["input"]], "attempt_2": [[0]]} for te in t["test"]]\n    return sub\n\n\ndef validate_submission(sub: dict, tasks: Dict[str, dict]) -> List[str]:\n    problems = []\n    for tid, t in tasks.items():\n        if tid not in sub:\n            problems.append(f"{tid}: missing")\n            continue\n        ent = sub[tid]\n        if not isinstance(ent, list) or len(ent) != len(t["test"]):\n            problems.append(f"{tid}: expected {len(t[\'test\'])} test entries")\n            continue\n        for j, e in enumerate(ent):\n            for a in ("attempt_1", "attempt_2"):\n                if a not in e or not is_grid(e[a]):\n                    problems.append(f"{tid}[{j}].{a}: not a valid grid")\n    extra = set(sub) - set(tasks)\n    if extra:\n        problems.append(f"extra ids: {sorted(extra)[:5]}")\n    return problems\n\n\ndef repair_submission(sub: dict, tasks: Dict[str, dict]) -> dict:\n    """Fill anything missing or invalid with the identity fallback, never raising."""\n    fixed = identity_submission(tasks)\n    for tid, t in tasks.items():\n        ent = sub.get(tid)\n        if not isinstance(ent, list):\n            continue\n        for j in range(min(len(ent), len(t["test"]))):\n            e = ent[j] if isinstance(ent[j], dict) else {}\n            for a in ("attempt_1", "attempt_2"):\n                if is_grid(e.get(a)):\n                    fixed[tid][j][a] = e[a]\n    return fixed\n\n\ndef write_submission(sub: dict, tasks: Dict[str, dict], path: str) -> dict:\n    sub = repair_submission(sub, tasks)\n    assert not validate_submission(sub, tasks)\n    d = os.path.dirname(os.path.abspath(path)) or "."\n    os.makedirs(d, exist_ok=True)\n    fd, tmp = tempfile.mkstemp(dir=d, prefix=".sub-", suffix=".json")\n    with os.fdopen(fd, "w") as f:\n        json.dump(sub, f)\n    os.replace(tmp, path)\n    return sub\n',
    'arc_harness/validator.py': '"""Cheap hard checks on a candidate test output, inferred from the demonstrations.\n\nTwo rules, each applied only when every demonstration obeys it:\n  shape:   same as input | integer up/down scale of the input | transposed input\n  palette: output colours are a subset of the test input\'s colours plus the colours any demonstration introduces\nA candidate that breaks a rule is demoted in ranking, never dropped (the rules are checked on the public sets\nby scripts/check_validator.py: they must never reject a true output there).\n"""\nfrom __future__ import annotations\n\nfrom typing import List, Optional, Set, Tuple\n\nfrom .grids import Grid, grid_shape\n\nShape = Tuple[int, int]\n\n\ndef _colors(g: Grid) -> Set[int]:\n    return {c for r in g for c in r}\n\n\ndef predict_shape(train: List[dict], test_input: Grid) -> Optional[Shape]:\n    ins = [grid_shape(p["input"]) for p in train]\n    outs = [grid_shape(p["output"]) for p in train]\n    ti = grid_shape(test_input)\n    if not train or ti is None or None in ins or None in outs:\n        return None\n    if all(i == o for i, o in zip(ins, outs)):\n        return ti\n    if all(o == (i[1], i[0]) for i, o in zip(ins, outs)) and any(i[0] != i[1] for i in ins):\n        return (ti[1], ti[0])\n    # (a constant output shape is NOT trusted: on the public sets it wrongly rejects true outputs,\n    #  e.g. eval 38007db0 and train c909285e, where the constant was a coincidence of the demonstrations)\n    # integer scale factors, constant across demonstrations\n    for num, den in [(k, 1) for k in range(2, 11)] + [(1, k) for k in range(2, 11)]:\n        if all(o[0] * den == i[0] * num and o[1] * den == i[1] * num for i, o in zip(ins, outs)):\n            if (ti[0] * num) % den == 0 and (ti[1] * num) % den == 0:\n                return (ti[0] * num // den, ti[1] * num // den)\n    return None\n\n\ndef allowed_colors(train: List[dict], test_input: Grid) -> Set[int]:\n    introduced: Set[int] = set()\n    for p in train:\n        introduced |= _colors(p["output"]) - _colors(p["input"])\n    return _colors(test_input) | introduced\n\n\ndef check(train: List[dict], test_input: Grid, cand: Grid) -> dict:\n    """Returns {"shape_ok": bool|None, "palette_ok": bool, "hard_ok": bool}. None = rule not applicable."""\n    ps = predict_shape(train, test_input)\n    shape_ok = None if ps is None else (grid_shape(cand) == ps)\n    palette_ok = _colors(cand) <= allowed_colors(train, test_input)\n    return {"shape_ok": shape_ok, "palette_ok": palette_ok, "hard_ok": (shape_ok is not False) and palette_ok}\n',
    'arc_harness/NOTICES.md': '# Third-party notices\n\n## Poetiq ARC-AGI solver (MIT)\n`arc_harness/prompts.py` (poetiq_prompt, POETIQ_SOLVER, POETIQ_FEEDBACK), `arc_harness/sandbox.py` (feedback\nformat) and the `poetiq` solver in `arc_harness/solvers.py` are condensed from poetiq-ai/poetiq-arc-agi-solver.\n\nMIT License. Copyright (c) 2025 Poetiq, Inc.\nPermission is hereby granted, free of charge, to any person obtaining a copy of this software and associated\ndocumentation files (the "Software"), to deal in the Software without restriction, including without limitation the\nrights to use, copy, modify, merge, publish, distribute, sublicense, and/or sell copies of the Software, and to\npermit persons to whom the Software is furnished to do so, subject to the following conditions: The above copyright\nnotice and this permission notice shall be included in all copies or substantial portions of the Software.\nTHE SOFTWARE IS PROVIDED "AS IS", WITHOUT WARRANTY OF ANY KIND, EXPRESS OR IMPLIED, INCLUDING BUT NOT LIMITED TO\nTHE WARRANTIES OF MERCHANTABILITY, FITNESS FOR A PARTICULAR PURPOSE AND NONINFRINGEMENT. IN NO EVENT SHALL THE\nAUTHORS OR COPYRIGHT HOLDERS BE LIABLE FOR ANY CLAIM, DAMAGES OR OTHER LIABILITY, WHETHER IN AN ACTION OF CONTRACT,\nTORT OR OTHERWISE, ARISING FROM, OUT OF OR IN CONNECTION WITH THE SOFTWARE OR THE USE OR OTHER DEALINGS IN THE\nSOFTWARE.\n\n## Data\nARC-AGI-2 (arcprize/ARC-AGI-2, Apache-2.0), ARC-AGI-1 (fchollet/ARC-AGI, Apache-2.0), RE-ARC (michaelhodel/re-arc,\nMIT), ConceptARC (victorvikram/ConceptARC, MIT), BARC (xu3kev/BARC, MIT). Used as evaluation data and for the\nvalidator checks only; no model is trained here.\n\n## Model\nQwen3.8-27B (Apache-2.0), served with vLLM (Apache-2.0).\n',
    'hybrid_vr.py': '"""Phase 2 of the ARC-AGI-2 hybrid notebook with the verify-and-refine harness (arc_harness, solver "vr") in place of\nversion 7\'s direct mode.\n\n    python hybrid_vr.py run <config.json>      as hybrid_p2.py run: install wait, server ladder, smoke request and\n                                               server shutdown are hybrid_p2\'s; the answering loop is the harness\n\nThe harness asks for the test grids and a transform(grid) program in one reply, runs the program on the\ndemonstrations (the verification bit), rescues replies cut in their reasoning, gives an unverified first sample one\nfeedback round, and spends what is left on second samples; selection puts verified programs first. Every task in\nthe challenges file is written: tasks outside the route keep B0\'s entry, routed tasks get B0\'s pool merged with the\nharness\'s candidates (cfg["vr"]["merge"]). submission.json is rewritten atomically every write_every_s seconds, so a\nkill leaves the last valid file.\n\nScheduling (cfg["route"] is shortest task JSON first): group 0 = the shortest ceil(n/4) tasks, every stage (first\nsample, repair, second samples) before group 1\'s first samples; within a group stage-major, shortest first. The\nharness\'s own deadline guard clips every request to what the per-stream rate allows before the collection deadline.\n\nThe model\'s programs run under cfg["vr"]["sandbox_python"] (the image\'s Python, with numpy and scipy): this process\nruns under the serving Python, whose numpy lives only in the vLLM site.\n"""\nimport json\nimport os\nimport sys\nimport time\nimport traceback\n\nHERE = os.path.dirname(os.path.abspath(__file__))\nsys.path.insert(0, HERE)\n\nPLACEHOLDER = [[0]]\n\n\ndef b0_pool(path, tasks):\n    """B0\'s submission as the harness\'s external pool {task: [[attempt_1, attempt_2] per test]}; placeholders out."""\n    try:\n        with open(path) as f:\n            sub = json.load(f)\n    except Exception:\n        return {}\n    pool = {}\n    for tid, task in tasks.items():\n        ent = sub.get(tid)\n        if not isinstance(ent, list):\n            continue\n        per_test = []\n        for j in range(len(task["test"])):\n            e = ent[j] if j < len(ent) and isinstance(ent[j], dict) else {}\n            per_test.append([g for g in (e.get("attempt_1"), e.get("attempt_2")) if g is not None and g != PLACEHOLDER])\n        pool[tid] = per_test\n    return pool\n\n\ndef read_samples(path):\n    """The harness\'s samples.jsonl by task; a line cut by a kill mid-write is skipped."""\n    out = {}\n    if not os.path.isfile(path):\n        return out\n    with open(path) as f:\n        for line in f:\n            try:\n                s = json.loads(line)\n            except ValueError:\n                continue\n            out.setdefault(s["task_id"], []).append(s)\n    return out\n\n\ndef full_submission(tasks, route, samples, pool, b0_path, merge):\n    """Every task: B0\'s entry; routed tasks: the harness\'s two attempts over B0\'s pool."""\n    from arc_harness.selection import two_attempts\n    try:\n        with open(b0_path) as f:\n            sub = json.load(f)\n    except Exception:\n        sub = {}\n    sub = {t: sub.get(t) for t in tasks}\n    for tid in route:\n        if tid in tasks:\n            sub[tid] = two_attempts(tasks[tid], samples.get(tid, []), "vr", pool.get(tid), merge, True)\n    return sub\n\n\ndef run_vr(cfg, st=None):\n    """hybrid_p2.run_routed\'s replacement; returns the summary keys hybrid_p2.run reports."""\n    vr = cfg["vr"]\n    os.environ["ARC_SANDBOX_PYTHON"] = vr["sandbox_python"]\n    from arc_harness.budget import Budget, Caps\n    from arc_harness.data import load_kaggle\n    from arc_harness.llm import VLLMChat\n    from arc_harness.solvers import Config, Runner\n    from arc_harness.submission import write_submission\n\n    t0 = time.time()\n    tasks = load_kaggle(cfg["challenges"])\n    route = [t for t in cfg["route"] if t in tasks]\n    quarter = set(route[:-(-len(route) // 4)])\n    pool = b0_pool(cfg["b0_submission"], tasks)\n    out = os.path.join(cfg["out_dir"], "vr")\n    llm = VLLMChat(cfg["base_url"], cfg["model"], top_p=cfg.get("top_p", 0.95), top_k=cfg.get("top_k", 20),\n                   template_kwargs=cfg.get("template_kwargs"))\n    caps = Caps.from_mean(vr["mean_tokens"], vr["caps"])\n    budget = Budget(vr["mean_tokens"], max(1, len(route)), deadline=float(cfg["collect_end_epoch"]),\n                    reserve_s=vr["reserve_s"], tokens_per_s=vr["tokens_per_s"],\n                    max_in_flight=int(cfg.get("concurrency", 20)), caps=caps)\n    conf = Config(solver="vr", temperature=cfg.get("temperature", 1.0), merge=vr["merge"], log=True,\n                  write_every_s=vr["write_every_s"], max_resamples=vr["max_resamples"], only_ids=route,\n                  quarter_ids=quarter, think1_band=vr.get("think1_band"), min_useful_band=vr.get("min_useful_band"))\n    print(f"vr: {len(route)} tasks routed ({len(quarter)} in group 0), {len(tasks)} written; caps {caps}; "\n          f"deadline in {float(cfg[\'collect_end_epoch\']) - time.time():.0f} s; merge {vr[\'merge\']}", flush=True)\n\n    class HybridRunner(Runner):\n        def maybe_write(self, force):\n            now = time.time()\n            if not force and now - self._last_write < self.cfg.write_every_s:\n                return\n            self._last_write = now\n            with self.lock:\n                snapshot = {tid: list(v) for tid, v in self.samples.items()}\n            sub = full_submission(tasks, route, snapshot, pool, cfg["b0_submission"], self.cfg.merge)\n            write_submission(sub, tasks, cfg["submission"])\n            print(f"vr | merge written: {sum(len(v) for v in snapshot.values())} samples", flush=True)\n\n    runner = HybridRunner({t: tasks[t] for t in route}, llm, budget, conf, out, None)\n    summary = runner.run()\n    answered = sum(1 for t in route if any(g is not None for s in runner.samples.get(t, []) for g in s["grids"]))\n    verified = sum(1 for t in route if any(s["verify"]["passed"] for s in runner.samples.get(t, [])))\n    wall = time.time() - t0\n    b = summary["budget"]\n    print(f"vr done: {summary[\'n_samples\']} samples, {answered} tasks with a grid, {verified} with a verified "\n          f"program; {b[\'spent\']} generated tokens in {wall:.0f} s ({b[\'spent\'] / max(1.0, wall):.1f} tok/s)",\n          flush=True)\n    return {"started": summary["n_samples"], "skipped": [], "answered_tasks": answered,\n            "merge": {"replaced": None, "verified_tasks": verified}, "agent_wall_s": round(wall, 1),\n            "gen_tok_per_s": round(b["spent"] / max(1.0, wall), 1)}\n\n\nif __name__ == "__main__":\n    try:\n        mode, path = sys.argv[1], sys.argv[2]\n        with open(path) as f:\n            config = json.load(f)\n        import hybrid_p2\n        if mode == "install":\n            hybrid_p2.install(config)\n        else:\n            hybrid_p2.run_routed = run_vr\n            hybrid_p2.run(config)\n    except BaseException:\n        print(traceback.format_exc(), flush=True)\n    sys.exit(0)\n',
}
for _rel, _txt in VR_FILES.items():
    _p = os.path.join(HYBRID_DIR, _rel)
    os.makedirs(os.path.dirname(_p), exist_ok=True)
    with open(_p, 'w') as _fh:
        _fh.write(_txt)
VR_CONFIG = {
    'mean_tokens': 60000,              # pool = mean x routed tasks; time, not the pool, binds in this window
    # first sample 49,152 tokens (43,152 + 6,000: version 7's cap), repair round 18,000 + 6,000, rescue 8,192
    'caps': {'think1': 43152, 'think2': 18000, 'answer': 6000, 'rescue': 8192},
    'think1_band': None,               # tasks beyond the shortest quarter: the same cap (version 7: band off)
    'max_resamples': 3,
    'merge': 'verified_first',         # a clean verified 27B answer takes attempt_1, B0's attempt_1 moves to attempt_2
    # requests end this long before the collection deadline (the CPU twin: 5 s, as its direct-mode caps)
    'reserve_s': 5.0 if globals().get('PREFLIGHT', False) else 120.0,
    # 12 tok/s per request x 20 in flight until measured (hybrid_p2's fallback); the CPU twin's stub answers at once,
    # so its compressed timeline plans as its direct-mode caps do (100,000 tok/s per request): every stage runs
    'tokens_per_s': 2000000.0 if globals().get('PREFLIGHT', False) else 240.0,
    # a first sample beyond the shortest quarter starts only with half its cap in the competition rerun; a commit
    # run's short window runs them at reduced caps (version 6's band rule: full fit in the rerun, 4,096 otherwise)
    'min_useful_band': None if HYBRID_RERUN else 4096,
    'write_every_s': 20.0,
    'sandbox_python': sys.executable,  # the image's Python (numpy, scipy) runs the model's programs
}
print('verify-and-refine harness written:', len(VR_FILES), 'files, sha256',
      hashlib.sha256(''.join(VR_FILES[k] for k in sorted(VR_FILES)).encode()).hexdigest()[:16], '| VR_CONFIG', VR_CONFIG)


verify-and-refine harness written: 18 files, sha256 310ecdc28bede7dd | VR_CONFIG {'mean_tokens': 60000, 'caps': {'think1': 43152, 'think2': 18000, 'answer': 6000, 'rescue': 8192}, 'think1_band': None, 'max_resamples': 3, 'merge': 'verified_first', 'reserve_s': 120.0, 'tokens_per_s': 240.0, 'min_useful_band': 4096, 'write_every_s': 20.0, 'sandbox_python': '/usr/bin/python3'}


In [15]:
%%writefile /kaggle/working/hybrid/hybrid_stub_server.py
"""Scripted stand-in for the vLLM OpenAI-compatible server in the hybrid notebook's CPU twin and tests (phase 2,
direct-answer mode, samples 0 and 1).

Serves GET /v1/models and POST /v1/chat/completions on 127.0.0.1:<port>. It finds the task from the first
demonstration pair in the user prompt, the sample from the seed (seed // 1000: 7 and 8 are sample 0, 1007 and 1008
sample 1) and a rescue request from chat_template_kwargs.enable_thinking == False, and answers from a script chosen
by the task's position in --task-ids (the route order), index % 7:

  0 none      no grid in the reply nor in the rescue; sample 1 answers the solution
  1 solve     the solution grids in ```grid blocks
  2 rescue    the reply runs to max_tokens inside its reasoning (no grid); the rescue request answers the solution
  3 slow      the reply takes --slow-s seconds (default 3600): cut by the HTTP timeout or a kill
  4 same      B0's attempt_1 grids (--b0-submission): the merge must leave attempt_2 alone; sample 1 answers the
              solution
  5 wrong     a wrong grid (the solution with its first cell changed); sample 1 answers the solution (the merge
              keeps sample 0's grid: it differs from attempt_1)
  6 overflow  max_tokens > 40000 gets HTTP 400 "maximum context length"; the retry at 0.75 x answers the solution

"What is 2+2?" (the smoke request) is answered "4". The solution is read from --solutions, else the test input
stands in for it. Every request is logged to --log (jsonl).

--tok-s R (version 5's deadline-cap tests; default 0, off): a task reply is sent only after its completion tokens / R
seconds, as a server decoding R tokens per second per request would (a reply that runs to max_tokens takes
max_tokens / R; a request cut by the client's timeout before then returns nothing).

    python hybrid_stub_server.py --port 1234 --challenges <file> --task-ids a,b,... [--solutions <file>]
        [--b0-submission <file>] [--log <file>] [--slow-s 3600] [--tok-s 0]
"""
import argparse
import json
import threading
import time
from http.server import BaseHTTPRequestHandler, ThreadingHTTPServer

KINDS = ("none", "solve", "rescue", "slow", "same", "wrong", "overflow")
SECOND_DRAW = {"none": "solve", "same": "solve", "wrong": "solve"}   # sample >= 1
REPLY_TOKENS = 3000
RESCUE_TOKENS = 400
OVERFLOW_ABOVE = 40000


def show(g, name):
    """grids.show's layout (the prompt carries it)."""
    return f"{name} ({len(g)}x{len(g[0]) if g else 0}):\n" + "\n".join("".join(str(int(v)) for v in r) for r in g)


def wrong_grid(g):
    w = [list(r) for r in g]
    w[0][0] = (w[0][0] + 1) % 10
    return w


def grid_blocks(grids):
    return "\n\n".join("```grid\n" + "\n".join("".join(str(v) for v in r) for r in g) + "\n```" for g in grids)


class Script:
    def __init__(self, challenges, task_ids, solutions=None, b0_submission=None, slow_s=3600.0, tok_s=0.0):
        self.ch, self.ids, self.sol, self.b0 = challenges, list(task_ids), solutions or {}, b0_submission or {}
        self.slow_s = float(slow_s)
        self.tok_s = float(tok_s or 0.0)
        self.kind = {t: KINDS[i % len(KINDS)] for i, t in enumerate(self.ids)}
        self.keys = {}
        for t in self.ids:
            p = challenges[t]["train"][0]
            self.keys[show(p["input"], "Input") + "\n" + show(p["output"], "Output")] = t
            # the verify-and-refine harness's layout: Input: / <Diagram> rows of space-separated digits </Diagram>
            _rows = lambda g: "\n".join(" ".join(str(int(v)) for v in r) for r in g)  # noqa: E731
            self.keys["Input:\n<Diagram>\n" + _rows(p["input"]) + "\n</Diagram>\nOutput:\n<Diagram>\n"
                      + _rows(p["output"]) + "\n</Diagram>"] = t

    def find_task(self, messages):
        text = "\n".join(m.get("content") or "" for m in messages if m.get("role") == "user")
        for key, t in self.keys.items():
            if key in text:
                return t
        return None

    def kind_for(self, t, seed):
        """The script for task t and the sample the seed belongs to (seed // 1000)."""
        kind = self.kind[t]
        sample = int(seed or 0) // 1000
        return SECOND_DRAW.get(kind, kind) if sample >= 1 else kind

    def solution(self, t):
        sol = self.sol.get(t)
        return [sol[j] if sol else x["input"] for j, x in enumerate(self.ch[t]["test"])]

    def reply(self, payload):
        """(http status, content, reasoning, completion tokens or None for max_tokens, task, kind, rescue)."""
        msgs = payload.get("messages", [])
        text = "\n".join(m.get("content") or "" for m in msgs)
        t = self.find_task(msgs)
        rescue = (payload.get("chat_template_kwargs") or {}).get("enable_thinking") is False
        if t is None:
            if "2+2" in text:
                return 200, "4", "", 2, None, "smoke", rescue
            return 200, "", "", 1, None, "unknown", rescue
        kind = self.kind_for(t, payload.get("seed"))
        max_tokens = int(payload.get("max_tokens") or 16384)
        good = "The rule holds on every example.\n\n" + grid_blocks(self.solution(t))
        if kind == "solve":
            return 200, good, "check every example", REPLY_TOKENS, t, kind, rescue
        if kind == "rescue":
            if rescue:
                return 200, good, "", RESCUE_TOKENS, t, kind, rescue
            return 200, "", "still analysing " * 50, None, t, kind, rescue
        if kind == "wrong":
            grids = [wrong_grid(g) for g in self.solution(t)]
            return 200, "Final answer:\n\n" + grid_blocks(grids), "compare pairs", REPLY_TOKENS, t, kind, rescue
        if kind == "same":
            grids = [e["attempt_1"] for e in self.b0.get(t, [])] or self.solution(t)
            return 200, "Final answer:\n\n" + grid_blocks(grids), "as before", REPLY_TOKENS, t, kind, rescue
        if kind == "slow":
            time.sleep(self.slow_s)
            return 200, good, "slow", REPLY_TOKENS, t, kind, rescue
        if kind == "overflow":
            if max_tokens > OVERFLOW_ABOVE:
                return (400, f"This model's maximum context length is 81920 tokens. However, you requested "
                             f"{max_tokens} output tokens.", "", 0, t, kind, rescue)
            return 200, good, "short", REPLY_TOKENS, t, kind, rescue
        if rescue:
            return 200, "I could not determine the rule.", "", RESCUE_TOKENS, t, kind, rescue
        return 200, "", "no rule found " * 50, None, t, kind, rescue


def make_handler(script, model, log_path, lock):
    counter = [0]

    class H(BaseHTTPRequestHandler):
        def log_message(self, *a):
            pass

        def _send(self, code, obj):
            body = json.dumps(obj).encode()
            self.send_response(code)
            self.send_header("Content-Type", "application/json")
            self.send_header("Content-Length", str(len(body)))
            self.end_headers()
            self.wfile.write(body)

        def do_GET(self):
            if self.path.rstrip("/").endswith("/v1/models"):
                return self._send(200, {"object": "list", "data": [{"id": model, "object": "model"}]})
            return self._send(404, {"error": "not found"})

        def do_POST(self):
            if not self.path.rstrip("/").endswith("/chat/completions"):
                return self._send(404, {"error": "not found"})
            payload = json.loads(self.rfile.read(int(self.headers.get("Content-Length", 0))) or b"{}")
            with lock:
                counter[0] += 1
                n = counter[0]
            t_in = time.time()
            code, content, reasoning, ntok, t, kind, rescue = script.reply(payload)
            max_tokens = int(payload.get("max_tokens") or 16384)
            finish = "stop"
            if ntok is None or ntok >= max_tokens:
                ntok, finish = max_tokens, "length"
            if script.tok_s > 0 and code == 200 and t is not None:
                time.sleep(ntok / script.tok_s)           # decoding at tok_s per request
            if log_path:
                with lock, open(log_path, "a") as f:
                    f.write(json.dumps({"n": n, "task_id": t, "kind": kind, "rescue": rescue, "seed": payload.get("seed"),
                                        "sample": int(payload.get("seed") or 0) // 1000,
                                        "max_tokens": max_tokens, "received": round(t_in, 2),
                                        "thinking_token_budget": payload.get("thinking_token_budget"),
                                        "chat_template_kwargs": payload.get("chat_template_kwargs"),
                                        "temperature": payload.get("temperature"), "top_p": payload.get("top_p"),
                                        "top_k": payload.get("top_k"),
                                        "tools": [x["function"]["name"] for x in payload.get("tools", [])],
                                        "status": code, "finish": finish if code == 200 else None,
                                        "completion_tokens": ntok if code == 200 else 0}) + "\n")
            try:
                if code != 200:
                    return self._send(code, {"object": "error", "message": content, "type": "BadRequestError",
                                             "code": code})
                prompt_tokens = len(json.dumps(payload.get("messages", []))) // 3
                msg = {"role": "assistant", "content": content, "reasoning_content": reasoning}
                return self._send(200, {
                    "id": f"stub-{n}", "object": "chat.completion", "model": model,
                    "choices": [{"index": 0, "message": msg, "finish_reason": finish}],
                    "usage": {"prompt_tokens": prompt_tokens, "completion_tokens": ntok,
                              "total_tokens": prompt_tokens + ntok}})
            except (BrokenPipeError, ConnectionResetError):
                return None            # the client timed out (slow kind)

    return H


def serve(port, challenges, task_ids, solutions=None, b0_submission=None, model="qwen", log_path=None, slow_s=3600.0,
          tok_s=0.0):
    """Start in a daemon thread; returns the server (call .shutdown() to stop)."""
    script = Script(challenges, task_ids, solutions, b0_submission, slow_s, tok_s)
    srv = ThreadingHTTPServer(("127.0.0.1", port), make_handler(script, model, log_path, threading.Lock()))
    srv.daemon_threads = True
    threading.Thread(target=srv.serve_forever, daemon=True).start()
    return srv


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--port", type=int, default=1234)
    ap.add_argument("--challenges", required=True)
    ap.add_argument("--task-ids", required=True)
    ap.add_argument("--solutions", default="")
    ap.add_argument("--b0-submission", default="")
    ap.add_argument("--model", default="qwen")
    ap.add_argument("--log", default="")
    ap.add_argument("--slow-s", type=float, default=3600.0)
    ap.add_argument("--tok-s", type=float, default=0.0, help="decode rate per request (0: reply at once)")
    args = ap.parse_args()
    with open(args.challenges) as f:
        ch = json.load(f)
    sol = json.load(open(args.solutions)) if args.solutions else None
    b0 = json.load(open(args.b0_submission)) if args.b0_submission else None
    script = Script(ch, args.task_ids.split(","), sol, b0, args.slow_s, args.tok_s)
    srv = ThreadingHTTPServer(("127.0.0.1", args.port), make_handler(script, args.model, args.log or None,
                                                                     threading.Lock()))
    srv.daemon_threads = True
    print(f"hybrid stub server on 127.0.0.1:{args.port} for {len(script.ids)} tasks", flush=True)
    srv.serve_forever()


if __name__ == "__main__":
    main()


Writing /kaggle/working/hybrid/hybrid_stub_server.py


In [16]:
%%writefile /kaggle/working/hybrid/hybrid_stub_b0.py
"""Stand-in for B0's starter.py in the hybrid notebook's CPU twin and tests: writes candidate files in B0's format
(/kaggle/inference_outputs/<task>_<test>.<augmentation>: a bz2 pickle of [{"beam_score", "score_aug", "solution"}])
with designed score_kgmon margins, then keeps running like a worker stuck in test-time training until it is killed
(or until --end-time when --no-hang is given). It first prints B0's start-up lines ("[Rank k] start!", "Loading
checkpoint shards: 100%") for each of 4 ranks.

Designed pool by the task's position in --task-ids (index % 5), for every test output of the task:
  0  no candidates                                               (fewer than 2: routed first)
  1  one candidate, wrong                                        (fewer than 2: routed second)
  2  two wrong candidates, margin 0.1                            (routed third)
  3  right candidate with 3 beams, margin 2.3                    (routed last)
  4  two wrong candidates, margin 1.1, plus one file cut short   (routed fourth; the cut file is skipped)
"Right" is the solution from --solutions, else the test input stands in for it.

    python hybrid_stub_b0.py --challenges <file> --task-ids a,b,... --out <dir> [--solutions <file>]
        [--end-time <epoch>] [--interval-s 1] [--no-hang]
"""
import argparse
import bz2
import json
import os
import pickle
import time

import numpy as np

AUGS = ("transpose.permute0123456789", "rot90.permute9876543210", "rot90.rot90.permute0123456789",
        "transpose.rot90.permute1234567890")


def bump(g, i, j, d):
    a = np.array(g, dtype=np.int64)
    a[i, j] = (a[i, j] + d) % 10
    return a


def beam(grid, beam_score, aug):
    return {"beam_score": float(beam_score), "score_aug": [float(aug)] * 8, "solution": np.array(grid, dtype=np.int64)}


def design(kind, right):
    """[(file suffix, [beams])] for one test output."""
    if kind == 0:
        return []
    if kind == 1:
        return [(AUGS[0], [beam(bump(right, -1, -1, 2), 0.3, 0.4)])]
    if kind == 2:   # A: 1 - 0.5 = 0.5, B: 1 - 0.6 = 0.4 -> margin 0.1
        return [(AUGS[0], [beam(bump(right, 0, 0, 3), 0.2, 0.5)]), (AUGS[1], [beam(bump(right, -1, -1, 4), 0.3, 0.6)])]
    if kind == 3:   # right: 3 - 0.2 = 2.8, B: 1 - 0.5 = 0.5 -> margin 2.3
        return [(AUGS[0], [beam(right, 0.1, 0.2), beam(bump(right, 0, 0, 5), 0.9, 0.5)]),
                (AUGS[1], [beam(right, 0.1, 0.2)]), (AUGS[2], [beam(right, 0.2, 0.2)])]
    # kind 4: A: 2 - 0.3 = 1.7, B: 1 - 0.4 = 0.6 -> margin 1.1
    return [(AUGS[0], [beam(bump(right, -1, -1, 1), 0.2, 0.3)]), (AUGS[1], [beam(bump(right, -1, -1, 1), 0.2, 0.3)]),
            (AUGS[2], [beam(bump(right, 0, 0, 6), 0.4, 0.4)])]


def write_task(task_id, idx, task, solutions, out):
    kind = idx % 5
    for j, x in enumerate(task["test"]):
        right = solutions[task_id][j] if solutions and task_id in solutions else x["input"]
        for suffix, beams in design(kind, right):
            with bz2.BZ2File(os.path.join(out, f"{task_id}_{j}.{suffix}"), "w") as f:
                pickle.dump(beams, f)
        if kind == 4 and j == 0:
            data = bz2.compress(pickle.dumps([beam(right, 0.0, 0.0)] * 50))
            with open(os.path.join(out, f"{task_id}_{j}.{AUGS[3]}"), "wb") as f:
                f.write(data[: len(data) // 2])          # a file cut by a kill mid-write


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--challenges", required=True)
    ap.add_argument("--task-ids", required=True)
    ap.add_argument("--out", required=True)
    ap.add_argument("--solutions", default="")
    ap.add_argument("--end-time", type=float, default=0.0)
    ap.add_argument("--interval-s", type=float, default=1.0)
    ap.add_argument("--no-hang", action="store_true")
    ap.add_argument("--env-report", default="", help="write this process's Python and import facts here (CPU twin)")
    args = ap.parse_args()
    if args.env_report:
        import importlib.util
        import sys
        spec = importlib.util.find_spec("vllm")
        report = {"version": sys.version.split()[0], "executable": sys.executable, "sys_path": sys.path,
                  "pythonpath": os.environ.get("PYTHONPATH", ""), "path": os.environ.get("PATH", ""),
                  "vllm_origin": spec.origin if spec else None}
        with open(args.env_report, "w") as f:
            json.dump(report, f)
    ch = json.load(open(args.challenges))
    sol = json.load(open(args.solutions)) if args.solutions else None
    os.makedirs(args.out, exist_ok=True)
    # B0's start-up lines as its ranks print them (the notebook records their times)
    for rank in range(4):
        print(f"[Rank {rank}] start!", flush=True)
        print("Loading checkpoint shards: 100%|##########| 2/2 [00:01<00:00,  1.14it/s]", flush=True)
    for i, t in enumerate(args.task_ids.split(",")):
        if args.end_time and time.time() > args.end_time:
            print(f"[stub B0] stop before {t}: end time", flush=True)
            break
        write_task(t, i, ch[t], sol, args.out)
        print(f"[stub B0] wrote {t} (design {i % 5})", flush=True)
        time.sleep(args.interval_s)
    if args.no_hang:
        while args.end_time and time.time() < args.end_time:
            time.sleep(0.5)
        print("[stub B0] done", flush=True)
        return
    print("[stub B0] in flight past the end time (like a task in test-time training); waiting to be killed", flush=True)
    while True:
        time.sleep(1)


if __name__ == "__main__":
    main()


Writing /kaggle/working/hybrid/hybrid_stub_b0.py


In [17]:
# Phase 1: B0 (its run line as a subprocess: starter.py, 4 workers, end time global_end_time) in its own process
# group, killed at phase1_kill_s if a task in flight still runs; meanwhile the phase-2 wheelhouse install in the
# background and submission.json rebuilt from B0's candidate files every refresh_s; then the GPUs are checked free.
# B0's start-up is timed from its own lines ("[Rank k] start!", "Loading checkpoint shards: 100%", the first
# "[Rank k] finished"), in seconds after cell 1 (the Kaggle log's clock is about 136 s later).
import json
import os
import shutil
import signal
import subprocess
import sys
import threading
import time
import traceback

sys.path.insert(0, HYBRID_DIR)
import hybrid_core as HC

P1_INFO = {'start_s': round(time.time() - NOTEBOOK_START_TIME, 1),
           'b0_prefetch_at_start': {k: PREFETCH_INFO.get(k) for k in ('status', 'start_s')}}
_b0s = P1_INFO['b0_startup'] = {'rank_start_s': {}, 'weights_loaded_s': [], 'first_task_done_s': None}
_p1_done = threading.Event()
_install = {}


def _since():
    return round(time.time() - NOTEBOOK_START_TIME, 1)


def _start_install():
    # the phase-2 wheelhouse + overlay install (CPU and disk only), at preinstall_s or when phase 1 ends
    _p1_done.wait(max(0.0, NOTEBOOK_START_TIME + HYBRID_BUDGET['preinstall_s'] - time.time()))
    _py = P2_PYTHON or (sys.executable if globals().get('PREFLIGHT', False) else None)
    if _py is None:
        _install['skipped'] = 'no Python 3.12 interpreter'
        print('*** phase-2 install skipped: no Python 3.12 interpreter on this image', flush=True)
        return
    # phase 2 waits on this marker; the install process rewrites it with its pid and then its result
    _marker = os.path.join(P2_DIR, '_install.json')
    HC.write_json_atomic(_marker, {'status': 'running', 'start_epoch': time.time(), 'by': 'notebook'})
    try:
        _log = open(os.path.join(P2_DIR, '_install.log'), 'w')
        _install['proc'] = subprocess.Popen([_py, os.path.join(HYBRID_DIR, 'hybrid_p2.py'), 'install',
                                             os.path.join(P2_DIR, '_install_config.json')], cwd=HYBRID_DIR,
                                            stdout=_log, stderr=subprocess.STDOUT, start_new_session=True,
                                            env=P2_ENV if _py == P2_PYTHON else None)
    except Exception as exc:
        HC.write_json_atomic(_marker, {'status': 'failed', 'error': repr(exc)[:500], 'by': 'notebook'})
        return
    _install['start_s'] = _since()
    print(f"[hybrid {_since():8.1f}s] phase-2 install started in the background ({_py})", flush=True)


def _refresh():
    # submission.json from B0's files so far (B0's selection; a file cut mid-write is skipped)
    while not _p1_done.wait(HYBRID_BUDGET['refresh_s']):
        try:
            _sub, _pool, _info = HC.build_b0_pool(CHALLENGES_DATA, B0_OUTPUTS)
            _p = HC.write_submission(SUBMISSION, _sub, CHALLENGES_DATA)
            HC.write_submission(B0_SUBMISSION, _sub, CHALLENGES_DATA)
            HYBRID_LOG.append(HC.check_stage(SUBMISSION, CHALLENGES_DATA, f'phase 1 refresh at {_since()} s'))
            print(f"[hybrid {_since():8.1f}s] submission.json rebuilt from B0: {_info['outputs_with_candidates']} "
                  f"outputs with candidates, {len(_info['skipped_files'])} files skipped, problems {_p[:2]}", flush=True)
        except Exception:
            print('*** refresh failed:', traceback.format_exc()[-800:], flush=True)


def _gpu_memory():
    try:
        _r = subprocess.run(['nvidia-smi', '--query-gpu=index,memory.used', '--format=csv,noheader,nounits'],
                            capture_output=True, text=True, timeout=60)
        return [int(x.split(',')[1]) for x in _r.stdout.strip().splitlines() if ',' in x]
    except Exception:
        return None


_threads = [threading.Thread(target=_start_install, daemon=True), threading.Thread(target=_refresh, daemon=True)]
for _t in _threads:
    _t.start()
_killed = []
try:
    _env = dict(os.environ, UNSLOTH_DISABLE_STATISTICS='1', TRITON_PTXAS_PATH='/usr/local/cuda/bin/ptxas',
                OMP_NUM_THREADS='12')
    # B0 runs on this image's Python exactly as in its own notebook: nothing of phase 2 (the Python 3.12 build, the
    # vLLM site) on its PATH or PYTHONPATH
    for _var in ('PATH', 'PYTHONPATH'):
        if _var in _env:
            _env[_var] = os.pathsep.join(p for p in _env[_var].split(os.pathsep)
                                         if not p.startswith((PY312_HOME, VLLM_SITE)))
    _b0py = shutil.which('python', path=_env.get('PATH')) or sys.executable
    _r = subprocess.run([_b0py, '-c', 'import sys; print(sys.version.split()[0])'], capture_output=True, text=True,
                        env=_env, timeout=300)
    P1_INFO['b0_python'] = {'path': _b0py, 'version': _r.stdout.strip(), 'kernel': sys.version.split()[0]}
    if globals().get('PREFLIGHT', False):
        _cmd = [_b0py, os.path.join(HYBRID_DIR, 'hybrid_stub_b0.py'), '--challenges', CHALLENGES, '--task-ids',
                ','.join(HYBRID_TASK_IDS), '--out', B0_OUTPUTS, '--end-time', str(global_end_time),
                '--env-report', os.path.join(HYBRID_DIR, '_stub_b0_env.json')]
        _cmd += ['--solutions', EVAL_SOLUTIONS] if os.path.isfile(EVAL_SOLUTIONS) else []
    else:
        _cmd = [_b0py, 'starter.py', '--end-time', str(global_end_time)]
    print(f"[hybrid {_since():8.1f}s] phase 1: {' '.join(os.path.basename(c) for c in _cmd[:2])} on "
          f"{len(HYBRID_TASK_IDS)} tasks, end time +{HYBRID_BUDGET['phase1_end_s']} s, kill +{HYBRID_BUDGET['phase1_kill_s']} s",
          flush=True)
    _p1 = subprocess.Popen(_cmd, cwd=WORK_DIR, env=_env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
                           encoding='utf-8', errors='replace', bufsize=1, start_new_session=True)

    def _kill_p1():
        _killed.append(_since())
        try:
            os.killpg(_p1.pid, signal.SIGKILL)
        except Exception as exc:
            print('phase 1 kill:', repr(exc), flush=True)

    _timer = threading.Timer(max(1.0, NOTEBOOK_START_TIME + HYBRID_BUDGET['phase1_kill_s'] - time.time()), _kill_p1)
    _timer.start()
    for _line in _p1.stdout:            # always drained: a blocked pipe would stall B0
        try:
            print(_line, end='', flush=True)
            if _line.startswith('[Rank ') and '] start!' in _line:
                _b0s['rank_start_s'].setdefault(_line[6:].split(']')[0].strip(), _since())
            elif 'Loading checkpoint shards: 100%' in _line and len(_b0s['weights_loaded_s']) < 16:
                _b0s['weights_loaded_s'].append(_since())
            elif _b0s['first_task_done_s'] is None and _line.startswith('[Rank ') and '] finished ' in _line:
                _b0s['first_task_done_s'] = _since()
        except Exception:
            pass
    _p1.wait()
    _timer.cancel()
    P1_INFO.update({'exit_code': _p1.returncode, 'end_s': _since(), 'killed_s': _killed[0] if _killed else None})
    try:
        os.killpg(_p1.pid, signal.SIGKILL)       # workers left behind by a dead parent
    except Exception:
        pass
except Exception:
    P1_INFO['error'] = traceback.format_exc()[-2000:]
    print('*** phase 1 error:', P1_INFO['error'], flush=True)
_p1_done.set()
_threads[1].join(timeout=600)
# GPUs free before the 27B server: every GPU under 1500 MiB (submit_main.kill's test), else kill what holds them
_mem = _gpu_memory()
for _i in range(36):
    if _mem is None or all(m < 1500 for m in _mem):
        break
    if _i == 6:
        try:
            _r = subprocess.run(['nvidia-smi', '--query-compute-apps=pid', '--format=csv,noheader'], capture_output=True,
                                text=True, timeout=60)
            for _pid in {int(x) for x in _r.stdout.split() if x.strip().isdigit()} - {os.getpid()}:
                os.kill(_pid, signal.SIGKILL)
        except Exception as exc:
            print('gpu holders kill:', repr(exc), flush=True)
    time.sleep(5)
    _mem = _gpu_memory()
P1_INFO.update({'gpu_memory_mib_after': _mem, 'gpus_free': None if _mem is None else all(m < 1500 for m in _mem),
                'gpu_check_end_s': _since()})
HYBRID_LOG.append(HC.check_stage(SUBMISSION, CHALLENGES_DATA, 'phase 1 end'))
_b0s['first_rank_start_s'] = min(_b0s['rank_start_s'].values(), default=None)
_b0s['first_weights_loaded_s'] = min(_b0s['weights_loaded_s'], default=None)
print(f"[hybrid {_since():8.1f}s] B0 start-up (s after cell 1): first rank {_b0s['first_rank_start_s']}, ranks "
      f"{_b0s['rank_start_s']}, weights loaded {_b0s['weights_loaded_s'][:8]}, first task done "
      f"{_b0s['first_task_done_s']}; prefetch {PREFETCH_INFO.get('status')}", flush=True)
print(f"[hybrid {_since():8.1f}s] phase 1 done: {json.dumps(P1_INFO)[:800]}", flush=True)


[hybrid     55.5s] phase 1: python starter.py on 5 tasks, end time +720 s, kill +840 s
[hybrid     60.0s] phase-2 install started in the background (/kaggle/working/py312/python/bin/python3.12)
🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
[Rank 0] start!
==((====))==  Unsloth 2025.9.7: Fast Qwen3 patching. Transformers: 4.55.4.
   \\   /|    NVIDIA L4. Num GPUs = 1. Max memory: 22.034 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.8.0+cu128. CUDA: 8.9. CUDA Toolkit: 12.8. Triton: 3.4.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.32.post2. FA2 = True]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!

Loading checkpoint shards: 100%|██████████| 2/2 [00:01<00:00,  1.06it/s]
Unsloth: Training embed_tokens in mixed precision to save VRAM
Unsloth: Training lm_head in mixed precision to save VRAM
🦥 

In [18]:
# Safety net: B0's submission as its final cell builds it (ArcDecoder(n_guesses=2).run_selection_algo(), score_kgmon,
# over every readable candidate file), written atomically to submission.json and hybrid/submission_b0.json; B0's
# ranked candidates with kgmon scores and margins to hybrid/b0_pool.json. The phase-2 route: every task, shortest task
# JSON first (ties by id), whatever B0 found (Q27F); the plan over it (hybrid v2): pass 1 = sample 0 on the shortest
# ceil(0.25 n) tasks (the band), pass 2 = sample 1 on the band, pass 3 = sample 0 on the rest; hybrid/route.json.
import json
import os
import time
import traceback

_tasks = {t: CHALLENGES_DATA[t] for t in HYBRID_TASK_IDS}
try:
    ROUTE = HC.route_order(_tasks)
except Exception:
    print('*** route error:', traceback.format_exc()[-1000:])
    ROUTE = sorted(HYBRID_TASK_IDS)
try:
    PLAN = HC.issue_plan(ROUTE, Q27_CONFIG['band_fraction'], Q27_CONFIG['band_samples'], Q27_CONFIG['beyond_band'])
except Exception:
    print('*** plan error:', traceback.format_exc()[-1000:])
    PLAN = [[t, 0, 1] for t in ROUTE]
_pool = {}
try:
    _sub, _pool, B0_INFO = HC.build_b0_pool(CHALLENGES_DATA, B0_OUTPUTS)
    _p = HC.write_submission(B0_SUBMISSION, _sub, CHALLENGES_DATA)
    if _p:
        raise RuntimeError(f'B0 submission invalid: {_p[:3]}')
    HC.write_submission(SUBMISSION, _sub, CHALLENGES_DATA)
    _pool = {t: _pool[t] for t in HYBRID_TASK_IDS}
    HC.write_json_atomic(B0_POOL, {'schema': HC.SCHEMA, 'info': B0_INFO, 'route': HC.route_table(_pool, _tasks),
                                   'pool': _pool})
    print(f"safety net: {B0_INFO['outputs_with_candidates']} of {sum(len(_tasks[t]['test']) for t in _tasks)} test "
          f"outputs with B0 candidates; {len(B0_INFO['skipped_files'])} unreadable files skipped")
except Exception:
    print('*** safety net error:', traceback.format_exc()[-2000:])
    print('safety net (fallback):', HC.ensure_valid(B0_SUBMISSION, CHALLENGES_DATA, fallbacks=[SUBMISSION]))
HYBRID_LOG.append(HC.check_stage(SUBMISSION, CHALLENGES_DATA, 'safety net'))
try:
    ROUTE_TABLE = HC.route_table(_pool if isinstance(_pool, dict) else {}, _tasks, Q27_CONFIG['band_fraction'],
                                 Q27_CONFIG['band_samples'])
    _nb = HC.band_size(len(ROUTE), Q27_CONFIG['band_fraction'])
    _passes = {str(p): sum(1 for j in PLAN if j[2] == p) for p in sorted({j[2] for j in PLAN})}
    HC.write_json_atomic(os.path.join(HYBRID_DIR, 'route.json'),
                         {'rule': 'shortest task JSON first (len(json.dumps(task))), ties by id',
                          'band': {'fraction': Q27_CONFIG['band_fraction'], 'tasks': _nb,
                                   'samples': Q27_CONFIG['band_samples'],
                                   'max_size': ROUTE_TABLE[_nb - 1]['size'] if _nb else None},
                          'plan_rule': 'pass 1: sample 0 on the band; pass 2: sample 1 on the band (not sent when '
                                       'sample 0 already differs from attempt_1 on every output); pass 3: sample 0 '
                                       'on the tasks beyond the band; each pass shortest first',
                          'samples_per_pass': _passes, 'route': ROUTE_TABLE, 'plan': PLAN}, indent=1)
    _with_b0 = sum(1 for r in ROUTE_TABLE if any(o[0] for o in r['outputs']))
    print(f"route ({len(ROUTE)} tasks, shortest first; {_with_b0} with B0 candidates; band {_nb} tasks): "
          f"{[(r['task_id'], r['size']) for r in ROUTE_TABLE[:12]]}{' ...' if len(ROUTE) > 12 else ''}")
    print(f"plan: {len(PLAN)} samples, per pass {_passes}; head {PLAN[:8]}")
except Exception:
    print('*** route log error:', traceback.format_exc()[-1000:])


safety net: 3 of 10 test outputs with B0 candidates; 0 unreadable files skipped
route (5 tasks, shortest first; 2 with B0 candidates; band 2 tasks): [('16de56c4', 3762), ('abc82100', 7103), ('4e34c42c', 7110), ('cbebaa4b', 10241), ('13e47133', 11422)]
plan: 7 samples, per pass {'1': 2, '2': 2, '3': 3}; head [['16de56c4', 0, 1], ['abc82100', 0, 1], ['16de56c4', 1, 2], ['abc82100', 1, 2], ['4e34c42c', 0, 3], ['cbebaa4b', 0, 3], ['13e47133', 0, 3]]


In [19]:
# Phase 2 (version 7-vr): hybrid_vr.py run (the harness in place of the direct mode; (wait for the install, 27B server ladder, smoke request, direct answers in PLAN order until
# issue_end_s, submission.json merged after every answer) in its own process group, killed at p2_kill_s if still
# running; any server left is killed; then the final merge from samples.jsonl (idempotent).
import json
import os
import signal
import subprocess
import sys
import threading
import time
import traceback

P2_INFO = {}
_e = lambda k: NOTEBOOK_START_TIME + HYBRID_BUDGET[k]   # noqa: E731
try:
    _threads[0].join(timeout=60)      # the background install has started (or was skipped) before phase 2 waits on it
    _cfg = dict(Q27_CONFIG, route=ROUTE, plan=PLAN, run_t0=NOTEBOOK_START_TIME, vr=VR_CONFIG,
                server_give_up_epoch=_e('server_give_up_s'),
                issue_end_epoch=_e('issue_end_s'), collect_end_epoch=_e('collect_end_s'))
    _cfg_path = os.path.join(P2_DIR, '_p2_config.json')
    with open(_cfg_path, 'w') as _f:
        json.dump(_cfg, _f, indent=1)
    _py = P2_PYTHON or (sys.executable if Q27_CONFIG['server'] == 'stub' else None)
    if _py is None:
        P2_INFO['skipped'] = 'no Python 3.12 interpreter on this image'
    elif time.time() >= _cfg['issue_end_epoch']:
        P2_INFO['skipped'] = 'issue deadline already passed'
    else:
        _left = _e('p2_kill_s') - time.time()
        print(f'phase 2: {len(ROUTE)} tasks routed, {len(PLAN)} samples planned; orchestrator killed in {_left:.0f} s '
              f'if still running', flush=True)
        _proc = subprocess.Popen([_py, os.path.join(HYBRID_DIR, 'hybrid_vr.py'), 'run', _cfg_path], cwd=HYBRID_DIR,
                                 stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, encoding='utf-8',
                                 errors='replace', bufsize=1,
                                 start_new_session=True, env=P2_ENV if _py == P2_PYTHON else None)
        _killed = []

        def _hard_stop():
            _killed.append(time.time() - NOTEBOOK_START_TIME)
            try:
                os.killpg(_proc.pid, signal.SIGKILL)
            except Exception as exc:
                print('phase 2 kill:', repr(exc), flush=True)

        _timer = threading.Timer(max(1.0, _left), _hard_stop)
        _timer.start()
        with open(os.path.join(P2_DIR, '_p2.log'), 'w') as _log:
            for _line in _proc.stdout:
                try:                    # always drained: a blocked pipe would stall phase 2
                    print(_line, end='', flush=True)
                    _log.write(_line)
                    if '| merge ' in _line:
                        HYBRID_LOG.append(HC.check_stage(SUBMISSION, CHALLENGES_DATA, 'phase 2 after an answer'))
                except Exception:
                    pass
        _proc.wait()
        _timer.cancel()
        P2_INFO.update({'exit_code': _proc.returncode, 'killed_s': round(_killed[0], 1) if _killed else None})
        if _killed:
            print(f'*** phase 2 killed {_killed[0]:.0f} s after start', flush=True)
            HYBRID_LOG.append(HC.check_stage(SUBMISSION, CHALLENGES_DATA, 'phase 2 after the kill'))
except Exception:
    P2_INFO['error'] = traceback.format_exc()[-2000:]
    print('*** phase 2 error:', P2_INFO['error'], flush=True)
# a server or an install left running (kill): its process group
for _pf in (os.path.join(P2_DIR, '_server.pid'),):
    if os.path.isfile(_pf):
        try:
            os.killpg(int(open(_pf).read().strip()), signal.SIGKILL)
            print('killed the server left running', flush=True)
        except Exception as exc:
            print('server kill:', repr(exc), flush=True)
        os.remove(_pf)
if _install.get('proc') is not None and _install['proc'].poll() is None and not globals().get('PREFLIGHT', False):
    try:
        os.killpg(_install['proc'].pid, signal.SIGKILL)
    except Exception:
        pass
try:
    # version 7-vr: the harness's selection over all its samples and B0's pool (idempotent with its last write)
    sys.path.insert(0, HYBRID_DIR)
    import hybrid_vr
    from arc_harness.data import load_kaggle
    from arc_harness.submission import write_submission
    _vt = load_kaggle(CHALLENGES)
    _vs = hybrid_vr.read_samples(os.path.join(P2_DIR, 'vr', 'samples.jsonl'))
    _vroute = [t for t in ROUTE if t in _vt]
    write_submission(hybrid_vr.full_submission(_vt, _vroute, _vs, hybrid_vr.b0_pool(B0_SUBMISSION, _vt), B0_SUBMISSION,
                                               VR_CONFIG['merge']), _vt, SUBMISSION)
    P2_INFO['final_merge'] = {'status': 'ok', 'samples': sum(len(v) for v in _vs.values()), 'tasks_sampled': len(_vs),
                              'verified_tasks': sum(1 for v in _vs.values() if any(s['verify']['passed'] for s in v))}
except Exception:
    P2_INFO['final_merge'] = {'status': 'error', 'error': traceback.format_exc()[-1000:]}
P2_INFO['ensure_valid'] = HC.ensure_valid(SUBMISSION, CHALLENGES_DATA, fallbacks=[B0_SUBMISSION])
HYBRID_LOG.append(HC.check_stage(SUBMISSION, CHALLENGES_DATA, 'phase 2 end'))
print(f"phase 2 done {time.time() - NOTEBOOK_START_TIME:.0f} s after start: {json.dumps(P2_INFO)[:800]}", flush=True)


phase 2: 5 tasks routed, 7 samples planned; orchestrator killed in 2393 s if still running
[p2    727.4s] start: ok (server=vllm)
[p2    727.4s] install: ok (background install, 169.2 s)
[p2    733.2s] torch: ok (torch 2.10.0+cu128 (CUDA 12.8), driver 580.178.04 (CUDA <= 13.0))
libcuda: ['/usr/local/nvidia/lib64/libcuda.so', '/usr/local/nvidia/lib64/libcuda.so.1', '/usr/local/nvidia/lib64/libcuda.so.580.178.04', '/usr/local/cuda/compat/libcuda.so']
/kaggle/working/py312/python/bin/python3.12 -m vllm.entrypoints.openai.api_server --model /kaggle/input/models/mikedan7/qwen3-8-27b-fp8-official/pytorch/hf-fp8/1 --served-model-name qwen --host 127.0.0.1 --port 1234 --tensor-parallel-size 4 --enable-auto-tool-choice --tool-call-parser qwen3_coder --reasoning-parser qwen3 --generation-config vllm --enable-prefix-caching --max-model-len 81920 --gpu-memory-utilization 0.90 --max-num-seqs 20 --reasoning-config {"reasoning_start_str": "<think>", "reasoning_end_str": "</think>"} --kv-cache-dtype f

In [20]:
# Final: submission.json valid whatever happened (else the safety net, else the placeholder); the serving install
# removed; hybrid/_notebook.json; in a commit run with Kaggle's evaluation solutions mounted, pass@2 of B0 alone and
# of the hybrid on the commit-run tasks.
import json
import os
import shutil
import time
import traceback

# version 7: the commit-run checks. (1) is version 7's own: B0 is version 2's (64 training sequences per task; all 16
# DFS views: 4 batches per test output, 2 of them transposed), read from B0's DFS record (hybrid/b0_dfs.jsonl, one line
# per task from arc_solver.py) and the training-set size the setup cell gave arc_solver.py (whose guard stops a task
# with another size). (2)-(4) are phase 2's (hybrid_p2.py as version 6nb's, unchanged); phase 2's own check (1), version
# 6's DFS time against H2, does not apply to version 7, whose B0 searches all views again.
HYBRID_V7_B0 = {'ttt_sequences': 64, 'batches_per_output': 4, 'transposed_per_output': 2, 'task_limit_s': 1200}
HYBRID_V7_CHECK_1 = '1_b0_as_version_2_64_sequences_4_dfs_batches_per_output_2_transposed'
HYBRID_V7_P2_CHECKS = ('2_zero_token_at_collection_le_2', '3_band_issued_only_when_allowed', '4_per_output_parser_active')


def v7_b0_record(path, sequences):
    """Check 1 from B0's DFS record and the training-set size: 64 sequences; on every task 4 batches built per test
    output, 2 of them transposed (version 2's two batch loops), all decoded unless B0's end time or its 1200-s task limit
    stopped the task (it decodes them in order, the transposed ones last); at least one task decoded every batch. The
    record's summary with 'pass' (True or False; None without a record) and 'detail'."""
    want = HYBRID_V7_B0
    recs = {}
    for r in HC.read_sample_lines(path) if path else []:
        if isinstance(r, dict) and r.get('task_id'):
            recs[r['task_id']] = r                      # the last line per task
    rows = list(recs.values())
    n_out = lambda r: int(r.get('outputs') or 0)  # noqa: E731
    built_otherwise = sorted(r['task_id'] for r in rows if n_out(r) < 1
                             or r.get('batches_built') != want['batches_per_output'] * n_out(r)
                             or r.get('transposed_batches') != want['transposed_per_output'] * n_out(r))
    every = [r['task_id'] for r in rows if r.get('batches_decoded') == r.get('batches_built')]
    stopped = {r['task_id']: [r.get('batches_decoded'), r.get('batches_built')] for r in rows
               if r.get('batches_decoded') != r.get('batches_built')}
    unexplained = sorted(t for t in stopped if not (recs[t].get('end_time_reached')
                                                    or float(recs[t].get('task_s') or 0) > want['task_limit_s']))
    dfs = [float(r.get('dfs_s') or 0) for r in rows]
    out = {'path': path, 'sequences': sequences, 'tasks': len(rows), 'decoded_every_batch': len(every),
           'stopped_by_the_clock': stopped, 'built_otherwise': built_otherwise,
           'fewer_decoded_without_a_stop': unexplained,
           'batches_per_output': sorted({round(r['batches_built'] / n_out(r), 2) for r in rows
                                         if n_out(r) and isinstance(r.get('batches_built'), int)}),
           'transposed_batches': sum(int(r.get('transposed_batches') or 0) for r in rows),
           'dfs_mean_s': round(sum(dfs) / len(dfs), 1) if dfs else None,
           'per_task': {r['task_id']: [r.get('dfs_s'), r.get('batches_decoded'), r.get('batches_built'),
                                       r.get('transposed_batches'), r.get('outputs')] for r in rows}}
    if str(sequences) != str(want['ttt_sequences']):
        out['pass'] = False
    elif not rows:
        out['pass'] = None
    else:
        out['pass'] = not built_otherwise and not unexplained and bool(every)
    shown = ', '.join(f'{t} {d} of {b}' for t, (d, b) in list(stopped.items())[:8]) + (', ...' if len(stopped) > 8 else '')
    out['detail'] = (f'{sequences} training sequences per task; '
                     + ('no B0 DFS record' if not rows else
                        f"{len(rows) - len(built_otherwise)} of {len(rows)} tasks with {want['batches_per_output']} "
                        f"batches per output, {want['transposed_per_output']} transposed; {len(every)} decoded every batch"
                        + (f"; {len(stopped)} stopped by B0's end time or task limit ({shown})" if stopped else '')
                        + (f'; built otherwise: {built_otherwise[:8]}' if built_otherwise else '')
                        + (f'; fewer decoded without a stop: {unexplained[:8]}' if unexplained else '')))
    return out


def v7_summary(b0, p2_v6):
    """The version-7 record: the lines 'v7 (version 2's B0, version 6nb's phase 2): ...' and 'v7 checks: ...'; check 1
    from v7_b0_record, checks 2-4 from phase 2's version-6 record (NO DATA without one)."""
    p2_v6 = p2_v6 or {}
    checks = {HYBRID_V7_CHECK_1: {'pass': b0['pass'], 'detail': b0['detail']}}
    for k in HYBRID_V7_P2_CHECKS:
        checks[k] = dict((p2_v6.get('checks') or {}).get(k) or {'pass': None, 'detail': 'no phase-2 record'})
    v6_line = str(p2_v6.get('line') or '')
    p2_part = (v6_line.split('), ', 1)[1] if v6_line.startswith('v6: B0 DFS') and '), ' in v6_line
               else v6_line or 'no phase-2 record')
    mean = 'n/a' if b0['dfs_mean_s'] is None else f"{b0['dfs_mean_s']:.0f}"
    line = (f"v7 (version 2's B0, version 6nb's phase 2): B0 {b0['sequences']} training sequences per task, DFS record "
            f"{b0['tasks']} tasks, batches per output {b0['batches_per_output']}, transposed {b0['transposed_batches']}, "
            f"{b0['decoded_every_batch']} decoded every batch, DFS mean s {mean}; phase 2: {p2_part}")
    verdict = {True: 'PASS', False: 'FAIL', None: 'NO DATA'}
    checks_line = 'v7 checks: ' + '; '.join(
        f"({k.split('_', 1)[0]}) {k.split('_', 1)[1].replace('_', ' ')}: {verdict.get(v.get('pass'), v.get('pass'))} "
        f"({v.get('detail')})" for k, v in checks.items())
    return {'line': line, 'checks_line': checks_line, 'checks': checks, 'b0_dfs': b0}

try:
    FINAL_INFO = HC.ensure_valid(SUBMISSION, CHALLENGES_DATA, fallbacks=[B0_SUBMISSION])
    HYBRID_LOG.append(HC.check_stage(SUBMISSION, CHALLENGES_DATA, 'final'))
    if not globals().get('PREFLIGHT', False):
        shutil.rmtree(VLLM_SITE, ignore_errors=True)
        shutil.rmtree(PY312_HOME, ignore_errors=True)
    _p2run = os.path.join(P2_DIR, '_p2_run.json')
    _p2r = json.load(open(_p2run)) if os.path.isfile(_p2run) else None
    _lines = HC.read_sample_lines(os.path.join(P2_DIR, 'samples.jsonl'))
    _sampled = {l['task_id'] for l in _lines}
    _answered = HC.answers_from_lines(_lines)
    _mrg = os.path.join(P2_DIR, '_merge.json')
    _per_task = (json.load(open(_mrg)).get('per_task') if os.path.isfile(_mrg) else None) or {}
    _by_pass = {str(p): {'planned': sum(1 for j in PLAN if j[2] == p),
                         'finished': sum(1 for l in _lines if l.get('pass') == p),
                         'with_grids': sum(1 for l in _lines if l.get('pass') == p and l.get('grids'))}
                for p in sorted({j[2] for j in PLAN})}
    HYBRID_SUMMARY = {
        'mode': 'competition rerun' if HYBRID_RERUN else 'commit run (SMOKE)', 'budget': HYBRID_BUDGET,
        'tasks': len(HYBRID_TASK_IDS), 'route_rule': 'shortest task JSON first, ties by id', 'route_head': ROUTE[:20],
        'band': {k: Q27_CONFIG[k] for k in ('band_fraction', 'band_samples', 'beyond_band')},
        'band_tasks': HC.band_size(len(ROUTE), Q27_CONFIG['band_fraction']), 'plan_head': PLAN[:20],
        'samples_by_pass': _by_pass, 'samples_finished': len(_lines),
        'tasks_with_27b_sample': len(_sampled), 'tasks_with_27b_grids': len(_answered),
        'last_route_position_sampled': max((ROUTE.index(t) for t in _sampled if t in ROUTE), default=None),
        'per_task': _per_task, 'b0_prefetch': PREFETCH_INFO,
        'phase1': P1_INFO, 'phase2': P2_INFO, 'phase2_run': _p2r, 'final': FINAL_INFO,
        'stages': HYBRID_LOG, 'end_s': round(time.time() - NOTEBOOK_START_TIME, 1),
    }
    print(f"27B: {len(_lines)} of {len(PLAN)} planned samples finished, by pass {_by_pass}; {len(_sampled)} of "
          f"{len(ROUTE)} routed tasks got a sample (route positions up to "
          f"{HYBRID_SUMMARY['last_route_position_sampled']}), {len(_answered)} with grids; merge "
          f"{P2_INFO.get('final_merge', {}).get('replaced')} attempt_2 replaced, attempt_2 by sample "
          f"{P2_INFO.get('final_merge', {}).get('attempt_2_from_sample')}")
    for _t in ROUTE[:40]:
        if _t in _per_task:
            print(f"  {_t}: samples {_per_task[_t]['samples']}, attempt_2 from {_per_task[_t]['outputs']}")
    print(f"B0 prefetch: {PREFETCH_INFO.get('status')}; B0 start-up {P1_INFO.get('b0_startup')}")
    if not HYBRID_RERUN and os.path.isfile(EVAL_SOLUTIONS):
        with open(EVAL_SOLUTIONS) as _f:
            _sol = json.load(_f)
        _b0 = json.load(open(B0_SUBMISSION))
        _hy = json.load(open(SUBMISSION))
        HYBRID_SUMMARY['score'] = {'b0': HC.pass_at_2(_b0, _sol, HYBRID_TASK_IDS),
                                   'hybrid': HC.pass_at_2(_hy, _sol, HYBRID_TASK_IDS)}
        print(f"*** commit-run tasks: B0 {HYBRID_SUMMARY['score']['b0']['outputs']}/"
              f"{HYBRID_SUMMARY['score']['b0']['n_outputs']} outputs, hybrid {HYBRID_SUMMARY['score']['hybrid']['outputs']}/"
              f"{HYBRID_SUMMARY['score']['hybrid']['n_outputs']}")
    # version 7: check 1 from B0's DFS record, checks 2-4 from phase 2's record (none if phase 2 was killed)
    try:
        HYBRID_SUMMARY['v7'] = v7_summary(v7_b0_record(Q27_CONFIG.get('b0_dfs_log'), os.environ.get('HYBRID_TTT_SEQUENCES')),
                                          (_p2r or {}).get('v6'))
        print(HYBRID_SUMMARY['v7']['line'])
        print(HYBRID_SUMMARY['v7']['checks_line'])
    except Exception:
        print('*** v7 checks:', traceback.format_exc()[-1000:])
    HC.write_json_atomic(os.path.join(HYBRID_DIR, '_notebook.json'), HYBRID_SUMMARY, indent=1)
    print('submission.json:', FINAL_INFO, '| stages valid:', all(r['valid'] for r in HYBRID_LOG))
except Exception:
    print('*** final cell error:', traceback.format_exc()[-2000:])
print(f"hybrid notebook done {time.time() - NOTEBOOK_START_TIME:.0f} s after start (hard stop "
      f"{HYBRID_BUDGET['hard_stop_s']} s)")


27B: 0 of 7 planned samples finished, by pass {'1': {'planned': 2, 'finished': 0, 'with_grids': 0}, '2': {'planned': 2, 'finished': 0, 'with_grids': 0}, '3': {'planned': 3, 'finished': 0, 'with_grids': 0}}; 0 of 5 routed tasks got a sample (route positions up to None), 0 with grids; merge None attempt_2 replaced, attempt_2 by sample None
B0 prefetch: done; B0 start-up {'rank_start_s': {'0': 212.8, '1': 275.5, '2': 335.5, '3': 391.5}, 'weights_loaded_s': [214.7, 214.7, 277.4, 277.4, 337.4, 337.4, 393.4, 393.4], 'first_task_done_s': 469.3, 'first_rank_start_s': 212.8, 'first_weights_loaded_s': 214.7}
*** commit-run tasks: B0 0/10 outputs, hybrid 0/10
v7 (version 2's B0, version 6nb's phase 2): B0 64 training sequences per task, DFS record 5 tasks, batches per output [4.0], transposed 20, 2 decoded every batch, DFS mean s 202; phase 2: no phase-2 record
v7 checks: (1) b0 as version 2 64 sequences 4 dfs batches per output 2 transposed: PASS (64 training sequences per task; 5 of 5 tasks wit